# GAT crypto

Prediccion conjunta, a 10 minutos, del simplex de cuotas transaccionales de
criptoactivos. Cada activo es un nodo. El objetivo es el vector completo
`p(t+1)` y la salida se normaliza con softmax, de modo que siempre pertenece
al simplex.

Este notebook replica para **cada** `*_p` (excepto `YS_p`) las 20 variables
del XGBoost de factores de BTC: seis rezagos y los 14 factores que sobrevivieron
seleccion temporal, importancia por permutacion y ablacion por familias.

Contratos metodologicos:

- horizonte fijo `t -> t+1`; ninguna feature usa `shift(-k)`;
- cortes cronologicos con purga; `shuffle=False` en todos los loaders;
- escalado ajustado exclusivamente en train;
- aristas calculadas con CLR historico hasta `t`, nunca con `p(t+1)`;
- test final intacto hasta terminar seleccion/early stopping;
- `YS_p` se excluye de los nodos, pero se verifica que las cuotas restantes
  sumen uno;
- las cabezas `gp` y `eta` son latentes. Con datos transaccionales solamente
  es identificable su cociente energetico `E=gp/eta`, no una interpretacion
  causal separada de ambos componentes.



In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "PyTorch no esta instalado. En Google Colab selecciona un runtime con GPU."
    ) from exc

from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, SequentialSampler




## 1. Configuracion reproducible

Las rutas corresponden a los cuatro Parquet creados por
`EXT NPP Crypto 01/2026`. Si Drive aparece bajo otra ruta, solo hay que
modificar este bloque.



In [ ]:
@dataclass(frozen=True)
class Config:
    dfyp_path: str = "/content/drive/MyDrive/0626dfyp.parquet"
    dftu_path: str = "/content/drive/MyDrive/0626dftu.parquet"
    price_path: str = "/content/drive/MyDrive/0626p.parquet"
    volume_path: str = "/content/drive/MyDrive/0626v.parquet"
    results_dir: str = "/content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_results"
    cache_dir: str = "/content/gat_crypto_cache"

    start_timestamp: str = "2026-01-01 00:00:00+00:00"
    frequency: str = "10min"
    horizon: int = 1
    validation_size: int = 2016
    test_size: int = 2016
    purge: int = 1
    sample_stride: int = 1

    edge_window: int = 1008       # siete dias de barras de 10 minutos
    edge_refresh: int = 1008      # grafo causal por semana
    edge_projection_dim: int = 24
    edge_top_k: int = 8
    edge_alpha_clr: float = 0.75
    edge_gamma_npp: float = 0.25
    min_active_share: float = 1e-12

    hidden_dim: int = 32
    attention_heads: int = 4
    dropout: float = 0.10
    batch_size: int = 8
    epochs: int = 20
    patience: int = 4
    learning_rate: float = 3e-4
    weight_decay: float = 1e-5
    grad_clip: float = 1.0

    lambda_npp: float = 0.05
    lambda_energy_ce: float = 0.25
    lambda_eta_anchor: float = 1e-4
    lambda_energy_smooth: float = 1e-4

    eps: float = 1e-12
    seed: int = 42
    rebuild_features: bool = True


CFG = Config()


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)




Dispositivo: cpu


## 2. Carga y auditoria del simplex

El notebook de extraccion guardo los Parquet con `index=False`. Por eso se
reconstruye el indice UTC a frecuencia exacta de 10 minutos. No se reordena
ninguna fila y no se hace backward-fill.



In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Fuera de Colab: se usan las rutas locales indicadas en CFG.")


def read_table(path: str) -> pd.DataFrame:
    suffix = Path(path).suffix.lower()
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Formato no soportado: {path}")


dfyp = read_table(CFG.dfyp_path)
dftu = read_table(CFG.dftu_path)
dfp = read_table(CFG.price_path)
dfv = read_table(CFG.volume_path)

if "YS" in dfyp.columns and "YS_p" not in dfyp.columns:
    dfyp = dfyp.rename(
        columns=lambda c: c[:-2] + "_p" if c.endswith("_y") else "YS_p" if c == "YS" else c
    )

lengths = {"dfyp": len(dfyp), "dftu": len(dftu), "dfp": len(dfp), "dfv": len(dfv)}
if len(set(lengths.values())) != 1:
    raise ValueError(f"Los cuatro archivos no tienen la misma longitud: {lengths}")

timestamps = pd.date_range(
    start=pd.Timestamp(CFG.start_timestamp), periods=len(dfyp), freq=CFG.frequency
)
for frame in (dfyp, dftu, dfp, dfv):
    frame.index = timestamps

p_cols = [c for c in dfyp.columns if c.endswith("_p") and c != "YS_p"]
if not p_cols:
    raise ValueError("No se encontraron columnas *_p en dfyp.")

assets = [c[:-2] for c in p_cols]
price_cols = [f"{asset}USDT" for asset in assets]
y_cols = [f"{asset}_y" for asset in assets]
volume_cols = [
    f"{asset}_V" if f"{asset}_V" in dfv.columns else f"{asset}_v"
    for asset in assets
]

missing = {
    "precio": [c for c in price_cols if c not in dfp.columns],
    "y": [c for c in y_cols if c not in dftu.columns],
    "volumen": [c for c in volume_cols if c not in dfv.columns],
}
missing = {k: v for k, v in missing.items() if v}
if missing:
    preview = {k: v[:20] for k, v in missing.items()}
    raise ValueError(
        "No se eliminan nodos silenciosamente. Faltan columnas para algunos *_p: "
        f"{preview}"
    )

P = dfyp[p_cols].to_numpy(dtype=np.float32, copy=True)
Y = dftu[y_cols].to_numpy(dtype=np.float32, copy=True)
PRICE = dfp[price_cols].to_numpy(dtype=np.float32, copy=True)
VOLUME = dfv[volume_cols].to_numpy(dtype=np.float32, copy=True)

if np.nanmin(P) < -1e-8:
    raise ValueError("dfyp contiene probabilidades negativas.")

# En la construccion original, NaN de un par ausente significa masa cero.
P = np.nan_to_num(P, nan=0.0, posinf=0.0, neginf=0.0)
P = np.clip(P, 0.0, None)
row_sums = P.sum(axis=1, keepdims=True, dtype=np.float64)
if np.any(row_sums <= 0):
    bad = np.flatnonzero(row_sums.ravel() <= 0)[:20]
    raise ValueError(f"Hay filas sin masa transaccional: {bad.tolist()}")
P = (P / row_sums).astype(np.float32)

simplex_error = np.abs(P.sum(axis=1, dtype=np.float64) - 1.0)
if simplex_error.max() > 1e-5:
    raise AssertionError("La normalizacion del simplex fallo.")
if "YS_p" in dfyp.columns:
    ys_p = pd.to_numeric(dfyp["YS_p"], errors="coerce").to_numpy()
    if np.nanmax(np.abs(ys_p - 1.0)) > 1e-5:
        raise ValueError("YS_p no es identicamente uno; revisar la construccion de dfyp.")

print(f"Filas: {len(P):,}; nodos: {len(assets):,}")
print(f"Error maximo de cierre del simplex: {simplex_error.max():.3e}")
print("Rango temporal:", timestamps[0], "->", timestamps[-1])




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Filas: 26,064; nodos: 471
Error maximo de cierre del simplex: 4.488e-08
Rango temporal: 2026-01-01 00:00:00+00:00 -> 2026-06-30 23:50:00+00:00


## 3. Las 20 variables del XGBoost, desarrolladas para todos los nodos

Variables finales del notebook `Vars NPP Crypto 01/2026`:

- `p_lag0` ... `p_lag5`;
- sorpresa de Shannon;
- RLIQ a 20m, 30m, 1h, 2h y 4h;
- skew de la cuota a 1 y 2 meses;
- indice `n = EMA_precio_1m / sorpresa`;
- MACD de precio 1 semana vs 2 semanas;
- estabilidad bajista a 2 y 3 meses;
- RSIp a 3 dias;
- curtosis de la cuota a 3 meses.

A diferencia de las celdas exploratorias antiguas, skew y curtosis se calculan
realmente sobre `p`, igual que en la celda final de preparacion del XGBoost.
El archivo de features es un memmap float32 para no exigir que ~4 GB residan
simultaneamente en RAM.



In [ ]:
WINDOWS = {
    "20": 2,
    "30": 3,
    "1h": 6,
    "2h": 12,
    "4h": 24,
    "3d": 432,
    "1s": 1008,
    "2s": 2016,
    "1m": 4032,
    "2m": 8064,
    "3m": 12128,
}

FEATURE_NAMES = [f"p_lag{lag}" for lag in range(6)] + [
    "shannon",
    "RLIQ_20",
    "RLIQ_30",
    "RLIQ_1h",
    "RLIQ_2h",
    "RLIQ_4h",
    "SKEW_p_1m",
    "n_model",
    "MACD_1s_2s",
    "RSTAB_3m",
    "SKEW_p_2m",
    "RSIp_3d",
    "KURT_p_3m",
    "RSTAB_2m",
]

assert len(FEATURE_NAMES) == 20
WARMUP = max(WINDOWS["3m"], WINDOWS["2m"], WINDOWS["1m"])

cache_dir = Path(CFG.cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)
feature_path = cache_dir / "gat_crypto_features_float32.dat"
feature_meta_path = cache_dir / "gat_crypto_features_meta.json"
feature_shape = (len(P), len(assets), len(FEATURE_NAMES))


def causal_ffill(values: np.ndarray) -> pd.DataFrame:
    # ffill solo usa observaciones <= t. No se aplica bfill a huecos iniciales.
    return pd.DataFrame(values, index=timestamps, columns=assets).ffill()


def build_feature_memmap() -> np.memmap:
    expected_meta = {
        "shape": list(feature_shape),
        "features": FEATURE_NAMES,
        "dtype": "float32",
    }
    if (
        feature_path.exists()
        and feature_meta_path.exists()
        and not CFG.rebuild_features
        and json.loads(feature_meta_path.read_text(encoding="utf-8")) == expected_meta
    ):
        print("Se reutiliza cache:", feature_path)
        return np.memmap(feature_path, mode="r+", dtype=np.float32, shape=feature_shape)

    print(f"Construyendo memmap de {np.prod(feature_shape) * 4 / 2**30:.2f} GiB")
    mm = np.memmap(feature_path, mode="w+", dtype=np.float32, shape=feature_shape)
    p_df = pd.DataFrame(P, index=timestamps, columns=assets)
    y_df = pd.DataFrame(np.nan_to_num(Y, nan=0.0), index=timestamps, columns=assets)
    price_df = causal_ffill(PRICE)

    slot = 0

    def write_feature(name: str, values) -> None:
        nonlocal slot
        if FEATURE_NAMES[slot] != name:
            raise AssertionError(f"Orden de features incoherente: {name} != {FEATURE_NAMES[slot]}")
        if isinstance(values, pd.DataFrame):
            arr = values.to_numpy(dtype=np.float32, copy=False)
        else:
            arr = np.asarray(values, dtype=np.float32)
        if arr.shape != feature_shape[:2]:
            raise ValueError(f"Shape invalido para {name}: {arr.shape}")
        mm[:, :, slot] = arr
        mm.flush()
        slot += 1
        del arr
        gc.collect()

    for lag in range(6):
        write_feature(f"p_lag{lag}", p_df.shift(lag))

    shannon = -np.log(np.clip(P, CFG.eps, 1.0)).astype(np.float32)
    write_feature("shannon", shannon)

    for label in ["20", "30", "1h", "2h", "4h"]:
        n = WINDOWS[label]
        p_mean = p_df.rolling(n, min_periods=n).mean()
        y_mean = y_df.clip(lower=0).rolling(n, min_periods=n).mean()
        y_std = y_df.clip(lower=0).rolling(n, min_periods=n).std()
        rliq = p_mean / (1.0 + y_std / (y_mean + CFG.eps))
        write_feature(f"RLIQ_{label}", rliq)
        del p_mean, y_mean, y_std, rliq
        gc.collect()

    write_feature(
        "SKEW_p_1m",
        p_df.rolling(WINDOWS["1m"], min_periods=WINDOWS["1m"]).skew(),
    )

    ema_1m = price_df.ewm(span=WINDOWS["1m"], adjust=False).mean()
    n_model = ema_1m.to_numpy(dtype=np.float32) / np.clip(shannon, CFG.eps, None)
    write_feature("n_model", n_model)
    del ema_1m, n_model, shannon
    gc.collect()

    ema_1s = price_df.ewm(span=WINDOWS["1s"], adjust=False).mean()
    ema_2s = price_df.ewm(span=WINDOWS["2s"], adjust=False).mean()
    write_feature("MACD_1s_2s", ema_1s - ema_2s)
    del ema_1s, ema_2s
    gc.collect()

    log_price = np.log(price_df.where(price_df > 0))
    downside = log_price.diff().clip(upper=0)

    def rstab(window: int) -> pd.DataFrame:
        semivol = np.sqrt(downside.pow(2).rolling(window, min_periods=window).mean())
        return 1.0 / (1.0 + semivol)

    write_feature("RSTAB_3m", rstab(WINDOWS["3m"]))
    write_feature(
        "SKEW_p_2m",
        p_df.rolling(WINDOWS["2m"], min_periods=WINDOWS["2m"]).skew(),
    )

    delta = price_df.diff()
    n = WINDOWS["3d"]
    gain = np.sqrt(delta.where(delta > 0, 0.0).rolling(n, min_periods=n).max())
    loss = np.sqrt((-delta.where(delta < 0, 0.0)).rolling(n, min_periods=n).max())
    rs = gain / loss.replace(0, np.nan)
    write_feature("RSIp_3d", 100.0 - 100.0 / (1.0 + rs))
    del delta, gain, loss, rs
    gc.collect()

    write_feature(
        "KURT_p_3m",
        p_df.rolling(WINDOWS["3m"], min_periods=WINDOWS["3m"]).kurt(),
    )
    write_feature("RSTAB_2m", rstab(WINDOWS["2m"]))

    if slot != len(FEATURE_NAMES):
        raise AssertionError(f"Se escribieron {slot} features, se esperaban {len(FEATURE_NAMES)}")
    mm.flush()
    feature_meta_path.write_text(json.dumps(expected_meta, indent=2), encoding="utf-8")
    del p_df, y_df, price_df, log_price, downside
    gc.collect()
    return mm


FEATURES = build_feature_memmap()

# Los factores ya estan materializados; solo P es necesario para targets y grafos.
del Y, PRICE, VOLUME, dfyp, dftu, dfp, dfv
gc.collect()




Construyendo memmap de 0.91 GiB


0

## 4. Cortes temporales, purga y escalado solo con train



In [ ]:
all_sample_t = np.arange(WARMUP, len(P) - CFG.horizon, CFG.sample_stride)
test_start = len(P) - CFG.horizon - CFG.test_size
validation_end = test_start - CFG.purge
validation_start = validation_end - CFG.validation_size
train_end = validation_start - CFG.purge

train_t = all_sample_t[all_sample_t < train_end]
validation_t = all_sample_t[
    (all_sample_t >= validation_start) & (all_sample_t < validation_end)
]
test_t = all_sample_t[all_sample_t >= test_start]

if min(map(len, (train_t, validation_t, test_t))) == 0:
    raise ValueError("Alguno de los cortes temporales quedo vacio.")
if train_t.max() + CFG.horizon >= validation_t.min():
    raise AssertionError("Train y validacion no respetan la purga.")
if validation_t.max() + CFG.horizon >= test_t.min():
    raise AssertionError("Validacion y test no respetan la purga.")


def fit_streaming_scaler(
    mm: np.memmap, sample_times: np.ndarray, chunk_times: int = 64
) -> Tuple[np.ndarray, np.ndarray]:
    sums = np.zeros(mm.shape[-1], dtype=np.float64)
    sums_sq = np.zeros(mm.shape[-1], dtype=np.float64)
    counts = np.zeros(mm.shape[-1], dtype=np.int64)
    for start in range(0, len(sample_times), chunk_times):
        idx = sample_times[start : start + chunk_times]
        block = np.asarray(mm[idx], dtype=np.float32).reshape(-1, mm.shape[-1])
        finite = np.isfinite(block)
        safe = np.where(finite, block, 0.0).astype(np.float64)
        sums += safe.sum(axis=0)
        sums_sq += np.square(safe).sum(axis=0)
        counts += finite.sum(axis=0)
    if np.any(counts == 0):
        bad = [FEATURE_NAMES[i] for i in np.flatnonzero(counts == 0)]
        raise ValueError(f"Features sin observaciones finitas en train: {bad}")
    mean = sums / counts
    var = np.maximum(sums_sq / counts - np.square(mean), 1e-12)
    return mean.astype(np.float32), np.sqrt(var).astype(np.float32)


FEATURE_MEAN, FEATURE_STD = fit_streaming_scaler(FEATURES, train_t)
print("Train:", timestamps[train_t[0]], "->", timestamps[train_t[-1]])
print("Validacion:", timestamps[validation_t[0]], "->", timestamps[validation_t[-1]])
print("Test:", timestamps[test_t[0]], "->", timestamps[test_t[-1]])




Train: 2026-03-26 05:20:00+00:00 -> 2026-06-02 23:20:00+00:00
Validacion: 2026-06-02 23:40:00+00:00 -> 2026-06-16 23:30:00+00:00
Test: 2026-06-16 23:50:00+00:00 -> 2026-06-30 23:40:00+00:00


## 5. Grafo dinamico causal

Correlacion directa de `p` es espuria dentro del simplex. Para cada ancla
semanal se usa exclusivamente la ventana historica terminada en esa ancla:

1. transformacion CLR con reemplazo multiplicativo numerico de ceros;
2. proyeccion aleatoria determinista de las trayectorias CLR estandarizadas;
3. similitud absoluta aproximada entre trayectorias;
4. proximidad energetica NPP `exp(-|E_i-E_j|/MAD)`, con
   `E(t)=-CLR(p(t))`, identificada salvo una constante aditiva;
5. top-k y simetrizacion.

No se estima una matriz de precision de 1.787 x 1.787: con una ventana menor
que el numero de nodos seria singular, y presentarla como correlacion parcial
seria matematicamente incoherente. El sketch CLR conserva la interpretacion
composicional y hace viable el grafo completo.



In [ ]:
class CausalGraphCache:
    def __init__(self, p: np.ndarray, cfg: Config, warmup: int):
        self.p = p
        self.cfg = cfg
        self.warmup = warmup
        self.n_nodes = p.shape[1]
        self.cache: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}

    def anchor_for(self, t: int) -> int:
        if t < self.warmup:
            raise ValueError("t anterior al warmup")
        anchor = self.warmup + ((t - self.warmup) // self.cfg.edge_refresh) * self.cfg.edge_refresh
        if anchor > t:
            raise AssertionError("El grafo intento usar un ancla futura.")
        return int(anchor)

    def _build(self, anchor: int) -> Tuple[np.ndarray, np.ndarray]:
        start = max(0, anchor - self.cfg.edge_window + 1)
        hist = np.asarray(self.p[start : anchor + 1], dtype=np.float64)
        active = np.nanmax(hist, axis=0) > self.cfg.min_active_share
        active_ids = np.flatnonzero(active)

        # Todos los nodos conservan al menos self-loop, incluso si aun no cotizan.
        src_parts = [np.arange(self.n_nodes, dtype=np.int64)]
        dst_parts = [np.arange(self.n_nodes, dtype=np.int64)]
        weight_parts = [np.ones(self.n_nodes, dtype=np.float32)]

        if len(active_ids) > 1:
            logp = np.log(np.clip(hist[:, active], self.cfg.eps, None))
            clr = logp - logp.mean(axis=1, keepdims=True)
            centered = clr - clr.mean(axis=0, keepdims=True)
            scale = centered.std(axis=0, keepdims=True)
            standardized = centered / np.where(scale > self.cfg.eps, scale, 1.0)

            rng = np.random.default_rng(self.cfg.seed)
            projection = rng.choice(
                np.array([-1.0, 1.0], dtype=np.float32),
                size=(len(hist), self.cfg.edge_projection_dim),
            ) / math.sqrt(self.cfg.edge_projection_dim)
            embedding = standardized.T @ projection
            norm = np.linalg.norm(embedding, axis=1, keepdims=True)
            embedding = embedding / np.where(norm > self.cfg.eps, norm, 1.0)
            clr_similarity = np.abs(embedding @ embedding.T)

            if self.cfg.edge_gamma_npp > 0:
                energy = -clr[-1]
                mad = np.median(np.abs(energy - np.median(energy))) + self.cfg.eps
                energy_proximity = np.exp(-np.abs(energy[:, None] - energy[None, :]) / mad)
                score = (
                    self.cfg.edge_alpha_clr * clr_similarity
                    + self.cfg.edge_gamma_npp * energy_proximity
                )
            else:
                # Ablacion pura: el grafo depende solo de CLR; no se calcula energia NPP.
                score = self.cfg.edge_alpha_clr * clr_similarity
            np.fill_diagonal(score, -np.inf)
            k = min(self.cfg.edge_top_k, len(active_ids) - 1)
            nbr = np.argpartition(-score, kth=k - 1, axis=1)[:, :k]
            row = np.arange(len(active_ids))[:, None]
            w = score[row, nbr].astype(np.float32)
            src = np.repeat(active_ids, k)
            dst = active_ids[nbr.reshape(-1)]

            # Direcciones reciprocas: la atencion agregara src -> dst.
            src_parts.extend([src, dst])
            dst_parts.extend([dst, src])
            weight_parts.extend([w.reshape(-1), w.reshape(-1)])

        src_all = np.concatenate(src_parts)
        dst_all = np.concatenate(dst_parts)
        w_all = np.concatenate(weight_parts)
        keys = src_all * self.n_nodes + dst_all
        unique_keys, inverse = np.unique(keys, return_inverse=True)
        unique_w = np.zeros(len(unique_keys), dtype=np.float32)
        np.maximum.at(unique_w, inverse, w_all)
        edge_index = np.vstack(
            [unique_keys // self.n_nodes, unique_keys % self.n_nodes]
        ).astype(np.int64)
        edge_weight = np.clip(unique_w, self.cfg.eps, 1.0).astype(np.float32)
        return edge_index, edge_weight

    def get(self, t: int) -> Tuple[np.ndarray, np.ndarray]:
        anchor = self.anchor_for(t)
        if anchor not in self.cache:
            self.cache[anchor] = self._build(anchor)
        return self.cache[anchor]


GRAPH_CACHE = CausalGraphCache(P, CFG, WARMUP)




## 6. Dataset secuencial y batching de grafos disjuntos



In [ ]:
class CryptoGraphDataset(Dataset):
    def __init__(self, sample_times: np.ndarray, graph_cache=None):
        self.sample_times = np.asarray(sample_times, dtype=np.int64)
        self.graph_cache = GRAPH_CACHE if graph_cache is None else graph_cache

    def __len__(self) -> int:
        return len(self.sample_times)

    def __getitem__(self, idx: int):
        t = int(self.sample_times[idx])
        raw = np.asarray(FEATURES[t], dtype=np.float32).copy()
        x = (raw - FEATURE_MEAN) / FEATURE_STD
        # NaN solo puede provenir de historia insuficiente o precio aun no listado;
        # se imputa a la media de train (cero despues del escalado), nunca con futuro.
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        target = np.asarray(P[t + CFG.horizon], dtype=np.float32).copy()
        target = target / target.sum(dtype=np.float64)
        edge_index, edge_weight = self.graph_cache.get(t)
        return {
            "x": torch.from_numpy(x),
            "target": torch.from_numpy(target),
            "edge_index": torch.from_numpy(edge_index),
            "edge_weight": torch.from_numpy(edge_weight),
            "t": t,
        }


def collate_graphs(samples: Sequence[dict]) -> dict:
    batch_size = len(samples)
    n_nodes = samples[0]["x"].shape[0]
    x = torch.cat([s["x"] for s in samples], dim=0)
    target = torch.stack([s["target"] for s in samples], dim=0)
    edges, weights = [], []
    for batch_id, sample in enumerate(samples):
        edges.append(sample["edge_index"] + batch_id * n_nodes)
        weights.append(sample["edge_weight"])
    return {
        "x": x,
        "target": target,
        "edge_index": torch.cat(edges, dim=1),
        "edge_weight": torch.cat(weights, dim=0),
        "times": np.array([s["t"] for s in samples], dtype=np.int64),
        "batch_size": batch_size,
        "n_nodes": n_nodes,
    }


train_ds = CryptoGraphDataset(train_t)
validation_ds = CryptoGraphDataset(validation_t)
test_ds = CryptoGraphDataset(test_t)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_graphs,
)
validation_loader = DataLoader(
    validation_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_graphs,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_graphs,
)

for loader in (train_loader, validation_loader, test_loader):
    if not isinstance(loader.sampler, SequentialSampler):
        raise AssertionError("Se detecto un sampler no secuencial.")




## 7. GAT ponderado sin dependencias externas

`edge_weight` entra como sesgo logaritmico de la atencion. El softmax de
atencion se calcula por nodo destino; el softmax final se calcula por grafo y
obliga a que la prediccion sea una distribucion.



In [ ]:
class WeightedGATLayer(nn.Module):
    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        heads: int = 1,
        concat: bool = True,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.out_dim = out_dim
        self.heads = heads
        self.concat = concat
        self.dropout = dropout
        self.linear = nn.Linear(in_dim, heads * out_dim, bias=False)
        self.att_src = nn.Parameter(torch.empty(heads, out_dim))
        self.att_dst = nn.Parameter(torch.empty(heads, out_dim))
        self.bias = nn.Parameter(torch.zeros(heads * out_dim if concat else out_dim))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.xavier_uniform_(self.linear.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)

    def forward(
        self, x: torch.Tensor, edge_index: torch.Tensor, edge_weight: torch.Tensor
    ) -> torch.Tensor:
        n = x.shape[0]
        src, dst = edge_index[0], edge_index[1]
        h = self.linear(x).view(n, self.heads, self.out_dim)
        logits = (h[src] * self.att_src).sum(-1) + (h[dst] * self.att_dst).sum(-1)
        logits = F.leaky_relu(logits, negative_slope=0.2)
        logits = logits + torch.log(edge_weight.clamp_min(1e-12)).unsqueeze(-1)

        dst_index = dst.unsqueeze(-1).expand(-1, self.heads)
        max_per_dst = torch.full(
            (n, self.heads), -torch.inf, dtype=logits.dtype, device=logits.device
        )
        max_per_dst.scatter_reduce_(
            0, dst_index, logits, reduce="amax", include_self=True
        )
        exp_logits = torch.exp(logits - max_per_dst[dst])
        denom = torch.zeros((n, self.heads), dtype=logits.dtype, device=logits.device)
        denom.index_add_(0, dst, exp_logits)
        alpha = exp_logits / denom[dst].clamp_min(1e-12)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        messages = alpha.unsqueeze(-1) * h[src]
        out = torch.zeros(
            (n, self.heads, self.out_dim), dtype=h.dtype, device=h.device
        )
        out.index_add_(0, dst, messages)
        if self.concat:
            out = out.reshape(n, self.heads * self.out_dim)
        else:
            out = out.mean(dim=1)
        return out + self.bias


class CryptoGAT(nn.Module):
    def __init__(self, n_features: int, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.gat1 = WeightedGATLayer(
            n_features,
            cfg.hidden_dim,
            heads=cfg.attention_heads,
            concat=True,
            dropout=cfg.dropout,
        )
        wide = cfg.hidden_dim * cfg.attention_heads
        self.norm1 = nn.LayerNorm(wide)
        self.gat2 = WeightedGATLayer(
            wide,
            cfg.hidden_dim,
            heads=1,
            concat=False,
            dropout=cfg.dropout,
        )
        self.skip = nn.Linear(n_features, cfg.hidden_dim)
        self.norm2 = nn.LayerNorm(cfg.hidden_dim)
        self.logit_head = nn.Linear(cfg.hidden_dim, 1)
        self.gp_head = nn.Linear(cfg.hidden_dim, 1)
        self.eta_head = nn.Linear(cfg.hidden_dim, 1)

    def forward(self, batch: dict) -> dict:
        x = batch["x"]
        edge_index = batch["edge_index"]
        edge_weight = batch["edge_weight"]
        b = batch["batch_size"]
        n = batch["n_nodes"]

        h = self.gat1(x, edge_index, edge_weight)
        h = self.norm1(F.elu(h))
        h = F.dropout(h, p=self.cfg.dropout, training=self.training)
        h = self.gat2(h, edge_index, edge_weight)
        h = self.norm2(F.elu(h + self.skip(x)))

        logits = self.logit_head(h).view(b, n)
        gp = torch.sigmoid(self.gp_head(h)).view(b, n)
        eta = F.softplus(self.eta_head(h)).view(b, n) + self.cfg.eps
        energy = gp / eta
        log_p = F.log_softmax(logits, dim=1)
        log_q_npp = F.log_softmax(-energy, dim=1)
        return {
            "logits": logits,
            "log_p": log_p,
            "p": log_p.exp(),
            "gp": gp,
            "eta": eta,
            "energy": energy,
            "log_q_npp": log_q_npp,
            "q_npp": log_q_npp.exp(),
        }




## 8. Loss distributiva y regularizacion NPP

Para el GAT libre:

`L = H(p(t+1), p_hat(t+1))`.

Para GAT+NPP se agrega exactamente el residuo de la solucion cerrada del
paper:

`r_j = log(p_hat_j) + gp_j/eta_j + log Z`,

donde `log Z = logsumexp(-gp/eta)`. Tambien se entrena la distribucion
energetica auxiliar `q_NPP=softmax(-gp/eta)` contra el mismo target. Esto no
usa informacion del test: el target entra solo en el gradiente de las muestras
pertenecientes a train.



In [ ]:
def distribution_ce(target: torch.Tensor, log_prob: torch.Tensor) -> torch.Tensor:
    return -(target * log_prob).sum(dim=1).mean()


def loss_components(output: dict, target: torch.Tensor, use_npp: bool) -> dict:
    pred_ce = distribution_ce(target, output["log_p"])
    zero = pred_ce.new_zeros(())
    if not use_npp:
        return {
            "total": pred_ce,
            "pred_ce": pred_ce,
            "npp_residual": zero,
            "energy_ce": zero,
            "eta_anchor": zero,
            "smooth": zero,
        }

    energy = output["energy"]
    log_z = torch.logsumexp(-energy, dim=1, keepdim=True)
    residual = output["log_p"] + energy + log_z
    npp_residual = residual.square().mean()
    energy_ce = distribution_ce(target, output["log_q_npp"])
    eta_anchor = torch.log(output["eta"].mean(dim=1).clamp_min(CFG.eps)).square().mean()
    if energy.shape[0] > 1:
        smooth = (energy[1:] - energy[:-1]).square().mean()
    else:
        smooth = zero
    total = (
        pred_ce
        + CFG.lambda_npp * npp_residual
        + CFG.lambda_energy_ce * energy_ce
        + CFG.lambda_eta_anchor * eta_anchor
        + CFG.lambda_energy_smooth * smooth
    )
    return {
        "total": total,
        "pred_ce": pred_ce,
        "npp_residual": npp_residual,
        "energy_ce": energy_ce,
        "eta_anchor": eta_anchor,
        "smooth": smooth,
    }


def to_device(batch: dict) -> dict:
    result = dict(batch)
    for key in ("x", "target", "edge_index", "edge_weight"):
        result[key] = result[key].to(DEVICE, non_blocking=True)
    return result


@torch.no_grad()
def validation_ce(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total, count = 0.0, 0
    for batch in loader:
        batch = to_device(batch)
        out = model(batch)
        per_graph = -(batch["target"] * out["log_p"]).sum(dim=1)
        total += per_graph.sum().item()
        count += len(per_graph)
    return total / count


def train_model(
    name: str, use_npp: bool, seed: Optional[int] = None,
    train_data_loader: Optional[DataLoader] = None,
    validation_data_loader: Optional[DataLoader] = None,
) -> Tuple[CryptoGAT, pd.DataFrame]:
    set_seed(CFG.seed if seed is None else seed)
    model = CryptoGAT(len(FEATURE_NAMES), CFG).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay
    )
    best_state: Optional[dict] = None
    best_val = math.inf
    stale = 0
    history = []
    active_train_loader = train_loader if train_data_loader is None else train_data_loader
    active_validation_loader = validation_loader if validation_data_loader is None else validation_data_loader

    for epoch in range(1, CFG.epochs + 1):
        model.train()
        sums: Dict[str, float] = {}
        seen = 0
        for raw_batch in active_train_loader:
            batch = to_device(raw_batch)
            optimizer.zero_grad(set_to_none=True)
            output = model(batch)
            pieces = loss_components(output, batch["target"], use_npp=use_npp)
            pieces["total"].backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            optimizer.step()

            b = batch["batch_size"]
            seen += b
            for key, value in pieces.items():
                sums[key] = sums.get(key, 0.0) + value.detach().item() * b

        val_ce = validation_ce(model, active_validation_loader)
        row = {"model": name, "epoch": epoch, "val_ce": val_ce}
        row.update({f"train_{k}": v / seen for k, v in sums.items()})
        history.append(row)
        print(row)

        if val_ce < best_val - 1e-7:
            best_val = val_ce
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= CFG.patience:
                print(f"Early stopping de {name} en epoch {epoch}.")
                break

    if best_state is None:
        raise RuntimeError("No se genero un checkpoint valido.")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)




## 9. Entrenamiento: GAT libre vs GAT + NPP



In [ ]:
FREE_MODEL, history_free = train_model("GAT libre", use_npp=False)
NPP_MODEL, history_npp = train_model("GAT + NPP", use_npp=True)

{'model': 'GAT libre', 'epoch': 1, 'val_ce': 2.992199237485842, 'train_total': 3.2060565743851184, 'train_pred_ce': 3.2060565743851184, 'train_npp_residual': 0.0, 'train_energy_ce': 0.0, 'train_eta_anchor': 0.0, 'train_smooth': 0.0}
{'model': 'GAT libre', 'epoch': 2, 'val_ce': 2.9863457003939247, 'train_total': 3.0573237293109496, 'train_pred_ce': 3.0573237293109496, 'train_npp_residual': 0.0, 'train_energy_ce': 0.0, 'train_eta_anchor': 0.0, 'train_smooth': 0.0}
{'model': 'GAT libre', 'epoch': 3, 'val_ce': 2.979345407027159, 'train_total': 3.051002669694434, 'train_pred_ce': 3.051002669694434, 'train_npp_residual': 0.0, 'train_energy_ce': 0.0, 'train_eta_anchor': 0.0, 'train_smooth': 0.0}
{'model': 'GAT libre', 'epoch': 4, 'val_ce': 2.9755222649133577, 'train_total': 3.048491597647479, 'train_pred_ce': 3.048491597647479, 'train_npp_residual': 0.0, 'train_energy_ce': 0.0, 'train_eta_anchor': 0.0, 'train_smooth': 0.0}
{'model': 'GAT libre', 'epoch': 5, 'val_ce': 2.971604663110399, 'train

## 10. Test final y benchmarks

El test se consume por primera vez aqui. Ademas de CE/KL se reporta distancia
de Aitchison, apropiada para datos composicionales. La persistencia usa
`p(t)` para predecir `p(t+1)`.



In [ ]:
@torch.no_grad()
def collect_predictions(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    p_list, q_list, gp_list, eta_list, y_list, t_list = [], [], [], [], [], []
    for raw_batch in loader:
        times_batch = raw_batch["times"]
        batch = to_device(raw_batch)
        out = model(batch)
        p_list.append(out["p"].cpu().numpy())
        q_list.append(out["q_npp"].cpu().numpy())
        gp_list.append(out["gp"].cpu().numpy())
        eta_list.append(out["eta"].cpu().numpy())
        y_list.append(batch["target"].cpu().numpy())
        t_list.append(times_batch)
    return {
        "p": np.concatenate(p_list),
        "q_npp": np.concatenate(q_list),
        "gp": np.concatenate(gp_list),
        "eta": np.concatenate(eta_list),
        "target": np.concatenate(y_list),
        "t": np.concatenate(t_list),
    }


def clr(values: np.ndarray, eps: float) -> np.ndarray:
    safe = np.clip(values.astype(np.float64), eps, None)
    safe /= safe.sum(axis=1, keepdims=True)
    log_values = np.log(safe)
    return log_values - log_values.mean(axis=1, keepdims=True)


def distribution_metrics(name: str, target: np.ndarray, pred: np.ndarray) -> dict:
    pred = np.clip(pred.astype(np.float64), CFG.eps, None)
    pred /= pred.sum(axis=1, keepdims=True)
    target = np.clip(target.astype(np.float64), 0.0, None)
    target /= target.sum(axis=1, keepdims=True)
    target_safe = np.clip(target, CFG.eps, None)
    ce = -np.sum(target * np.log(pred), axis=1)
    entropy = -np.sum(target * np.log(target_safe), axis=1)
    midpoint = 0.5 * (target_safe + pred)
    js = 0.5 * np.sum(target * (np.log(target_safe) - np.log(midpoint)), axis=1)
    js += 0.5 * np.sum(pred * (np.log(pred) - np.log(midpoint)), axis=1)
    aitchison = np.sqrt(np.mean(np.square(clr(target, CFG.eps) - clr(pred, CFG.eps)), axis=1))
    return {
        "Modelo": name,
        "CE": ce.mean(),
        "KL": (ce - entropy).mean(),
        "JS": js.mean(),
        "Aitchison": aitchison.mean(),
        "MAE_nodo": np.mean(np.abs(target - pred)),
        "RMSE_nodo": np.sqrt(np.mean(np.square(target - pred))),
        "Error_cierre_max": np.max(np.abs(pred.sum(axis=1) - 1.0)),
        "N_test": len(target),
    }


pred_free = collect_predictions(FREE_MODEL, test_loader)
pred_npp = collect_predictions(NPP_MODEL, test_loader)
if not np.array_equal(pred_free["t"], pred_npp["t"]):
    raise AssertionError("Los modelos no fueron evaluados en las mismas fechas.")
if not np.allclose(pred_free["target"], pred_npp["target"]):
    raise AssertionError("Los modelos no comparten exactamente el mismo target.")

target_test = pred_free["target"]
persistence = P[pred_free["t"]]
metrics = pd.DataFrame(
    [
        distribution_metrics("Persistencia", target_test, persistence),
        distribution_metrics("GAT libre", target_test, pred_free["p"]),
        distribution_metrics("GAT + NPP", target_test, pred_npp["p"]),
        distribution_metrics("Cabeza energetica NPP", target_test, pred_npp["q_npp"]),
    ]
).sort_values("CE")
display(metrics)


def node_error_metrics(name: str, target: np.ndarray, pred: np.ndarray) -> pd.DataFrame:
    """Metricas de error por activo sobre exactamente el mismo bloque temporal de test."""
    pred = np.clip(pred.astype(np.float64), CFG.eps, None)
    pred /= pred.sum(axis=1, keepdims=True)
    target = np.clip(target.astype(np.float64), 0.0, None)
    target /= target.sum(axis=1, keepdims=True)
    error = pred - target
    clr_error = clr(pred, CFG.eps) - clr(target, CFG.eps)
    mean_share = np.mean(target, axis=0)
    return pd.DataFrame(
        {
            "Modelo": name,
            "Nodo": assets,
            "MAE": np.mean(np.abs(error), axis=0),
            "RMSE": np.sqrt(np.mean(np.square(error), axis=0)),
            "Sesgo": np.mean(error, axis=0),
            "Participacion_media_real": mean_share,
            "MAE_relativa": np.mean(np.abs(error), axis=0) / np.maximum(mean_share, CFG.eps),
            "RMSE_CLR": np.sqrt(np.mean(np.square(clr_error), axis=0)),
            "Tasa_masa_casi_cero": np.mean(target <= 1e-8, axis=0),
            "N_test": target.shape[0],
        }
    )


metrics_by_node = pd.concat(
    [
        node_error_metrics("Persistencia", target_test, persistence),
        node_error_metrics("GAT libre", target_test, pred_free["p"]),
        node_error_metrics("GAT + NPP", target_test, pred_npp["p"]),
        node_error_metrics("Cabeza energetica NPP", target_test, pred_npp["q_npp"]),
    ],
    ignore_index=True,
)
display(metrics_by_node.sort_values(["Nodo", "RMSE", "MAE"]))




,Modelo,CE,KL,JS,Aitchison,MAE_nodo,RMSE_nodo,Error_cierre_max,N_test
2,GAT + NPP,2.974283,0.181977,0.038915,4.687773,0.000841,0.007264,4.440892e-16,2016
3,Cabeza energetica NPP,2.974413,0.182107,0.039093,4.676658,0.000843,0.007271,3.330669e-16,2016
1,GAT libre,2.974996,0.182690,0.038981,4.634165,0.000842,0.007277,3.330669e-16,2016
0,Persistencia,3.056244,0.263938,0.051146,2.777890,0.000969,0.008658,4.440892e-16,2016


## 11. Auditoria final y persistencia de resultados



In [ ]:
def audit_no_leakage() -> dict:
    feature_text = " ".join(FEATURE_NAMES).lower()
    checks = {
        "sin_target_en_features": "target" not in feature_text and "t+1" not in feature_text,
        "horizonte_positivo": CFG.horizon > 0,
        "train_antes_validacion": train_t.max() + CFG.horizon < validation_t.min(),
        "validacion_antes_test": validation_t.max() + CFG.horizon < test_t.min(),
        "train_sin_shuffle": isinstance(train_loader.sampler, SequentialSampler),
        "simplex_target": bool(np.allclose(target_test.sum(axis=1), 1.0, atol=1e-6)),
        "grafo_causal_train": all(GRAPH_CACHE.anchor_for(int(t)) <= t for t in train_t),
        "grafo_causal_validation": all(GRAPH_CACHE.anchor_for(int(t)) <= t for t in validation_t),
        "grafo_causal_test": all(GRAPH_CACHE.anchor_for(int(t)) <= t for t in test_t),
    }
    checks = {key: bool(value) for key, value in checks.items()}
    if not all(checks.values()):
        raise AssertionError(f"Fallo la auditoria anti-leakage: {checks}")
    return checks


audit = audit_no_leakage()
print(json.dumps(audit, indent=2))

results_dir = Path(CFG.results_dir)
results_dir.mkdir(parents=True, exist_ok=True)
metrics.to_csv(results_dir / "metricas_test.csv", index=False)
metrics_by_node.to_csv(results_dir / "metricas_test_por_nodo.csv", index=False)
for model_name, file_name in {
    "Persistencia": "metricas_por_nodo_persistencia.csv",
    "GAT libre": "metricas_por_nodo_gat_libre.csv",
    "GAT + NPP": "metricas_por_nodo_gat_npp.csv",
    "Cabeza energetica NPP": "metricas_por_nodo_cabeza_energetica_npp.csv",
}.items():
    metrics_by_node.loc[metrics_by_node["Modelo"] == model_name].to_csv(
        results_dir / file_name, index=False
    )
pd.concat([history_free, history_npp], ignore_index=True).to_csv(
    results_dir / "historial_entrenamiento.csv", index=False
)
torch.save(FREE_MODEL.state_dict(), results_dir / "gat_libre.pt")
torch.save(NPP_MODEL.state_dict(), results_dir / "gat_npp.pt")
np.savez_compressed(
    results_dir / "predicciones_test.npz",
    assets=np.asarray(assets),
    t=pred_free["t"],
    timestamps=np.asarray(timestamps[pred_free["t"] + CFG.horizon].astype(str)),
    target=target_test.astype(np.float32),
    persistence=persistence.astype(np.float32),
    gat_free=pred_free["p"].astype(np.float32),
    gat_npp=pred_npp["p"].astype(np.float32),
    q_npp=pred_npp["q_npp"].astype(np.float32),
    gp_latent=pred_npp["gp"].astype(np.float32),
    eta_latent=pred_npp["eta"].astype(np.float32),
)
(results_dir / "config.json").write_text(
    json.dumps(asdict(CFG), indent=2, ensure_ascii=False), encoding="utf-8"
)
(results_dir / "auditoria_no_leakage.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Resultados guardados en:", results_dir)




## Nota de identificacion cientifica

El NPP determina exactamente la energia relativa
`E_j = gp_j / eta_j = -log(p_j) - log(Z)` y, por la restriccion de cierre,
`p=softmax(-E)`. Con cuotas y volumenes transaccionales se puede contrastar
esa estructura energetica y su utilidad predictiva. No obstante, separar
causalmente `gp` y `eta` requiere proxies independientes de los capitales
economico, social, estructural, simbolico y estrategico. Por eso este notebook
guarda ambas cabezas como variables **latentes** y centra la comparacion
cientifica en:

1. GAT libre vs GAT regularizado por NPP;
2. estabilidad fuera de muestra en el test cronologico;
3. CE/KL/JS y distancia de Aitchison, no MSE aislado;
4. coherencia del simplex y del grafo CLR causal.


## Nota cientifica de la ejecucion de referencia

Sobre 2.016 instantes de test cronologico y 471 nodos, el GAT + NPP redujo frente a persistencia aproximadamente 31% el KL, 24% el JS, 13% el MAE y 16% el RMSE. Supero a persistencia en RMSE en 427 de los 440 nodos activos, por lo que la mejora no se limita a BTC ni a los activos de mayor participacion.

El error cuadratico, sin embargo, esta fuertemente concentrado: el nodo `U` aporta cerca de 51%, BTC cerca de 23% y los cinco primeros nodos cerca de 88% del total. Por ello, el RMSE agregado describe principalmente los nodos dominantes y los errores extremos. La identidad economica de `U` debe auditarse contra la fuente antes de extraer conclusiones sustantivas.

Frente al GAT libre, NPP gana en RMSE en 246 de 440 nodos activos, pero en MAE solo en 163. La ventaja agregada del NPP parece provenir de reducir errores grandes y mejorar nodos economicamente relevantes, no de dominar universalmente el error absoluto cotidiano. Esta evidencia es favorable pero debe confirmarse con varias semillas y ventanas temporales.

La divergencia de Aitchison no contradice las mejoras en CE, KL, JS, MAE y RMSE. Aitchison pondera simetricamente discrepancias de log-ratios y es muy sensible a masas nulas o diminutas. Como Softmax produce probabilidades estrictamente positivas, la cola de nodos intermitentes puede empeorar Aitchison aunque mejore la asignacion de la masa economicamente observada. RMSE tampoco debe interpretarse como un intervalo simetrico o una banda de confianza.

La celda final del script genera ademas `nota_cientifica.json` directamente desde cada nueva corrida, evitando que la interpretacion dependa de cifras escritas manualmente.


## 12. Ablación pre-registrada: diez semillas y cuatro modelos

Esta sección contrasta, sobre exactamente los mismos cortes temporales y las mismas diez semillas emparejadas:

1. **Persistencia:** predice p(t+1)=p(t).
2. **GAT 100% libre:** loss predictiva pura y aristas construidas exclusivamente con similitud CLR. No calcula ni usa proximidad energética NPP.
3. **GAT NPP solo aristas:** loss predictiva pura sobre el grafo híbrido original (75% CLR y 25% proximidad energética).
4. **GAT + NPP:** mismo grafo híbrido y regularización NPP completa.

El dataset, features, purga, scaler de train, ventanas, top-k, arquitectura, optimizador, early stopping y test permanecen fijos. En cada semilla los tres GAT parten de la misma inicialización; solo cambia el tratamiento NPP. El test no participa en selección ni early stopping.

In [ ]:
from pathlib import Path

SEEDS = [11, 23, 42, 67, 101, 137, 211, 307, 401, 503]
FORCE_RERUN = False
ACTIVE_SHARE_THRESHOLD = 1e-8
BOOTSTRAP_REPS = 20_000
ABLATION_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_ablation_10_seeds"
)
ABLATION_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# Grafo totalmente libre de NPP: solo dependencia composicional CLR.
PURE_CFG = Config(**{
    **asdict(CFG),
    "edge_alpha_clr": 1.0,
    "edge_gamma_npp": 0.0,
})
PURE_GRAPH_CACHE = CausalGraphCache(P, PURE_CFG, WARMUP)

pure_train_ds = CryptoGraphDataset(train_t, graph_cache=PURE_GRAPH_CACHE)
pure_validation_ds = CryptoGraphDataset(validation_t, graph_cache=PURE_GRAPH_CACHE)
pure_test_ds = CryptoGraphDataset(test_t, graph_cache=PURE_GRAPH_CACHE)

pure_train_loader = DataLoader(
    pure_train_ds, batch_size=CFG.batch_size, shuffle=False,
    num_workers=0, collate_fn=collate_graphs,
)
pure_validation_loader = DataLoader(
    pure_validation_ds, batch_size=CFG.batch_size, shuffle=False,
    num_workers=0, collate_fn=collate_graphs,
)
pure_test_loader = DataLoader(
    pure_test_ds, batch_size=CFG.batch_size, shuffle=False,
    num_workers=0, collate_fn=collate_graphs,
)

print("Semillas emparejadas:", SEEDS)
print("Grafo hibrido:", CFG.edge_alpha_clr, CFG.edge_gamma_npp)
print("Grafo libre:", PURE_CFG.edge_alpha_clr, PURE_CFG.edge_gamma_npp)

In [ ]:
def aitchison_distance(target: np.ndarray, pred: np.ndarray) -> float:
    if target.shape != pred.shape or target.ndim != 2:
        raise ValueError("Aitchison requiere matrices 2D con la misma forma.")
    if target.shape[1] < 2:
        raise ValueError("Aitchison requiere al menos dos nodos activos.")
    clr_error = clr(target, CFG.eps) - clr(pred, CFG.eps)
    per_time = np.sqrt(np.mean(np.square(clr_error), axis=1))
    return float(per_time.mean())


def seed_run_complete(run_dir: Path) -> bool:
    required = [
        "metricas_globales.csv", "metricas_por_nodo.csv",
        "historial_gat_libre_100pct.csv", "historial_gat_npp_aristas.csv",
        "historial_gat_npp.csv", "predicciones_test.npz",
    ]
    return all((run_dir / name).exists() for name in required)


def run_seed_experiment(seed: int, force: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    run_dir = ABLATION_RESULTS_ROOT / f"seed_{seed:04d}"
    run_dir.mkdir(parents=True, exist_ok=True)
    if seed_run_complete(run_dir) and not force:
        return (
            pd.read_csv(run_dir / "metricas_globales.csv"),
            pd.read_csv(run_dir / "metricas_por_nodo.csv"),
        )

    # Misma semilla => misma inicializacion para una comparacion emparejada.
    pure_model, pure_history = train_model(
        "GAT 100% libre", use_npp=False, seed=seed,
        train_data_loader=pure_train_loader,
        validation_data_loader=pure_validation_loader,
    )
    edge_model, edge_history = train_model(
        "GAT NPP solo aristas", use_npp=False, seed=seed,
        train_data_loader=train_loader,
        validation_data_loader=validation_loader,
    )
    npp_model, npp_history = train_model(
        "GAT + NPP", use_npp=True, seed=seed,
        train_data_loader=train_loader,
        validation_data_loader=validation_loader,
    )

    pure_prediction = collect_predictions(pure_model, pure_test_loader)
    edge_prediction = collect_predictions(edge_model, test_loader)
    npp_prediction = collect_predictions(npp_model, test_loader)
    if not (
        np.array_equal(pure_prediction["t"], edge_prediction["t"])
        and np.array_equal(edge_prediction["t"], npp_prediction["t"])
    ):
        raise AssertionError("Los modelos no fueron evaluados en las mismas fechas.")

    target = pure_prediction["target"]
    persistence_prediction = P[pure_prediction["t"]]
    predictions = {
        "Persistencia": persistence_prediction,
        "GAT 100% libre": pure_prediction["p"],
        "GAT NPP solo aristas": edge_prediction["p"],
        "GAT + NPP": npp_prediction["p"],
    }
    global_frame = pd.DataFrame([
        {**distribution_metrics(name, target, pred), "seed": seed}
        for name, pred in predictions.items()
    ])
    active_mask = target.mean(axis=0) >= ACTIVE_SHARE_THRESHOLD
    for row_index, pred in enumerate(predictions.values()):
        global_frame.loc[row_index, "Aitchison_activos"] = aitchison_distance(
            target[:, active_mask], pred[:, active_mask]
        )
        global_frame.loc[row_index, "N_nodos_activos"] = int(active_mask.sum())

    node_frame = pd.concat([
        node_error_metrics(name, target, pred).assign(seed=seed)
        for name, pred in predictions.items()
    ], ignore_index=True)

    global_frame.to_csv(run_dir / "metricas_globales.csv", index=False)
    node_frame.to_csv(run_dir / "metricas_por_nodo.csv", index=False)
    pure_history.to_csv(run_dir / "historial_gat_libre_100pct.csv", index=False)
    edge_history.to_csv(run_dir / "historial_gat_npp_aristas.csv", index=False)
    npp_history.to_csv(run_dir / "historial_gat_npp.csv", index=False)
    torch.save(pure_model.state_dict(), run_dir / "gat_libre_100pct.pt")
    torch.save(edge_model.state_dict(), run_dir / "gat_npp_solo_aristas.pt")
    torch.save(npp_model.state_dict(), run_dir / "gat_npp.pt")
    np.savez_compressed(
        run_dir / "predicciones_test.npz", t=pure_prediction["t"], target=target,
        persistencia=persistence_prediction, gat_libre=pure_prediction["p"],
        gat_npp_aristas=edge_prediction["p"], gat_npp=npp_prediction["p"],
    )
    del pure_model, edge_model, npp_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return global_frame, node_frame

In [ ]:
ABLATION_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
all_global, all_nodes = [], []
for position, seed in enumerate(SEEDS, start=1):
    print(f"[{position}/{len(SEEDS)}] Semilla {seed}")
    global_frame, node_frame = run_seed_experiment(seed, force=FORCE_RERUN)
    all_global.append(global_frame)
    all_nodes.append(node_frame)

metrics_seeds = pd.concat(all_global, ignore_index=True)
metrics_nodes_seeds = pd.concat(all_nodes, ignore_index=True)
metrics_seeds.to_csv(ABLATION_RESULTS_ROOT / "metricas_globales_por_semilla.csv", index=False)
metrics_nodes_seeds.to_csv(
    ABLATION_RESULTS_ROOT / "metricas_por_nodo_y_semilla.csv", index=False
)
display(metrics_seeds.sort_values(["seed", "CE"]))

In [ ]:
CONTRASTS = [
    ("GAT + NPP", "GAT NPP solo aristas"),
    ("GAT NPP solo aristas", "GAT 100% libre"),
    ("GAT + NPP", "GAT 100% libre"),
    ("GAT 100% libre", "Persistencia"),
    ("GAT NPP solo aristas", "Persistencia"),
    ("GAT + NPP", "Persistencia"),
]
CONTRAST_METRICS = [
    "CE", "KL", "JS", "MAE_nodo", "RMSE_nodo", "Aitchison", "Aitchison_activos"
]


def bootstrap_mean_ci(values: np.ndarray, reps: int, seed: int = 20260831) -> tuple[float, float]:
    values = np.asarray(values, dtype=np.float64)
    if len(values) == 1:
        return float(values[0]), float(values[0])
    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(reps, len(values)))
    bootstrap_means = values[sample_indices].mean(axis=1)
    low, high = np.quantile(bootstrap_means, [0.025, 0.975])
    return float(low), float(high)


wide = metrics_seeds.pivot(index="seed", columns="Modelo", values=CONTRAST_METRICS)
contrast_rows, delta_rows = [], []
for model_a, model_b in CONTRASTS:
    for metric in CONTRAST_METRICS:
        delta = wide[(metric, model_a)] - wide[(metric, model_b)]
        low, high = bootstrap_mean_ci(delta.to_numpy(), BOOTSTRAP_REPS)
        contrast_rows.append(
            {
                "modelo_A": model_a,
                "modelo_B": model_b,
                "metrica": metric,
                "delta": "A - B; negativo favorece A",
                "media_delta": delta.mean(),
                "mediana_delta": delta.median(),
                "std_delta": delta.std(ddof=1),
                "ic95_bootstrap_inferior": low,
                "ic95_bootstrap_superior": high,
                "semillas_ganadas_A": int((delta < 0).sum()),
                "n_semillas": len(delta),
            }
        )
        delta_rows.extend(
            {
                "seed": int(run_seed),
                "modelo_A": model_a,
                "modelo_B": model_b,
                "metrica": metric,
                "delta_A_menos_B": float(value),
            }
            for run_seed, value in delta.items()
        )

contrast_summary = pd.DataFrame(contrast_rows)
contrast_deltas = pd.DataFrame(delta_rows)
contrast_summary.to_csv(ABLATION_RESULTS_ROOT / "contraste_emparejado_resumen.csv", index=False)
contrast_deltas.to_csv(ABLATION_RESULTS_ROOT / "deltas_por_semilla.csv", index=False)
model_summary = (
    metrics_seeds.groupby("Modelo", sort=False)[CONTRAST_METRICS]
    .agg(["mean", "std", "median", "min", "max"])
)
model_summary.columns = [f"{metric}_{stat}" for metric, stat in model_summary.columns]
model_summary.reset_index().to_csv(ABLATION_RESULTS_ROOT / "resumen_por_modelo.csv", index=False)
display(contrast_summary)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, metric in zip(axes.ravel(), ["CE", "KL", "JS", "MAE_nodo", "RMSE_nodo", "Aitchison"]):
    groups = [
        frame[metric].to_numpy()
        for _, frame in metrics_seeds.groupby("Modelo", sort=False)
    ]
    labels = list(metrics_seeds["Modelo"].drop_duplicates())
    ax.boxplot(groups, labels=labels, showmeans=True)
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=25)
fig.suptitle("Robustez por semillas: distribución de métricas de test")
fig.tight_layout()
fig.savefig(ABLATION_RESULTS_ROOT / "boxplots_metricas_por_semilla.png", dpi=180, bbox_inches="tight")
plt.show()

## 13. Lectura pre-registrada del contraste

- **GAT NPP solo aristas vs GAT 100% libre** identifica el aporte de la proximidad energética en la topología.
- **GAT + NPP vs GAT NPP solo aristas** identifica el aporte marginal de la regularización y la cabeza energética durante el entrenamiento.
- **GAT + NPP vs GAT 100% libre** mide el efecto NPP total.
- Cada comparación es emparejada por semilla y usa el mismo test cronológico.

No se declarará superioridad por una sola media. Se reportarán media, mediana, dispersión, victorias por semilla e intervalo bootstrap del delta. Diez semillas miden estabilidad de optimización; la robustez temporal requerirá después walk-forward con múltiples ventanas y scaler/grafo recalculados exclusivamente con el pasado.

## 14. Extensiones experimentales: opciones 3, 5, 6 y 7

Este bloque amplía la ablación de diez semillas sin modificar sus resultados. Las cuatro familias quedan separadas:

1. **Opción 3 — intensidad del mecanismo:** grilla causal de peso energético y fuerza de regularización NPP.
2. **Opción 5 — identificación interna:** ablaciones de cada término y cabeza energética autónoma.
3. **Opción 6 — coherencia composicional:** frontera entre CE/KL y pérdida geométrica CLR/Aitchison.
4. **Opción 7 — auditoría y sensibilidad:** nodos dominantes, cola, exclusiones y concentración del error.

Las opciones 3, 5 y 6 seleccionan exclusivamente con validación. El test actual permanece congelado y no decide configuraciones. La opción 7 es descriptiva y reutiliza predicciones de test ya generadas; no retroalimenta entrenamiento ni selección.

In [ ]:
from dataclasses import replace

EXTENDED_ROOT = Path(
    "/content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1"
)
EXTENDED_ROOT.mkdir(parents=True, exist_ok=True)

EXTENDED_SEEDS = [11, 23, 42]
EXPERIMENT_VERSION = 1

# Los tres bloques siguientes multiplican entrenamientos. Se activan por separado.
RUN_OPTION_3_INTENSITY = False
RUN_OPTION_5_IDENTIFICATION = False
RUN_OPTION_6_COMPOSITIONAL = True

# Esta auditoria no reentrena: usa las predicciones de las diez semillas ya guardadas.
RUN_OPTION_7_SENSITIVITY = True

print("Raiz de extensiones:", EXTENDED_ROOT)
print("Semillas de desarrollo:", EXTENDED_SEEDS)

### Infraestructura común de desarrollo

Las funciones siguientes permiten variar grafo y términos de la loss sin tocar el experimento confirmado. Cada corrida usa loaders cronológicos, reinicia la misma semilla y aplica early stopping solo sobre validación. Al terminar cada combinación se guardan atómicamente el modelo, el historial completo, la configuración, las métricas y un manifiesto `COMPLETED`. Si Colab se desconecta, la próxima ejecución recupera cada combinación terminada y continúa desde la primera pendiente, sin volver a entrenar las anteriores.

In [ ]:
EXTENDED_LOADER_CACHE = {}


def make_graph_config(edge_gamma_npp: float) -> Config:
    gamma = float(edge_gamma_npp)
    if not 0.0 <= gamma <= 1.0:
        raise ValueError("edge_gamma_npp debe pertenecer a [0, 1].")
    return replace(
        CFG,
        edge_alpha_clr=1.0 - gamma,
        edge_gamma_npp=gamma,
    )


def get_extended_loaders(edge_gamma_npp: float):
    key = round(float(edge_gamma_npp), 8)
    if key in EXTENDED_LOADER_CACHE:
        return EXTENDED_LOADER_CACHE[key]

    cfg = make_graph_config(key)
    graph_cache = CausalGraphCache(P, cfg, WARMUP)
    datasets = {
        "train": CryptoGraphDataset(train_t, graph_cache=graph_cache),
        "validation": CryptoGraphDataset(validation_t, graph_cache=graph_cache),
        "test": CryptoGraphDataset(test_t, graph_cache=graph_cache),
    }
    loaders = {
        name: DataLoader(
            dataset,
            batch_size=cfg.batch_size,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_graphs,
        )
        for name, dataset in datasets.items()
    }
    EXTENDED_LOADER_CACHE[key] = (cfg, loaders)
    return cfg, loaders


def torch_clr(probabilities: torch.Tensor, eps: float) -> torch.Tensor:
    safe = probabilities.clamp_min(eps)
    safe = safe / safe.sum(dim=1, keepdim=True)
    log_safe = safe.log()
    return log_safe - log_safe.mean(dim=1, keepdim=True)


def full_npp_weights(scale: float = 1.0, geometry: float = 0.0) -> dict:
    factor = float(scale)
    return {
        "pred_ce": 1.0,
        "npp_residual": CFG.lambda_npp * factor,
        "energy_ce": CFG.lambda_energy_ce * factor,
        "eta_anchor": CFG.lambda_eta_anchor * factor,
        "smooth": CFG.lambda_energy_smooth * factor,
        "geometry": float(geometry),
    }


def extended_loss_components(output: dict, target: torch.Tensor, weights: dict, cfg: Config) -> dict:
    pred_ce = distribution_ce(target, output["log_p"])
    energy_ce = distribution_ce(target, output["log_q_npp"])
    energy = output["energy"]
    log_z = torch.logsumexp(-energy, dim=1, keepdim=True)
    residual = output["log_p"] + energy + log_z
    npp_residual = residual.square().mean()
    eta_anchor = torch.log(output["eta"].mean(dim=1).clamp_min(cfg.eps)).square().mean()
    smooth = (
        (energy[1:] - energy[:-1]).square().mean()
        if energy.shape[0] > 1
        else pred_ce.new_zeros(())
    )
    geometry = (
        torch_clr(target, cfg.eps) - torch_clr(output["p"], cfg.eps)
    ).square().mean()
    pieces = {
        "pred_ce": pred_ce,
        "npp_residual": npp_residual,
        "energy_ce": energy_ce,
        "eta_anchor": eta_anchor,
        "smooth": smooth,
        "geometry": geometry,
    }
    total = sum(float(weights.get(name, 0.0)) * value for name, value in pieces.items())
    return {"total": total, **pieces}


@torch.no_grad()
def evaluate_variant(model: nn.Module, loader: DataLoader, prediction_key: str, cfg: Config) -> dict:
    if prediction_key not in {"p", "q_npp"}:
        raise ValueError("prediction_key debe ser 'p' o 'q_npp'.")
    log_key = "log_p" if prediction_key == "p" else "log_q_npp"
    model.eval()
    ce_sum = 0.0
    clr_mse_sum = 0.0
    aitchison_sum = 0.0
    count = 0
    for raw_batch in loader:
        batch = to_device(raw_batch)
        output = model(batch)
        target = batch["target"]
        prediction = output[prediction_key]
        per_graph_ce = -(target * output[log_key]).sum(dim=1)
        clr_error = torch_clr(target, cfg.eps) - torch_clr(prediction, cfg.eps)
        per_graph_clr_mse = clr_error.square().mean(dim=1)
        batch_n = len(per_graph_ce)
        ce_sum += per_graph_ce.sum().item()
        clr_mse_sum += per_graph_clr_mse.sum().item()
        aitchison_sum += per_graph_clr_mse.sqrt().sum().item()
        count += batch_n
    return {
        "CE": ce_sum / count,
        "CLR_MSE": clr_mse_sum / count,
        "Aitchison": aitchison_sum / count,
        "N": count,
    }


def train_extended_variant(
    name: str,
    cfg: Config,
    loaders: dict,
    weights: dict,
    seed: int,
    prediction_key: str = "p",
):
    set_seed(seed)
    model = CryptoGAT(len(FEATURE_NAMES), cfg).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay
    )
    best_state = None
    best_objective = math.inf
    stale = 0
    history = []

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        sums = {}
        seen = 0
        for raw_batch in loaders["train"]:
            batch = to_device(raw_batch)
            optimizer.zero_grad(set_to_none=True)
            output = model(batch)
            pieces = extended_loss_components(output, batch["target"], weights, cfg)
            pieces["total"].backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            batch_n = batch["batch_size"]
            seen += batch_n
            for key, value in pieces.items():
                sums[key] = sums.get(key, 0.0) + value.detach().item() * batch_n

        val = evaluate_variant(model, loaders["validation"], prediction_key, cfg)
        # El criterio refleja exactamente CE + lambda geometrica * CLR_MSE.
        val_objective = val["CE"] + float(weights.get("geometry", 0.0)) * val["CLR_MSE"]
        row = {
            "model": name,
            "seed": seed,
            "epoch": epoch,
            "val_objective": val_objective,
            "val_ce": val["CE"],
            "val_aitchison": val["Aitchison"],
            "val_clr_mse": val["CLR_MSE"],
        }
        row.update({f"train_{key}": value / seen for key, value in sums.items()})
        history.append(row)

        if val_objective < best_objective - 1e-7:
            best_objective = val_objective
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= cfg.patience:
                break

    if best_state is None:
        raise RuntimeError(f"No se genero checkpoint para {name}.")
    model.load_state_dict(best_state)
    final_validation = evaluate_variant(model, loaders["validation"], prediction_key, cfg)
    return model, pd.DataFrame(history), final_validation


def upsert_experiment_row(path: Path, row: dict, keys: list[str]) -> pd.DataFrame:
    new_row = pd.DataFrame([row])
    if path.exists():
        frame = pd.read_csv(path)
        for key in keys:
            frame = frame.loc[frame[key].astype(str) != str(row[key])] if len(keys) == 1 else frame
        if len(keys) > 1:
            duplicate = np.ones(len(frame), dtype=bool)
            for key in keys:
                duplicate &= frame[key].astype(str).to_numpy() == str(row[key])
            frame = frame.loc[~duplicate]
        frame = pd.concat([frame, new_row], ignore_index=True)
    else:
        frame = new_row
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)
    return frame


def completed_keys(path: Path, keys: list[str]) -> set[tuple]:
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    return set(map(tuple, frame[keys].astype(str).to_numpy()))


def persist_extended_run(
    run_dir: Path, model: nn.Module, history: pd.DataFrame, row: dict,
    weights: dict, cfg: Config, prediction_key: str,
) -> None:
    """Guarda atomicamente todos los artefactos; COMPLETED es siempre el ultimo archivo."""
    run_dir.mkdir(parents=True, exist_ok=True)
    model_tmp = run_dir / "model_state.pt.tmp"
    torch.save(model.state_dict(), model_tmp)
    os.replace(model_tmp, run_dir / "model_state.pt")

    history_tmp = run_dir / "training_history.csv.tmp"
    history.to_csv(history_tmp, index=False)
    os.replace(history_tmp, run_dir / "training_history.csv")

    payloads = {
        "result.json": row,
        "specification.json": {
            "weights": {key: float(value) for key, value in weights.items()},
            "config": asdict(cfg),
            "prediction_key": prediction_key,
            "feature_names": list(FEATURE_NAMES),
        },
    }
    for filename, payload in payloads.items():
        temporary = run_dir / f"{filename}.tmp"
        temporary.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
        os.replace(temporary, run_dir / filename)

    completed_tmp = run_dir / "COMPLETED.tmp"
    completed_tmp.write_text("ok\n", encoding="utf-8")
    os.replace(completed_tmp, run_dir / "COMPLETED")

    registry_row = {
        **row,
        "run_id": run_dir.name,
        "artifact_directory": str(run_dir),
        "model_file": str(run_dir / "model_state.pt"),
        "history_file": str(run_dir / "training_history.csv"),
        "specification_file": str(run_dir / "specification.json"),
        "completed": True,
        "hidden_dim": cfg.hidden_dim,
        "attention_heads": cfg.attention_heads,
        "dropout": cfg.dropout,
        "batch_size": cfg.batch_size,
        "epochs_max": cfg.epochs,
        "patience": cfg.patience,
        "learning_rate": cfg.learning_rate,
        "weight_decay": cfg.weight_decay,
        "grad_clip": cfg.grad_clip,
        "edge_top_k": cfg.edge_top_k,
        "edge_alpha_clr_cfg": cfg.edge_alpha_clr,
        "edge_gamma_npp_cfg": cfg.edge_gamma_npp,
        "lambda_npp_cfg": cfg.lambda_npp,
        "lambda_energy_ce_cfg": cfg.lambda_energy_ce,
        "lambda_eta_anchor_cfg": cfg.lambda_eta_anchor,
        "lambda_energy_smooth_cfg": cfg.lambda_energy_smooth,
    }
    upsert_experiment_row(
        EXTENDED_ROOT / "registro_modelos.csv", registry_row,
        ["experiment_option", "experiment_version", "seed", "run_id"],
    )


def recover_extended_row(run_dir: Path):
    required = [
        run_dir / "COMPLETED", run_dir / "model_state.pt",
        run_dir / "training_history.csv", run_dir / "result.json",
        run_dir / "specification.json",
    ]
    if not all(path.exists() for path in required):
        return None
    return json.loads((run_dir / "result.json").read_text(encoding="utf-8"))

## 15. Intensidad del mecanismo NPP

La grilla evalúa una relación dosis-respuesta con dos ejes:

- `edge_gamma_npp`: proporción energética del grafo; el resto del peso corresponde a CLR.
- `lambda_scale`: multiplicador común de los términos NPP de la loss.

La configuración se ordena solo por CE media de validación. No se consulta test. El objetivo no es maximizar una cifra aislada, sino comprobar si aparece una región estable de mejora o una respuesta errática.

**Reanudacion directa:** con un runtime vacio se puede ejecutar solamente la celda de codigo de esta seccion. La celda restaura la infraestructura causal y lee los artefactos terminados de Drive; no corre los modelos base ni la ablacion de diez semillas.

In [ ]:
# Reanudacion directa de la opcion 3 desde un runtime vacio.
OPTION_3_DIRECT_RESUME = "train_extended_variant" not in globals()
if OPTION_3_DIRECT_RESUME:
    import json as direct_json
    from pathlib import Path as DirectPath
    try:
        from google.colab import drive as direct_drive
        direct_drive.mount("/content/drive", force_remount=True)
    except ImportError:
        pass
    direct_dir = DirectPath("/content/drive/MyDrive/Neural/NPP/Cripto")
    direct_candidates = [
        direct_dir / "GAT crypto - contraste experimental.ipynb",
        direct_dir / "GAT_crypto_contraste_experimental.ipynb",
    ]
    direct_notebook_path = next(
        (path for path in direct_candidates if path.exists()), None
    )
    if direct_notebook_path is None:
        direct_matches = sorted(direct_dir.glob("*GAT*contraste*experimental*.ipynb"))
        direct_notebook_path = direct_matches[0] if direct_matches else None
    if direct_notebook_path is None:
        raise FileNotFoundError("No se encontro el notebook de contraste en Drive.")
    direct_notebook = direct_json.loads(
        direct_notebook_path.read_text(encoding="utf-8")
    )
    direct_option_6_source = None
    for direct_index, direct_cell in enumerate(direct_notebook["cells"]):
        direct_source = "".join(direct_cell.get("source", []))
        if (
            direct_cell.get("cell_type") == "markdown"
            and direct_source.strip().startswith("## 17.")
        ):
            for direct_following in direct_notebook["cells"][direct_index + 1:]:
                if direct_following.get("cell_type") == "code":
                    direct_option_6_source = "".join(direct_following.get("source", []))
                    break
            break
    if direct_option_6_source is None:
        raise RuntimeError("No se encontro el bootstrap de reanudacion en la seccion 17.")
    direct_bootstrap_definition = direct_option_6_source.split(
        "\nbootstrap_extended_resume()", 1
    )[0]
    exec(
        compile(direct_bootstrap_definition, "direct_resume_bootstrap", "exec"),
        globals(),
    )
    bootstrap_extended_resume()
    RUN_OPTION_3_INTENSITY = True
    print("Opcion 3 reanudada directamente; se omiten las secciones anteriores.")

OPTION_3_GAMMAS = [0.0, 0.10, 0.25, 0.50]
OPTION_3_LAMBDA_SCALES = [0.0, 0.5, 1.0, 2.0]
OPTION_3_PATH = EXTENDED_ROOT / "opcion_3_intensidad_validacion.csv"

if RUN_OPTION_3_INTENSITY:
    done = completed_keys(
        OPTION_3_PATH,
        ["experiment_version", "seed", "edge_gamma_npp", "lambda_scale"],
    )
    for gamma in OPTION_3_GAMMAS:
        cfg_variant, loaders_variant = get_extended_loaders(gamma)
        for scale in OPTION_3_LAMBDA_SCALES:
            weights = full_npp_weights(scale=scale)
            for seed in EXTENDED_SEEDS:
                key = tuple(map(str, [EXPERIMENT_VERSION, seed, gamma, scale]))
                run_dir = EXTENDED_ROOT / "opcion_3_runs" / (
                    f"seed_{seed:04d}_gamma_{gamma:.2f}_lambda_{scale:.2f}"
                )
                if key in done:
                    continue
                recovered = recover_extended_row(run_dir)
                if recovered is not None:
                    upsert_experiment_row(
                        OPTION_3_PATH, recovered,
                        ["experiment_version", "seed", "edge_gamma_npp", "lambda_scale"],
                    )
                    done.add(key)
                    continue
                name = f"gamma={gamma:.2f};lambda_scale={scale:.2f}"
                model, history, val = train_extended_variant(
                    name, cfg_variant, loaders_variant, weights, seed, prediction_key="p"
                )
                row = {
                    "experiment_version": EXPERIMENT_VERSION,
                    "experiment_option": "3_intensidad",
                    "model_family": "CryptoGAT",
                    "model_variant": "GAT + NPP (grilla de intensidad)",
                    "prediction_head": "p",
                    "graph_type": "CLR puro" if gamma == 0.0 else "hibrido CLR + NPP",
                    "uses_npp_edges": bool(gamma > 0.0),
                    "uses_npp_loss": bool(scale > 0.0),
                    "uses_compositional_loss": False,
                    "seed": seed,
                    "edge_gamma_npp": gamma,
                    "edge_alpha_clr": 1.0 - gamma,
                    "lambda_scale": scale,
                    "val_CE": val["CE"],
                    "val_Aitchison": val["Aitchison"],
                    "val_CLR_MSE": val["CLR_MSE"],
                    "best_epoch": int(history.loc[history["val_objective"].idxmin(), "epoch"]),
                }
                persist_extended_run(
                    run_dir, model, history, row, weights, cfg_variant, "p"
                )
                upsert_experiment_row(
                    OPTION_3_PATH,
                    row,
                    ["experiment_version", "seed", "edge_gamma_npp", "lambda_scale"],
                )
                done.add(key)
                del model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    option_3_results = pd.read_csv(OPTION_3_PATH)
    option_3_summary = (
        option_3_results.groupby(["edge_gamma_npp", "lambda_scale"], as_index=False)
        .agg(
            val_CE_mean=("val_CE", "mean"),
            val_CE_std=("val_CE", "std"),
            val_Aitchison_mean=("val_Aitchison", "mean"),
            val_CLR_MSE_mean=("val_CLR_MSE", "mean"),
            n_seeds=("seed", "nunique"),
        )
        .sort_values("val_CE_mean")
    )
    option_3_summary.to_csv(EXTENDED_ROOT / "opcion_3_intensidad_resumen.csv", index=False)
    display(option_3_summary)
else:
    print("Opcion 3 preparada. Activar RUN_OPTION_3_INTENSITY para ejecutar la grilla.")

Opcion 3 preparada. Activar RUN_OPTION_3_INTENSITY para ejecutar la grilla.


In [ ]:
# """
# ANALISIS DE INTENSIDAD DEL MECANISMO NPP
#
# La grilla comprende 16 configuraciones y tres semillas emparejadas,
# combinando cuatro intensidades NPP en el grafo (gamma_npp) con cuatro
# escalas de los terminos NPP de la funcion objetivo (lambda_scale).
#
# La mejor configuracion predictiva dentro de la region evaluada es:
#
#         gamma_npp  = 0.50
#         lambda_scale = 1.00
#
# con:
#
#         CE media       = 2.961361
#         Aitchison      = 4.662777
#         CLR_MSE        = 21.800400
#
# Frente al control sin NPP:
#
#         gamma_npp  = 0.00
#         lambda_scale = 0.00
#         CE media       = 2.963339
#
# la configuracion seleccionada reduce CE aproximadamente 0.0668%.
# Aunque el efecto absoluto es pequeño, la mejora aparece en las tres
# semillas, por lo que no depende de una inicializacion aislada.
#
# Al promediar todas las escalas de loss, la CE mejora progresivamente
# cuando aumenta la presencia NPP en las aristas:
#
#         gamma = 0.00 -> CE = 2.962748
#         gamma = 0.10 -> CE = 2.962581
#         gamma = 0.25 -> CE = 2.962124
#         gamma = 0.50 -> CE = 2.961767
#
# Esto constituye evidencia preliminar de una relacion dosis-respuesta
# en la topologia: una mayor participacion de la proximidad energetica
# dentro del intervalo estudiado mejora la prediccion media.
#
# Sin embargo, gamma = 0.50 es el limite superior de la grilla. Por lo
# tanto, todavia no puede afirmarse que 0.50 sea el optimo; solo puede
# afirmarse que la funcion no alcanzo un maximo interior entre 0 y 0.50.
#
# La respuesta a lambda_scale es no lineal. Promediando los cuatro grafos:
#
#         lambda = 0.0 -> CE = 2.962938
#         lambda = 0.5 -> CE = 2.962132
#         lambda = 1.0 -> CE = 2.961938
#         lambda = 2.0 -> CE = 2.962212
#
# Aparece una region favorable aproximadamente entre 0.5 y 1.0, seguida
# por rendimientos decrecientes. No existe, sin embargo, un unico lambda
# optimo para todos los valores de gamma, lo que indica interaccion entre
# la topologia NPP y la intensidad de su regularizacion.
#
# Existe tambien un trade-off geometrico. Frente al control, la mejor
# configuracion en CE:
#
#         mejora CE          ~0.0668%
#         empeora Aitchison   ~3.26%
#         empeora CLR_MSE     ~6.57%
#
# Por lo tanto, los terminos NPP escalados por lambda_scale favorecen
# principalmente el ajuste de la masa economica medida mediante CE, pero
# no garantizan por si solos coherencia en log-ratios. Esto es consistente
# con la seccion 17, donde la incorporacion explicita de una loss CLR
# recupera gran parte de la coherencia composicional.
#
# La evidencia debe considerarse exploratoria: se utilizaron tres semillas
# de validacion y 34 de las 48 corridas seleccionaron la epoca maxima 20.
# Antes de declarar una intensidad definitiva se requiere confirmar las
# configuraciones seleccionadas con mayor horizonte de entrenamiento,
# mas semillas y ventanas temporales externas.
# """

## 16. Identificación interna del NPP

Este bloque mantiene fijo el grafo híbrido y retira o incorpora términos de la función objetivo. Incluye una cabeza energética autónoma, cuya predicción evaluada es `q_NPP=softmax(-gp/eta)` y cuya loss no utiliza la cabeza libre.

La comparación identifica qué parte aporta señal: residuo cerrado, CE energética, anclaje de escala, suavidad o acoplamiento completo. Todas las decisiones permanecen en validación.

**Reanudacion directa:**

In [ ]:
# Reanudacion directa de la opcion 5 desde un runtime vacio.
OPTION_5_DIRECT_RESUME = "train_extended_variant" not in globals()
if OPTION_5_DIRECT_RESUME:
    import json as direct_json
    from pathlib import Path as DirectPath
    try:
        from google.colab import drive as direct_drive
        direct_drive.mount("/content/drive")
    except ImportError:
        pass
    direct_dir = DirectPath("/content/drive/MyDrive/Neural/NPP/Cripto")
    direct_candidates = [
        direct_dir / "GAT crypto - contraste experimental.ipynb",
        direct_dir / "GAT_crypto_contraste_experimental.ipynb",
    ]
    direct_notebook_path = next(
        (path for path in direct_candidates if path.exists()), None
    )
    if direct_notebook_path is None:
        direct_matches = sorted(direct_dir.glob("*GAT*contraste*experimental*.ipynb"))
        direct_notebook_path = direct_matches[0] if direct_matches else None
    if direct_notebook_path is None:
        raise FileNotFoundError("No se encontro el notebook de contraste en Drive.")
    direct_notebook = direct_json.loads(
        direct_notebook_path.read_text(encoding="utf-8")
    )
    direct_option_6_source = None
    for direct_index, direct_cell in enumerate(direct_notebook["cells"]):
        direct_source = "".join(direct_cell.get("source", []))
        if (
            direct_cell.get("cell_type") == "markdown"
            and direct_source.strip().startswith("## 17.")
        ):
            for direct_following in direct_notebook["cells"][direct_index + 1:]:
                if direct_following.get("cell_type") == "code":
                    direct_option_6_source = "".join(direct_following.get("source", []))
                    break
            break
    if direct_option_6_source is None:
        raise RuntimeError("No se encontro el bootstrap de reanudacion en la seccion 17.")
    direct_bootstrap_definition = direct_option_6_source.split(
        "\nbootstrap_extended_resume()", 1
    )[0]
    exec(
        compile(direct_bootstrap_definition, "direct_resume_bootstrap", "exec"),
        globals(),
    )
    bootstrap_extended_resume()
    RUN_OPTION_5_IDENTIFICATION = True
    print("Opcion 5 reanudada directamente; se omiten las secciones anteriores.")

OPTION_5_PATH = EXTENDED_ROOT / "opcion_5_identificacion_validacion.csv"

OPTION_5_VARIANTS = {
    "predictiva_sola": {
        "prediction_key": "p",
        "weights": {"pred_ce": 1.0},
    },
    "residuo_npp": {
        "prediction_key": "p",
        "weights": {"pred_ce": 1.0, "npp_residual": CFG.lambda_npp},
    },
    "ce_energetica": {
        "prediction_key": "p",
        "weights": {"pred_ce": 1.0, "energy_ce": CFG.lambda_energy_ce},
    },
    "residuo_mas_ce_energetica": {
        "prediction_key": "p",
        "weights": {
            "pred_ce": 1.0,
            "npp_residual": CFG.lambda_npp,
            "energy_ce": CFG.lambda_energy_ce,
        },
    },
    "npp_completo": {
        "prediction_key": "p",
        "weights": full_npp_weights(scale=1.0),
    },
    "energia_autonoma": {
        "prediction_key": "q_npp",
        "weights": {
            "pred_ce": 0.0,
            "energy_ce": 1.0,
            "eta_anchor": CFG.lambda_eta_anchor,
            "smooth": CFG.lambda_energy_smooth,
        },
    },
}

if RUN_OPTION_5_IDENTIFICATION:
    cfg_hybrid, loaders_hybrid = get_extended_loaders(CFG.edge_gamma_npp)
    done = completed_keys(
        OPTION_5_PATH,
        ["experiment_version", "seed", "variant"],
    )
    for variant, specification in OPTION_5_VARIANTS.items():
        for seed in EXTENDED_SEEDS:
            key = tuple(map(str, [EXPERIMENT_VERSION, seed, variant]))
            run_dir = EXTENDED_ROOT / "opcion_5_runs" / f"seed_{seed:04d}_{variant}"
            if key in done:
                continue
            recovered = recover_extended_row(run_dir)
            if recovered is not None:
                upsert_experiment_row(
                    OPTION_5_PATH, recovered,
                    ["experiment_version", "seed", "variant"],
                )
                done.add(key)
                continue
            model, history, val = train_extended_variant(
                variant,
                cfg_hybrid,
                loaders_hybrid,
                specification["weights"],
                seed,
                prediction_key=specification["prediction_key"],
            )
            row = {
                "experiment_version": EXPERIMENT_VERSION,
                "experiment_option": "5_identificacion",
                "model_family": "CryptoGAT",
                "model_variant": variant,
                "prediction_head": specification["prediction_key"],
                "graph_type": "hibrido CLR + NPP",
                "uses_npp_edges": True,
                "uses_npp_loss": bool(variant != "predictiva_sola"),
                "uses_compositional_loss": False,
                "seed": seed,
                "variant": variant,
                "prediction_key": specification["prediction_key"],
                "val_CE": val["CE"],
                "val_Aitchison": val["Aitchison"],
                "val_CLR_MSE": val["CLR_MSE"],
                "best_epoch": int(history.loc[history["val_objective"].idxmin(), "epoch"]),
            }
            persist_extended_run(
                run_dir, model, history, row, specification["weights"],
                cfg_hybrid, specification["prediction_key"],
            )
            upsert_experiment_row(
                OPTION_5_PATH,
                row,
                ["experiment_version", "seed", "variant"],
            )
            done.add(key)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    option_5_results = pd.read_csv(OPTION_5_PATH)
    option_5_summary = (
        option_5_results.groupby(["variant", "prediction_key"], as_index=False)
        .agg(
            val_CE_mean=("val_CE", "mean"),
            val_CE_std=("val_CE", "std"),
            val_Aitchison_mean=("val_Aitchison", "mean"),
            val_CLR_MSE_mean=("val_CLR_MSE", "mean"),
            n_seeds=("seed", "nunique"),
        )
        .sort_values("val_CE_mean")
    )
    option_5_summary.to_csv(EXTENDED_ROOT / "opcion_5_identificacion_resumen.csv", index=False)
    display(option_5_summary)
else:
    print("Opcion 5 preparada. Activar RUN_OPTION_5_IDENTIFICATION para ejecutar las ablaciones.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dispositivo: cuda
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Filas: 26,064; nodos: 471
Error maximo de cierre del simplex: 4.488e-08
Rango temporal: 2026-01-01 00:00:00+00:00 -> 2026-06-30 23:50:00+00:00
Se reutiliza cache: /content/gat_crypto_cache/gat_crypto_features_float32.dat
Train: 2026-03-26 05:20:00+00:00 -> 2026-06-02 23:20:00+00:00
Validacion: 2026-06-02 23:40:00+00:00 -> 2026-06-16 23:30:00+00:00
Test: 2026-06-16 23:50:00+00:00 -> 2026-06-30 23:40:00+00:00
Raiz de extensiones: /content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1
Semillas de desarrollo: [11, 23, 42]
Reanudacion lista: no se ejecutaron entrenamientos ante

,variant,prediction_key,val_CE_mean,val_CE_std,val_Aitchison_mean,val_CLR_MSE_mean,n_seeds
0,ce_energetica,p,2.961651,0.000794,4.571888,20.963333,3
5,residuo_npp,p,2.961745,0.000675,4.641931,21.607565,3
4,residuo_mas_ce_energetica,p,2.961799,0.000387,4.628488,21.483126,3
2,npp_completo,p,2.961806,0.000335,4.635802,21.550880,3
3,predictiva_sola,p,2.962534,0.001193,4.535048,20.629574,3
1,energia_autonoma,q_npp,2.969290,0.003284,3.896464,15.280940,3


In [ ]:
# """
# ABLACION DE IDENTIFICACION DE LOS TERMINOS NPP

# Manteniendo fija la topologia hibrida CLR+NPP, la variante que agrega
# unicamente energy_ce obtiene la mejor Cross-Entropy de validacion
# (2.961651), mejorando ligeramente al modelo puramente predictivo
# (2.962534). Esto sugiere que la representacion energetica NPP aporta
# informacion predictiva adicional.

# El residuo NPP tambien mejora ligeramente CE, pero su combinacion con
# energy_ce y la loss NPP completa no supera a energy_ce aislada,
# indicando ausencia de una sinergia aditiva simple entre los terminos.

# La cabeza energetica autonoma q_npp presenta el resultado mas relevante:
# sin utilizar la CE predictiva convencional, obtiene CE=2.969290, solo
# ~0.23% peor que el predictor libre, mientras reduce Aitchison ~14% y
# CLR-MSE ~26%.

# Esto sugiere que q_npp captura una estructura composicional subyacente
# especialmente coherente, mientras la cabeza predictiva p captura mejor
# las desviaciones dinamicas de corto plazo.

# Preliminarmente, energy_ce aparece como el componente NPP con mayor
# utilidad predictiva marginal, mientras q_npp constituye la evidencia
# mas fuerte de capacidad estructural autonoma del mecanismo energetico.
# """

## 17. Coherencia composicional CLR/Aitchison

La pérdida geométrica agrega el error cuadrático en coordenadas CLR a la CE predictiva y a la regularización NPP. Se evalúa una grilla pequeña de `lambda_geometry` y se construye una frontera de Pareto sobre validación:

- menor CE: mejor asignación de masa;
- menor Aitchison: mejor reproducción de log-ratios.

No se selecciona con test. CLR se utiliza para entrenar porque es computacionalmente equivalente a trabajar en el subespacio composicional sin construir una base ILR densa de 471 por 470.

**Reanudacion directa:** despues de reiniciar el runtime se puede ejecutar esta unica celda de codigo. Ella monta Drive, restaura solo la infraestructura causal necesaria y reutiliza un cache persistente de features. No ejecuta el entrenamiento base, la ablacion de diez semillas ni las opciones 3 y 5.

In [ ]:
# Punto de reanudacion autonomo: permite empezar directamente en la seccion 17.
def bootstrap_extended_resume() -> None:
    required = {
        "CFG", "P", "FEATURES", "train_t", "validation_t",
        "CryptoGAT", "train_extended_variant", "distribution_metrics",
        "ABLATION_RESULTS_ROOT", "EXTENDED_ROOT",
    }
    if required.issubset(globals()):
        return

    import json as resume_json
    import os as resume_os
    import shutil as resume_shutil
    from pathlib import Path as ResumePath
    from dataclasses import replace as resume_replace
    try:
        from google.colab import drive as resume_drive
        resume_drive.mount("/content/drive")
    except ImportError:
        pass

    notebook_dir = ResumePath("/content/drive/MyDrive/Neural/NPP/Cripto")
    exact_candidates = [
        notebook_dir / "GAT crypto - contraste experimental.ipynb",
        notebook_dir / "GAT_crypto_contraste_experimental.ipynb",
    ]
    notebook_path = next((path for path in exact_candidates if path.exists()), None)
    if notebook_path is None:
        matches = sorted(notebook_dir.glob("*GAT*contraste*experimental*.ipynb"))
        notebook_path = matches[0] if matches else None
    if notebook_path is None:
        raise FileNotFoundError(
            "No se encontro el notebook en Drive para reconstruir el entorno."
        )

    notebook = resume_json.loads(notebook_path.read_text(encoding="utf-8"))

    def code_after_heading(prefix: str) -> str:
        cells = notebook["cells"]
        for index, cell in enumerate(cells):
            source = "".join(cell.get("source", []))
            if cell.get("cell_type") == "markdown" and source.strip().startswith(prefix):
                for following in cells[index + 1:]:
                    if following.get("cell_type") == "code":
                        return "".join(following.get("source", []))
        raise KeyError(f"No se encontro una celda de codigo despues de {prefix}.")

    def execute_source(source: str, label: str) -> None:
        exec(compile(source, f"{notebook_path.name}:{label}", "exec"), globals())

    execute_source(code_after_heading("# GAT crypto"), "imports")
    execute_source(code_after_heading("## 1."), "config")
    resume_cfg = resume_replace(
        globals()["CFG"], cache_dir="/content/gat_crypto_cache", rebuild_features=False
    )
    globals()["CFG"] = resume_cfg
    execute_source(code_after_heading("## 2."), "data")

    persistent_cache = notebook_dir / "GAT_crypto_resume_state" / "feature_cache"
    local_cache = ResumePath(resume_cfg.cache_dir)
    local_cache.mkdir(parents=True, exist_ok=True)
    persistent_cache.mkdir(parents=True, exist_ok=True)
    cache_names = ["gat_crypto_features_float32.dat", "gat_crypto_features_meta.json"]
    if all((persistent_cache / name).exists() for name in cache_names):
        for name in cache_names:
            destination = local_cache / name
            if not destination.exists():
                resume_shutil.copy2(persistent_cache / name, destination)

    execute_source(code_after_heading("## 3."), "features")
    for name in cache_names:
        source_path = local_cache / name
        destination = persistent_cache / name
        if source_path.exists() and not destination.exists():
            temporary = persistent_cache / f"{name}.tmp"
            resume_shutil.copy2(source_path, temporary)
            resume_os.replace(temporary, destination)

    for heading, label in [
        ("## 4.", "split"), ("## 5.", "graph"),
        ("## 6.", "dataset"), ("## 7.", "model"),
        ("## 8.", "loss_and_train"),
    ]:
        execute_source(code_after_heading(heading), label)

    metrics_source = code_after_heading("## 10.")
    metrics_definitions = metrics_source.split(
        "\npred_free = collect_predictions", 1
    )[0]
    execute_source(metrics_definitions, "metric_definitions")

    ablation_source = code_after_heading("## 12.")
    ablation_constants = ablation_source.split("\n# Grafo totalmente libre", 1)[0]
    execute_source(ablation_constants, "ablation_paths")
    execute_source(code_after_heading("## 14."), "extended_config")
    execute_source(code_after_heading("### Infraestructura"), "extended_helpers")
    globals()["EXTENDED_RESUME_READY"] = True
    print("Reanudacion lista: no se ejecutaron entrenamientos anteriores.")


bootstrap_extended_resume()
import matplotlib.pyplot as plt

OPTION_6_GEOMETRY_LAMBDAS = [0.0, 1e-5, 1e-4, 1e-3, 1e-2]
OPTION_6_PATH = EXTENDED_ROOT / "opcion_6_geometria_validacion.csv"


def pareto_flags(frame: pd.DataFrame, x: str, y: str) -> np.ndarray:
    values = frame[[x, y]].to_numpy(dtype=np.float64)
    flags = np.ones(len(values), dtype=bool)
    for i, candidate in enumerate(values):
        dominates = np.all(values <= candidate, axis=1) & np.any(values < candidate, axis=1)
        if np.any(dominates):
            flags[i] = False
    return flags


if RUN_OPTION_6_COMPOSITIONAL:
    cfg_hybrid, loaders_hybrid = get_extended_loaders(CFG.edge_gamma_npp)
    done = completed_keys(
        OPTION_6_PATH,
        ["experiment_version", "seed", "lambda_geometry"],
    )
    for lambda_geometry in OPTION_6_GEOMETRY_LAMBDAS:
        weights = full_npp_weights(scale=1.0, geometry=lambda_geometry)
        for seed in EXTENDED_SEEDS:
            key = tuple(map(str, [EXPERIMENT_VERSION, seed, lambda_geometry]))
            run_dir = EXTENDED_ROOT / "opcion_6_runs" / (
                f"seed_{seed:04d}_geometry_{lambda_geometry:.0e}"
            )
            if key in done:
                continue
            recovered = recover_extended_row(run_dir)
            if recovered is not None:
                upsert_experiment_row(
                    OPTION_6_PATH, recovered,
                    ["experiment_version", "seed", "lambda_geometry"],
                )
                done.add(key)
                continue
            name = f"npp_geometry={lambda_geometry:.0e}"
            model, history, val = train_extended_variant(
                name,
                cfg_hybrid,
                loaders_hybrid,
                weights,
                seed,
                prediction_key="p",
            )
            row = {
                "experiment_version": EXPERIMENT_VERSION,
                "experiment_option": "6_composicional",
                "model_family": "CryptoGAT",
                "model_variant": "GAT + NPP + loss CLR",
                "prediction_head": "p",
                "graph_type": "hibrido CLR + NPP",
                "uses_npp_edges": True,
                "uses_npp_loss": True,
                "uses_compositional_loss": bool(lambda_geometry > 0.0),
                "seed": seed,
                "lambda_geometry": lambda_geometry,
                "val_CE": val["CE"],
                "val_Aitchison": val["Aitchison"],
                "val_CLR_MSE": val["CLR_MSE"],
                "best_epoch": int(history.loc[history["val_objective"].idxmin(), "epoch"]),
            }
            persist_extended_run(
                run_dir, model, history, row, weights, cfg_hybrid, "p"
            )
            upsert_experiment_row(
                OPTION_6_PATH,
                row,
                ["experiment_version", "seed", "lambda_geometry"],
            )
            done.add(key)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    option_6_results = pd.read_csv(OPTION_6_PATH)
    option_6_summary = (
        option_6_results.groupby("lambda_geometry", as_index=False)
        .agg(
            val_CE_mean=("val_CE", "mean"),
            val_CE_std=("val_CE", "std"),
            val_Aitchison_mean=("val_Aitchison", "mean"),
            val_Aitchison_std=("val_Aitchison", "std"),
            val_CLR_MSE_mean=("val_CLR_MSE", "mean"),
            n_seeds=("seed", "nunique"),
        )
    )
    option_6_summary["pareto"] = pareto_flags(
        option_6_summary, "val_CE_mean", "val_Aitchison_mean"
    )
    option_6_summary.to_csv(EXTENDED_ROOT / "opcion_6_geometria_resumen.csv", index=False)
    display(option_6_summary.sort_values(["pareto", "val_CE_mean"], ascending=[False, True]))

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(option_6_summary["val_CE_mean"], option_6_summary["val_Aitchison_mean"])
    for _, row in option_6_summary.iterrows():
        ax.annotate(f'{row["lambda_geometry"]:.0e}', (row["val_CE_mean"], row["val_Aitchison_mean"]))
    ax.set_xlabel("CE media de validacion")
    ax.set_ylabel("Aitchison medio de validacion")
    ax.set_title("Frontera masa vs geometria composicional")
    fig.tight_layout()
    fig.savefig(EXTENDED_ROOT / "opcion_6_frontera_pareto.png", dpi=180)
    plt.show()
else:
    print("Opcion 6 preparada. Activar RUN_OPTION_6_COMPOSITIONAL para ejecutar la grilla geometrica.")

In [ ]:
# """
# NOTA METODOLOGICA: PAPEL DE gamma_npp Y lambda_geometry

# 1. gamma_npp y lambda_geometry actuan sobre espacios geometricos distintos
# --------------------------------------------------------------------------

# gamma_npp interviene durante la CONSTRUCCION DE LA TOPOLOGIA DEL GRAFO.

# El score utilizado para seleccionar las aristas combina:

#     score_ij = alpha_clr * S_CLR(i,j) + gamma_npp * S_NPP(i,j)

# donde S_CLR mide similitud composicional historica entre nodos y S_NPP mide
# proximidad entre sus energias NPP de primer orden.

# Por lo tanto, gamma_npp modifica la geometria/topologia LOCAL del sistema:
# determina que nodos son considerados proximos y, en consecuencia, cuales
# intercambian informacion mediante el mecanismo de atencion del GAT.

# En terminos conceptuales:

#     gamma_npp -> geometria de las interacciones entre nodos.

# lambda_geometry, en cambio, no modifica las aristas. Actua durante el
# ENTRENAMIENTO sobre la geometria GLOBAL del simplex predicho mediante una
# funcion objetivo del tipo:

#     L_total = L_pred + lambda_geometry * L_geometry

# donde L_pred representa la perdida predictiva (por ejemplo, Cross-Entropy)
# y L_geometry penaliza errores en el espacio composicional CLR/Aitchison.

# Por lo tanto:

#     gamma_npp      -> estructura quien interactua con quien.
#     lambda_geometry -> estructura como debe organizarse globalmente
#                        la distribucion probabilistica predicha.

# Ambos mecanismos son complementarios: gamma_npp incorpora estructura NPP
# en la topologia del grafo, mientras lambda_geometry fuerza coherencia
# composicional en la solucion final producida por el GAT.


# 2. Seleccion preliminar de lambda_geometry
# ------------------------------------------

# El sweep de validacion muestra una frontera clara entre precision predictiva
# y coherencia composicional.

# Resultados principales:

#     lambda = 0:
#         CE          = 2.961797
#         Aitchison   = 4.633726
#         CLR_MSE     = 21.531628



#     lambda = 1e-3:
#         CE          = 2.964038
#         Aitchison   = 2.133120
#         CLR_MSE     = 4.981605



# El sweet spot se encuentra aproximadamente entre lambda = 1e-4 y 1e-3.
# Para los experimentos principales se selecciona:

######################          lambda_geometry = 1e-3         #####################

# La razon es que, respecto a lambda = 0, este valor:

#     - aumenta la Cross-Entropy solo ~0.076%;
#     - reduce Aitchison aproximadamente ~54%;
#     - reduce CLR_MSE aproximadamente ~77%.

# Por lo tanto, lambda = 1e-3 constituye preliminarmente el mejor compromiso:
# la geometria composicional mejora drasticamente mientras la capacidad
# predictiva medida mediante Cross-Entropy permanece practicamente intacta.

# lambda = 1e-2 obtiene una geometria ligeramente mejor, pero presenta
# rendimientos decrecientes: la mejora adicional en Aitchison/CLR es pequena
# respecto a 1e-3 y requiere un sacrificio predictivo mayor.


# 3. Hipotesis sobre compatibilidad entre los objetivos
# -----------------------------------------------------

# Los resultados sugieren preliminarmente que la perdida predictiva y la
# perdida geometrica NO son fuertemente antagonicas.

# Si sus gradientes fueran altamente opuestos, una reduccion de ~54-77% del
# error geometrico deberia generar un deterioro considerable de la
# Cross-Entropy. Sin embargo, con lambda = 1e-3 la CE empeora solo ~0.076%.

# Esto sugiere como hipotesis que los gradientes de ambos objetivos presentan
# una alineacion positiva o, al menos, no negativa:

#                     <g_pred, g_geom>
#     cos(theta) = --------------------------- >= 0
#                  ||g_pred|| * ||g_geom||

# donde:

#     g_pred = grad_theta(L_pred)
#     g_geom = grad_theta(L_geometry)

# Interpretacion:

#     cos(theta) > 0:
#         ambos objetivos actualizan los parametros en direcciones compatibles.

#     cos(theta) ~= 0:
#         los objetivos son aproximadamente ortogonales; mejorar uno afecta
#         poco al otro.

#     cos(theta) < 0:
#         existe conflicto directo entre precision predictiva y geometria
#         composicional.

# Esta conclusion NO puede establecerse solo a partir del sweep de lambda.
# Debe verificarse experimentalmente.


# EXPERIMENTO PROPUESTO: COSENO ENTRE GRADIENTES
# ----------------------------------------------

# Durante el entrenamiento, antes de combinar las dos losses:

# 1. Calcular L_pred y L_geometry por separado.

# 2. Obtener sus gradientes respecto de los mismos parametros del modelo:

#        g_pred = grad(L_pred, theta)
#        g_geom = grad(L_geometry, theta)

# 3. Aplanar y concatenar los gradientes de todos los parametros compartidos.

# 4. Calcular en cada batch:

#                          g_pred · g_geom
#        cosine_grad = ---------------------------
#                      ||g_pred|| * ||g_geom||

# 5. Registrar cosine_grad durante todo el entrenamiento y analizar:

#        - media;
#        - mediana;
#        - desviacion estandar;
#        - porcentaje de batches con coseno > 0;
#        - porcentaje con coseno < 0;
#        - evolucion temporal por epoch.

# 6. Repetir el experimento para:

#        lambda = 1e-4
#        lambda = 1e-3
#        lambda = 1e-2

#    manteniendo exactamente iguales dataset, arquitectura, seeds y splits.

# La hipotesis preliminar es:

#        E[cosine_grad] >= 0

# especialmente alrededor de lambda = 1e-3.

# Si se confirma, significaria que la mejora de la geometria composicional no
# es simplemente una restriccion externa que compite contra la prediccion,
# sino que ambas funciones objetivo tienden a favorecer regiones compatibles
# del espacio parametrico.

# El resultado mas fuerte seria observar simultaneamente:

#     Aitchison ↓
#     CLR_MSE   ↓
#     CE        ~ constante
#     cos(g_pred, g_geom) >= 0

# lo que proporcionaria evidencia de que coherencia composicional y capacidad
# predictiva son, en gran medida, objetivos geometricamente compatibles.
# """

## 18. Auditoría de nodos dominantes y cola

Esta sección es post-hoc y descriptiva. No entrena ni selecciona modelos. Recalcula métricas desde las predicciones guardadas bajo escenarios predefinidos:

- universo completo;
- sin `U`;
- sin BTC;
- sin `U` y BTC;
- sin los cinco y diez nodos de mayor participación;
- únicamente nodos activos.

También exporta concentración del error cuadrático, sensibilidad leave-one-out de los diez nodos principales, métricas por estrato de masa y una tabla que rastrea cada ticker hasta sus columnas fuente. Si el runtime se reinicio, se ejecuta primero la celda de codigo de la seccion 17 y luego esta celda; no hace falta correr ninguna seccion anterior.

In [ ]:
OPTION_7_ROOT = EXTENDED_ROOT / "opcion_7_sensibilidad"
OPTION_7_ROOT.mkdir(parents=True, exist_ok=True)


def subset_and_close(values: np.ndarray, mask: np.ndarray, eps: float = 0.0) -> np.ndarray:
    subset = np.clip(values[:, mask].astype(np.float64), eps, None)
    denominator = subset.sum(axis=1, keepdims=True)
    if np.any(denominator <= 0):
        raise ValueError("Un escenario de sensibilidad dejo filas sin masa.")
    return subset / denominator


def load_saved_predictions(seed: int) -> dict:
    path = ABLATION_RESULTS_ROOT / f"seed_{seed:04d}" / "predicciones_test.npz"
    if not path.exists():
        raise FileNotFoundError(f"Falta la prediccion guardada para la semilla {seed}: {path}")
    with np.load(path) as archive:
        return {key: archive[key] for key in archive.files}


if RUN_OPTION_7_SENSITIVITY:
    prediction_keys = {
        "Persistencia": "persistencia",
        "GAT 100% libre": "gat_libre",
        "GAT NPP solo aristas": "gat_npp_aristas",
        "GAT + NPP": "gat_npp",
    }
    model_metadata = {
        "Persistencia": {
            "model_family": "Benchmark", "model_variant": "Persistencia",
            "prediction_head": "ultima observacion", "graph_type": "sin grafo",
            "uses_npp_edges": False, "uses_npp_loss": False,
        },
        "GAT 100% libre": {
            "model_family": "CryptoGAT", "model_variant": "GAT 100% libre",
            "prediction_head": "p", "graph_type": "CLR puro",
            "uses_npp_edges": False, "uses_npp_loss": False,
        },
        "GAT NPP solo aristas": {
            "model_family": "CryptoGAT", "model_variant": "GAT NPP solo aristas",
            "prediction_head": "p", "graph_type": "hibrido CLR + NPP",
            "uses_npp_edges": True, "uses_npp_loss": False,
        },
        "GAT + NPP": {
            "model_family": "CryptoGAT", "model_variant": "GAT + NPP",
            "prediction_head": "p", "graph_type": "hibrido CLR + NPP",
            "uses_npp_edges": True, "uses_npp_loss": True,
        },
    }
    first = load_saved_predictions(SEEDS[0])
    reference_target = first["target"].astype(np.float64)
    mean_share = reference_target.mean(axis=0)
    descending = np.argsort(-mean_share)
    active_mask = mean_share >= ACTIVE_SHARE_THRESHOLD

    def without_nodes(nodes):
        mask = np.ones(len(assets), dtype=bool)
        for node in nodes:
            if node in assets:
                mask[assets.index(node)] = False
        return mask

    scenarios = {
        "completo": np.ones(len(assets), dtype=bool),
        "sin_U": without_nodes(["U"]),
        "sin_BTC": without_nodes(["BTC"]),
        "sin_U_y_BTC": without_nodes(["U", "BTC"]),
        "sin_top_5": without_nodes([assets[i] for i in descending[:5]]),
        "sin_top_10": without_nodes([assets[i] for i in descending[:10]]),
        "solo_activos": active_mask.copy(),
    }

    sensitivity_rows = []
    concentration_rows = []
    leave_one_out_rows = []

    for seed in SEEDS:
        saved = load_saved_predictions(seed)
        target = saved["target"].astype(np.float64)
        if not np.allclose(target, reference_target, atol=1e-12, rtol=0.0):
            raise AssertionError("El target de test cambio entre semillas.")

        for model_name, archive_key in prediction_keys.items():
            prediction = saved[archive_key].astype(np.float64)
            squared = np.square(prediction - target).mean(axis=0)
            total_squared = squared.sum()
            for node_id, node in enumerate(assets):
                concentration_rows.append(
                    {
                        **model_metadata[model_name],
                        "seed": seed,
                        "Modelo": model_name,
                        "Nodo": node,
                        "Participacion_media_real": mean_share[node_id],
                        "MSE_nodo": squared[node_id],
                        "Contribucion_error_cuadratico": squared[node_id] / total_squared,
                    }
                )

            for scenario_name, mask in scenarios.items():
                target_subset = subset_and_close(target, mask)
                prediction_subset = subset_and_close(prediction, mask, eps=CFG.eps)
                metrics_subset = distribution_metrics(
                    model_name, target_subset, prediction_subset
                )
                metrics_subset.update(
                    {
                        **model_metadata[model_name],
                        "seed": seed,
                        "Escenario": scenario_name,
                        "N_nodos": int(mask.sum()),
                    }
                )
                sensitivity_rows.append(metrics_subset)

            for node_id in descending[:10]:
                mask = np.ones(len(assets), dtype=bool)
                mask[node_id] = False
                metrics_subset = distribution_metrics(
                    model_name,
                    subset_and_close(target, mask),
                    subset_and_close(prediction, mask, eps=CFG.eps),
                )
                metrics_subset.update(
                    {
                        **model_metadata[model_name],
                        "seed": seed,
                        "Nodo_excluido": assets[node_id],
                        "Participacion_nodo_excluido": mean_share[node_id],
                    }
                )
                leave_one_out_rows.append(metrics_subset)

    sensitivity = pd.DataFrame(sensitivity_rows)
    concentration = pd.DataFrame(concentration_rows)
    leave_one_out = pd.DataFrame(leave_one_out_rows)

    sensitivity.to_csv(OPTION_7_ROOT / "metricas_por_escenario.csv", index=False)
    concentration.to_csv(OPTION_7_ROOT / "concentracion_error_por_nodo.csv", index=False)
    leave_one_out.to_csv(OPTION_7_ROOT / "sensibilidad_leave_one_out_top10.csv", index=False)

    sensitivity_summary = (
        sensitivity.groupby(["Escenario", "Modelo"], as_index=False)
        .agg(
            CE_mean=("CE", "mean"),
            KL_mean=("KL", "mean"),
            JS_mean=("JS", "mean"),
            MAE_mean=("MAE_nodo", "mean"),
            RMSE_mean=("RMSE_nodo", "mean"),
            Aitchison_mean=("Aitchison", "mean"),
            n_seeds=("seed", "nunique"),
        )
    )
    sensitivity_summary.to_csv(OPTION_7_ROOT / "resumen_por_escenario.csv", index=False)

    concentration_summary = (
        concentration.groupby(["Modelo", "Nodo"], as_index=False)
        .agg(
            Participacion_media_real=("Participacion_media_real", "mean"),
            MSE_nodo=("MSE_nodo", "mean"),
            Contribucion_error_cuadratico=("Contribucion_error_cuadratico", "mean"),
        )
        .sort_values(["Modelo", "Contribucion_error_cuadratico"], ascending=[True, False])
    )
    concentration_summary.to_csv(
        OPTION_7_ROOT / "concentracion_error_resumen.csv", index=False
    )

    node_metrics_path = ABLATION_RESULTS_ROOT / "metricas_por_nodo_y_semilla.csv"
    node_metrics = pd.read_csv(node_metrics_path)
    share_bins = [-1e-15, 1e-8, 1e-5, 1e-3, 1e-2, np.inf]
    share_labels = ["inactivo_o_casi_cero", "micro", "pequeno", "mediano", "dominante"]
    node_metrics["Estrato_masa"] = pd.cut(
        node_metrics["Participacion_media_real"],
        bins=share_bins,
        labels=share_labels,
        include_lowest=True,
        right=False,
    )
    tail_summary = (
        node_metrics.groupby(["Modelo", "Estrato_masa"], observed=True, as_index=False)
        .agg(
            N_nodos=("Nodo", "nunique"),
            Participacion_media=("Participacion_media_real", "mean"),
            MAE_mean=("MAE", "mean"),
            RMSE_mean=("RMSE", "mean"),
            RMSE_CLR_mean=("RMSE_CLR", "mean"),
            Tasa_cero_mean=("Tasa_masa_casi_cero", "mean"),
        )
    )
    tail_summary.to_csv(OPTION_7_ROOT / "metricas_por_estrato_de_masa.csv", index=False)

    audit_nodes = list(dict.fromkeys(["U", "BTC", "ETH"] + [assets[i] for i in descending[:10]]))
    source_audit = []
    for node in audit_nodes:
        source_audit.append(
            {
                "Nodo": node,
                "p_col": f"{node}_p",
                "p_col_presente": f"{node}_p" in p_cols,
                "y_col": f"{node}_y",
                "y_col_presente": f"{node}_y" in y_cols,
                "precio_col": f"{node}USDT",
                "precio_col_presente": f"{node}USDT" in price_cols,
                "volumen_col_V": f"{node}_V",
                "volumen_col_V_presente": f"{node}_V" in volume_cols,
                "volumen_col_v": f"{node}_v",
                "volumen_col_v_presente": f"{node}_v" in volume_cols,
                "Participacion_media_real": mean_share[assets.index(node)],
            }
        )
    source_audit = pd.DataFrame(source_audit)
    source_audit.to_csv(OPTION_7_ROOT / "auditoria_columnas_fuente.csv", index=False)

    display(sensitivity_summary)
    display(concentration_summary.groupby("Modelo", sort=False).head(10))
    display(tail_summary)
    display(source_audit)
else:
    print("Opcion 7 preparada. Activar RUN_OPTION_7_SENSITIVITY para ejecutar la auditoria.")

In [ ]:
# """
# AUDITORIA DE SENSIBILIDAD A LA MASA DEL SIMPLEX

# La ventaja predictiva del GAT+NPP no depende exclusivamente de los nodos
# dominantes. Su mejora en CE/KL frente al GAT libre persiste al excluir BTC,
# U, ambos y los top-5/top-10 nodos, llegando incluso a aumentar ligeramente
# en términos relativos.

# El error cuadratico esta fuertemente concentrado: U y BTC explican ~74% del
# MSE del GAT+NPP, y los principales 10 nodos concentran ~92.5%. Esto explica
# la gran diferencia observada entre MAE y RMSE: la mayoria de los nodos posee
# errores absolutos pequeños, mientras unos pocos concentran errores grandes.

# La auditoria tambien muestra que el Aitchison global esta fuertemente
# afectado por nodos inactivos o de masa casi nula. Estos presentan MAE
# practicamente cero pero errores CLR extremadamente altos debido al caracter
# log-ratio de la metrica y a que el softmax produce probabilidades positivas.

# Al restringir el simplex a nodos activos, la distancia Aitchison del GAT+NPP
# cae sustancialmente, indicando que parte importante de su aparente
# incoherencia geometrica global proviene del tratamiento de ceros estructurales
# y no necesariamente de los componentes economicamente activos.
# """

## 19. Reglas de interpretación de las extensiones

1. Las opciones 3, 5 y 6 son experimentos de desarrollo. Sus rankings pertenecen a validación y no autorizan a reusar el test para seguir ajustando.
2. Las tres semillas de desarrollo sirven como control operativo. La configuración elegida debe confirmarse después con más semillas y una ventana temporal nueva.
3. La opción 7 describe sensibilidad del test ya observado. No puede utilizarse para escoger retrospectivamente features, nodos o hiperparámetros y volver a reportar el mismo test como evidencia nueva.
4. Una mejora de CE/KL con deterioro de Aitchison se interpreta como intercambio entre asignación de masa y geometría de log-ratios, no como superioridad universal.
5. La identificación de `U` y otros nodos dominantes debe resolverse contra las columnas fuente antes de atribuir significado económico.

## 20. Evaluación temporal secuencial con modelos congelados — sin reentrenamiento

Ejecutar **solo la siguiente celda**, incluso en un runtime vacío. CPU es suficiente.

Se recorren las predicciones de test ya guardadas de las diez semillas y los cuatro modelos. Por defecto se evalúan bloques consecutivos no solapados de un día, anclados al primer timestamp de test, y el acumulado hasta el final de cada bloque. `VENTANA_TEMPORAL` permite cambiar el tamaño antes de ejecutar; no se selecciona automáticamente el mejor tamaño con test.

**Alcance:** es un backtest temporal retrospectivo sobre predicciones congeladas; no es un walk-forward con reajuste ni genera nuevas ventanas externas al test disponible. Los pesos permanecen fijos; las predicciones originales ya incorporaban información causal disponible en cada instante y el grafo dinámico original. Las ventanas acumuladas se solapan y no son réplicas independientes. La dispersión entre semillas no es un intervalo de incertidumbre temporal.

**Entradas exactas:** `GAT_crypto_results/config.json`, `GAT_crypto_results/predicciones_test.npz` (metadatos de activos y referencia del target), y `GAT_crypto_ablation_10_seeds/seed_XXXX/predicciones_test.npz`. Se comprueban índices, targets y persistencia entre semillas. El formato antiguo no guardó activos/config por semilla; la referencia base se acepta solo si coincide exactamente en target e índices. Fechas de formato object se reconstruyen de config e índices sin deserializar pickle.

**Salidas:** CSV por semilla, resumen por ventana/modelo, deltas emparejados y manifiesto con hashes, configuración y fechas. Cada fila de resultados identifica modelo y semilla. Se guardan en `GAT_crypto_extended_audits_v1/opcion_20_temporal_sin_entreno/`, bajo una subcarpeta de especificación y ejecución. Persistencia se repite únicamente para emparejar. Si falta un archivo, la celda informa la ruta y se detiene: nunca inicia un entrenamiento.

No necesita ejecutar las secciones 1–19. La tabla aparece al completar las lecturas; durante la ejecución se informa la semilla en curso.


In [ ]:
# Ejecutar esta celda sola. No requiere GPU ni celdas previas.
CARPETA_PROYECTO = '/content/drive/MyDrive/Neural/NPP/Cripto'
VENTANA_TEMPORAL = '1D'

import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)


POSTHOC_VERSION = 'frozen_predictions_v1_target_precision_fix'
POSTHOC_SEEDS = [11, 23, 42, 67, 101, 137, 211, 307, 401, 503]
POSTHOC_MODELS = {
    'Persistencia': 'persistencia',
    'GAT 100% libre': 'gat_libre',
    'GAT NPP solo aristas': 'gat_npp_aristas',
    'GAT + NPP': 'gat_npp',
}


def ph_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def ph_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def ph_json(value, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)


def ph_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(tmp, path)


def ph_mount(root):
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        try:
            drive.mount('/content/drive')
            list(Path(root).iterdir())
        except OSError:
            drive.mount('/content/drive', force_remount=True)
    if not Path(root).is_dir():
        raise FileNotFoundError(f'No existe la carpeta del proyecto: {root}')


def ph_prob(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError('Predicciones/target deben ser matrices finitas y no negativas.')
    mass = values.sum(axis=1, keepdims=True)
    if np.any(mass <= 0) or not np.allclose(mass, 1., atol=1e-5, rtol=0):
        raise ValueError('El artefacto no contiene un simplex cerrado.')
    return values / mass


def ph_context(root):
    root = Path(root)
    ph_mount(root)
    base = root / 'GAT_crypto_results'
    config_path = base / 'config.json'
    reference_path = base / 'predicciones_test.npz'
    cfg = json.loads(config_path.read_text(encoding='utf-8'))
    with np.load(reference_path, allow_pickle=False) as saved:
        assets = saved['assets'].astype(str)
        t = saved['t'].copy()
        target_raw = saved['target'].copy()
        # Older exports stored timestamps as object arrays. Do not unpickle them.
        try:
            saved_dates = pd.to_datetime(saved['timestamps'].astype(str), utc=True)
        except ValueError as error:
            if 'Object arrays cannot be loaded' not in str(error):
                raise
            saved_dates = None
    if t.ndim != 1 or not np.issubdtype(t.dtype, np.integer) or len(t) < 2:
        raise ValueError('Indices temporales de referencia invalidos.')
    if np.any(np.diff(t) <= 0) or np.any(t < 0) or len(set(assets)) != len(assets):
        raise ValueError('Fechas duplicadas/desordenadas o activos duplicados.')
    target = ph_prob(target_raw)
    if target.shape != (len(t), len(assets)):
        raise ValueError('El orden/dimension de activos no coincide con el target.')
    step = pd.Timedelta(cfg['frequency'])
    expected_dates = pd.Timestamp(cfg['start_timestamp']) + (t + int(cfg['horizon'])) * step
    dates = pd.DatetimeIndex(expected_dates)
    if saved_dates is not None and not np.array_equal(saved_dates.asi8, dates.asi8):
        raise ValueError('Config y timestamps guardados pertenecen a experimentos distintos.')
    ablation = root / 'GAT_crypto_ablation_10_seeds'
    paths = {s: ablation / f'seed_{s:04d}' / 'predicciones_test.npz' for s in POSTHOC_SEEDS}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('Faltan predicciones. No se entrenara nada: ' + ', '.join(missing))
    manifest = {
        'version': POSTHOC_VERSION, 'training_performed': False,
        'prediction_source': 'predicciones_test.npz de ablacion de diez semillas',
        'test_reused': True, 'new_external_validation': False,
        'reference_path': str(reference_path), 'reference_sha256': ph_hash(reference_path),
        'config_path': str(config_path), 'config_sha256': ph_hash(config_path),
        'config': cfg, 'seeds': POSTHOC_SEEDS, 'assets': assets.tolist(),
        'n_test': len(t), 'target_start': str(dates[0]), 'target_end': str(dates[-1]),
        'timestamps_source': 'config + indices t; legacy object array not unpickled' if saved_dates is None else 'saved timestamps checked against config and t',
        'sources': [],
    }
    return dict(root=root, cfg=cfg, assets=assets, t=t, target=target,
                target_raw=target_raw, dates=dates, paths=paths, manifest=manifest)


def ph_load_seed(ctx, seed):
    path = ctx['paths'][seed]
    digest = ph_hash(path)
    with np.load(path, allow_pickle=False) as saved:
        if not np.array_equal(saved['t'], ctx['t']):
            raise ValueError(f'Semilla {seed}: distinto orden temporal.')
        seed_target_raw = saved['target'].copy()
        seed_target = ph_prob(seed_target_raw)
        reference = ctx.get('ablation_target_reference', ctx['target'])
        # El export base convierte a float32; el dataset puede producir float64
        # al dividir por una suma float64. Comparar el simplex, no sus bits.
        # Tolerancia relativa de cuatro eps float32, sin tolerancia absoluta:
        # un cero no se acepta como una cuota positiva, por diminuta que sea.
        tolerance = 4 * np.finfo(np.float32).eps
        if seed_target.shape != reference.shape:
            raise ValueError(f'Semilla {seed}: dimensiones del target incompatibles.')
        delta = np.abs(seed_target - reference)
        scale = np.maximum(np.abs(seed_target), np.abs(reference))
        same_support = np.array_equal(seed_target == 0, reference == 0)
        close = same_support and np.all(delta <= tolerance * scale)
        max_abs = float(delta.max())
        max_rel = float(np.max(np.divide(delta, scale, out=np.zeros_like(delta), where=scale > 0)))
        if not close:
            raise ValueError(
                f'Semilla {seed}: discrepancia real de target; no se mezclan archivos. '
                f'Max abs={max_abs:.6g}, max rel={max_rel:.6g}, '
                f'mismo soporte={same_support}, tolerancia rel={tolerance:.6g}. '
                'Revisar export base y ablacion; no hace falta reentrenar para diagnosticar.'
            )
        ctx['manifest'].setdefault('target_alignment', []).append({
            'seed': int(seed), 'comparison': 'normalized simplex, identical zero support',
            'relative_tolerance': float(tolerance), 'absolute_tolerance': 0.,
            'max_absolute_difference': max_abs, 'max_relative_difference': max_rel,
            'reference': 'first ablation seed' if 'ablation_target_reference' in ctx else 'base export',
        })
        if 'ablation_target_reference' not in ctx:
            # Evaluar contra los valores reales del mismo paquete de predicciones.
            ctx['ablation_target_reference'] = seed_target.copy()
            ctx['target'] = seed_target.copy()
            ctx['manifest']['metric_target_source'] = str(path)
        if max_abs > 0:
            print(f'Semilla {seed}: redondeo compatible verificado; diferencia maxima {max_abs:.3g}.', flush=True)
        pred = {name: ph_prob(saved[key]) for name, key in POSTHOC_MODELS.items()}
    for name, values in pred.items():
        if values.shape != ctx['target'].shape:
            raise ValueError(f'{seed}/{name}: dimensiones incompatibles.')
    if 'persistence_reference' not in ctx:
        ctx['persistence_reference'] = pred['Persistencia'].copy()
    elif not np.array_equal(pred['Persistencia'], ctx['persistence_reference']):
        raise ValueError('Persistencia cambio entre semillas; no se mezclan experimentos.')
    ctx['manifest']['sources'].append({'seed': seed, 'path': str(path), 'sha256': digest})
    return pred


def ph_metadata(model, seed):
    return dict(Modelo=model, seed=int(seed), modelo_entrenado=False,
                familia='Benchmark' if model == 'Persistencia' else 'CryptoGAT',
                npp_aristas=model in {'GAT NPP solo aristas', 'GAT + NPP'},
                npp_loss=model == 'GAT + NPP')


def ph_metrics(target, pred, eps):
    target = ph_prob(target)
    pred = np.maximum(ph_prob(pred), eps)
    pred /= pred.sum(axis=1, keepdims=True)
    safe = np.maximum(target, eps)
    lp, ly = np.log(pred), np.log(safe)
    ce = -(target * lp).sum(axis=1)
    entropy = -(target * ly).sum(axis=1)
    mid = .5 * (safe + pred)
    js = .5 * (target * (ly - np.log(mid))).sum(axis=1)
    js += .5 * (pred * (lp - np.log(mid))).sum(axis=1)
    clr_error = (ly - ly.mean(axis=1, keepdims=True)) - (lp - lp.mean(axis=1, keepdims=True))
    error = pred - target
    return dict(CE=float(ce.mean()), KL=float((ce - entropy).mean()), JS=float(js.mean()),
                MAE=float(np.abs(error).mean()), RMSE=float(np.sqrt(np.square(error).mean())),
                Aitchison=float(np.sqrt(np.square(clr_error).mean(axis=1)).mean()),
                CLR_MSE=float(np.square(clr_error).mean()))


def ph_temporal(root, window='1D'):
    ctx = ph_context(root)
    duration = pd.Timedelta(window)
    if duration <= pd.Timedelta(0):
        raise ValueError('La ventana debe ser positiva.')
    groups = np.asarray((ctx['dates'] - ctx['dates'][0]) // duration)
    windows = [np.flatnonzero(groups == value) for value in np.unique(groups)]
    specification = {'window': str(duration), 'origin': str(ctx['dates'][0]),
                     'evaluation': 'ventanas no solapadas y acumuladas; pesos congelados'}
    tag = hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()[:10]
    out = ctx['root'] / 'GAT_crypto_extended_audits_v1' / 'opcion_20_temporal_sin_entreno' / tag
    out.mkdir(parents=True, exist_ok=True)
    # Each evaluation writes a fresh run directory, preserving previous completed evaluations.
    out = out / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    out.mkdir()
    rows = []
    for seed in POSTHOC_SEEDS:
        print(f'Temporal: leyendo predicciones de semilla {seed}; cero entrenamientos.', flush=True)
        pred = ph_load_seed(ctx, seed)
        seed_rows = []
        for model, values in pred.items():
            for number, idx in enumerate(windows, 1):
                for kind, selected in [('bloque', idx), ('acumulada', np.arange(idx[-1] + 1))]:
                    seed_rows.append({**ph_metadata(model, seed), 'ventana': number, 'tipo': kind,
                        'inicio': str(ctx['dates'][selected[0]]), 'fin': str(ctx['dates'][selected[-1]]),
                        'N_test': len(selected),
                        **ph_metrics(ctx['target'][selected], values[selected], float(ctx['cfg']['eps']))})
        ph_csv(pd.DataFrame(seed_rows), out / f'seed_{seed:04d}.csv')
        rows.extend(seed_rows)
    frame = pd.DataFrame(rows)
    metrics = ['CE', 'KL', 'JS', 'MAE', 'RMSE', 'Aitchison', 'CLR_MSE']
    keys = ['tipo', 'ventana', 'inicio', 'fin', 'N_test']
    summary = frame.groupby(keys + ['Modelo'], as_index=False)[metrics].agg(['mean', 'std'])
    summary.columns = ['_'.join(filter(None, map(str, c))) if isinstance(c, tuple) else c for c in summary.columns]
    ph_csv(frame, out / 'metricas_por_ventana_modelo_semilla.csv')
    ph_csv(summary, out / 'resumen_por_ventana.csv')
    pairs = []
    for a, b in [('GAT + NPP', 'GAT 100% libre'), ('GAT NPP solo aristas', 'GAT 100% libre'),
                 ('GAT + NPP', 'GAT NPP solo aristas')] + [(m, 'Persistencia') for m in POSTHOC_MODELS if m != 'Persistencia']:
        merged = frame[frame.Modelo == a].merge(frame[frame.Modelo == b], on=keys + ['seed'], suffixes=('_A', '_B'), validate='one_to_one')
        for metric in metrics:
            for _, row in merged.iterrows():
                ref = row[metric + '_B']
                pairs.append({**{k: row[k] for k in keys + ['seed']}, 'modelo_A': a, 'modelo_B': b,
                              'metrica': metric, 'delta_A_menos_B': row[metric + '_A'] - ref,
                              'mejora_pct': 100 * (ref - row[metric + '_A']) / ref if abs(ref) > 1e-15 else np.nan})
    paired = pd.DataFrame(pairs)
    ph_csv(paired, out / 'deltas_emparejados.csv')
    ctx['manifest']['specification'] = specification
    ctx['manifest']['interpretation'] = 'Retrospectivo sobre test conocido; acumuladas dependientes. SD entre semillas, no IC temporal. Persistencia repetida solo para emparejar.'
    ph_json(ctx['manifest'], out / 'manifest_completo.json')
    print('Evaluacion terminada. Resumen de bloques; menor es mejor. SD mide semillas, no tiempo.')
    display(summary[summary['tipo'] == 'bloque'])
    print('Resultados guardados en:', out)
    return out


def ph_train_statistics(ctx, warmup=12128):
    cfg = ctx['cfg']
    path = Path(cfg['dfyp_path'])
    print('Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.', flush=True)
    digest = ph_hash(path)
    frame = pd.read_parquet(path) if path.suffix.lower() in {'.parquet', '.pq'} else pd.read_csv(path)
    if 'YS' in frame.columns and 'YS_p' not in frame.columns:
        frame = frame.rename(columns=lambda c: c[:-2] + '_p' if c.endswith('_y') else 'YS_p' if c == 'YS' else c)
    cols = [c for c in frame.columns if c.endswith('_p') and c != 'YS_p']
    if cols != [a + '_p' for a in ctx['assets']]:
        raise ValueError('dfyp no conserva el universo/orden de activos de las predicciones.')
    horizon, stride, purge = int(cfg['horizon']), int(cfg['sample_stride']), int(cfg['purge'])
    test_start = len(frame) - horizon - int(cfg['test_size'])
    validation_end = test_start - purge
    validation_start = validation_end - int(cfg['validation_size'])
    train_end = validation_start - purge
    all_t = np.arange(warmup, len(frame) - horizon, stride)
    train_t = all_t[all_t < train_end]
    val_t = all_t[(all_t >= validation_start) & (all_t < validation_end)]
    test_t = all_t[all_t >= test_start]
    if not np.array_equal(test_t, ctx['t']):
        raise ValueError('dfyp/config no reproduce el test archivado; no se recalculan cortes silenciosamente.')
    if len(train_t) < 2 or len(val_t) == 0 or train_t[-1] + horizon >= val_t[0] or val_t[-1] + horizon >= test_t[0]:
        raise ValueError('Split vacio o purga incompatible.')
    # Identical normalization to the original extraction; only train targets fit thresholds.
    def normalize_rows(indices):
        values = frame.iloc[indices][cols].to_numpy(dtype=np.float32, copy=True)
        if np.any(values < -1e-8):
            raise ValueError('dfyp contiene masas negativas.')
        values = np.nan_to_num(values, nan=0., posinf=0., neginf=0.)
        values = np.clip(values, 0., None)
        total = values.sum(axis=1, keepdims=True, dtype=np.float64)
        if np.any(total <= 0):
            raise ValueError('dfyp contiene filas sin masa.')
        return (values / total).astype(np.float32)
    if not np.array_equal(normalize_rows(test_t + horizon), ctx['target_raw']):
        raise ValueError('dfyp fue alterado o pertenece a otra corrida: target test diferente.')
    # Validate persistence against raw inputs before fitting any bins.
    with np.load(ctx['paths'][POSTHOC_SEEDS[0]], allow_pickle=False) as saved:
        if not np.array_equal(normalize_rows(test_t), saved['persistencia']):
            raise ValueError('Persistencia archivada no corresponde al dfyp actual.')
    train = ph_prob(normalize_rows(train_t + horizon))
    mu = train.mean(axis=0)
    sigma = train.std(axis=0, ddof=1)
    ctx['manifest']['train_bins'] = {
        'dfyp_path': str(path), 'dfyp_sha256': digest, 'warmup': warmup,
        'fit_rows': 'targets P[train_t + horizon]', 'ddof': 1, 'n_train': len(train_t),
        'train_origin_first': int(train_t[0]), 'train_origin_last': int(train_t[-1]),
        'last_train_target': int(train_t[-1] + horizon), 'validation_first_origin': int(val_t[0]),
        'test_first_origin': int(test_t[0]), 'fit_uses_validation_or_test': False,
        'legacy_provenance': 'Config y assets de resultados base, alineados exactamente con target y persistencia de ablacion. Warmup original fijo 12128; archivos antiguos sin hash historico de train.'}
    return train, mu, sigma, train_t + horizon


def ph_bin_edges(mu, sigma, z_cuts):
    edges = []
    # float32 normalization can add tiny jitter to otherwise constant series.
    constant = sigma <= 1e-8
    for mean, std, is_constant in zip(mu, sigma, constant):
        if is_constant:
            # Three explicit classes around the constant train value, tolerance in share units.
            cuts = [mean - 1e-8, mean + 1e-8]
        else:
            cuts = mean + np.asarray(z_cuts) * std
        edges.append(np.unique([float(c) for c in cuts if 0 < c < 1]))
    return edges, constant


def ph_labels(values, edges):
    return np.column_stack([np.searchsorted(cuts, values[:, j], side='right')
                            for j, cuts in enumerate(edges)]).astype(np.uint8)


def ph_class_metrics(actual, predicted, n_classes):
    confusion = np.bincount(actual.astype(int) * n_classes + predicted.astype(int),
                            minlength=n_classes ** 2).reshape(n_classes, n_classes)
    support, guessed = confusion.sum(axis=1), confusion.sum(axis=0)
    tp = confusion.diagonal()
    recall = np.divide(tp, support, out=np.zeros(n_classes, float), where=support > 0)
    f1 = np.divide(2 * tp, support + guessed, out=np.zeros(n_classes, float), where=(support + guessed) > 0)
    delta = np.abs(actual.astype(int) - predicted.astype(int))
    result = dict(accuracy=float(np.mean(actual == predicted)),
                  balanced_accuracy=float(recall[support > 0].mean()),
                  macro_f1=float(f1[(support + guessed) > 0].mean()),
                  error_ordinal=float(delta.mean()), dentro_un_bin=float((delta <= 1).mean()),
                  tasa_subestimacion=float((predicted < actual).mean()),
                  tasa_sobreestimacion=float((predicted > actual).mean()),
                  n_clases=n_classes, clases_reales_presentes=int((support > 0).sum()))
    return result, confusion


def ph_bins(root, z_cuts=(-3., -2., -1., 0., 1., 2., 3.)):
    z_cuts = np.asarray(z_cuts, dtype=float)
    if z_cuts.ndim != 1 or len(z_cuts) == 0 or len(z_cuts) > 250 or not np.isfinite(z_cuts).all() or np.any(np.diff(z_cuts) <= 0):
        raise ValueError('Cortes z deben ser finitos, estrictamente crecientes y tener longitud 1..250.')
    ctx = ph_context(root)
    train, mu, sigma, train_rows = ph_train_statistics(ctx)
    edges, constant = ph_bin_edges(mu, sigma, z_cuts)
    train_labels, actual = ph_labels(train, edges), ph_labels(ctx['target'], edges)
    specification = {'z_cuts': z_cuts.tolist(), 'constant_sigma_tolerance': 1e-8,
                     'constant_mass_tolerance': 1e-8, 'edge_rule': 'lower <= p < upper; last includes 1'}
    tag = hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()[:10]
    out = ctx['root'] / 'GAT_crypto_extended_audits_v1' / 'opcion_21_bins_sin_entreno' / tag
    out.mkdir(parents=True, exist_ok=True)
    out = out / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    out.mkdir()
    threshold_rows, occupancy = [], []
    for j, cuts in enumerate(edges):
        limits = [0.] + cuts.tolist() + [1.]
        for label in range(len(cuts) + 1):
            threshold_rows.append(dict(Nodo=ctx['assets'][j], bin=label, inferior=limits[label],
                superior=limits[label + 1], superior_inclusivo=label == len(cuts),
                mu_train=mu[j], sigma_train=sigma[j], constante_train=bool(constant[j])))
            occupancy.append(dict(Nodo=ctx['assets'][j], bin=label,
                n_train=int((train_labels[:, j] == label).sum()), n_test=int((actual[:, j] == label).sum())))
    ph_csv(pd.DataFrame(threshold_rows), out / 'umbrales_train_por_nodo.csv')
    ph_csv(pd.DataFrame(occupancy), out / 'ocupacion_train_test.csv')
    ph_npz(out / 'etiquetas_reales.npz', assets=ctx['assets'], train_target_rows=train_rows,
           train_labels=train_labels, test_labels=actual, t=ctx['t'], timestamps=np.asarray(ctx['dates'].astype(str)))
    del train, train_labels
    rows = []
    class_names = ['accuracy', 'balanced_accuracy', 'macro_f1', 'error_ordinal', 'dentro_un_bin',
                   'tasa_subestimacion', 'tasa_sobreestimacion']
    for seed in POSTHOC_SEEDS:
        print(f'Bins: etiquetando predicciones guardadas de semilla {seed}; cero entrenamientos.', flush=True)
        predictions = ph_load_seed(ctx, seed)
        seed_rows, confusion_rows, encoded = [], [], {}
        for model, values in predictions.items():
            predicted = ph_labels(values, edges)
            encoded[POSTHOC_MODELS[model]] = predicted
            for j, cuts in enumerate(edges):
                metrics, confusion = ph_class_metrics(actual[:, j], predicted[:, j], len(cuts) + 1)
                meta = {**ph_metadata(model, seed), 'Nodo': ctx['assets'][j], 'constante_train': bool(constant[j])}
                seed_rows.append({**meta, **metrics,
                                  'MAE_continuo': float(np.abs(values[:, j] - ctx['target'][:, j]).mean()),
                                  'RMSE_continuo': float(np.sqrt(np.square(values[:, j] - ctx['target'][:, j]).mean()))})
                for a, b in zip(*np.nonzero(confusion)):
                    confusion_rows.append({**meta, 'bin_real': int(a), 'bin_predicho': int(b), 'n': int(confusion[a, b])})
        ph_csv(pd.DataFrame(seed_rows), out / f'metricas_por_nodo_seed_{seed:04d}.csv')
        ph_csv(pd.DataFrame(confusion_rows), out / f'confusion_por_nodo_seed_{seed:04d}.csv')
        ph_npz(out / f'predicciones_bins_seed_{seed:04d}.npz', seed=np.array(seed), assets=ctx['assets'],
               t=ctx['t'], target_bins=actual, **encoded)
        rows.extend(seed_rows)
    frame = pd.DataFrame(rows)
    metrics = class_names + ['MAE_continuo', 'RMSE_continuo']
    # Macro average over nodes: bin IDs refer to different thresholds for each asset.
    summary_frames = []
    for cohort, selected in [('todos', frame), ('variables_en_train', frame[~frame.constante_train]),
                             ('constantes_en_train', frame[frame.constante_train])]:
        if len(selected):
            summary_frames.append(selected.groupby(['Modelo', 'seed'], as_index=False)[metrics].mean().assign(cohorte=cohort))
    summary_seed = pd.concat(summary_frames, ignore_index=True)
    summary = summary_seed.groupby(['cohorte', 'Modelo'], as_index=False)[metrics].agg(['mean', 'std'])
    summary.columns = ['_'.join(filter(None, map(str, c))) if isinstance(c, tuple) else c for c in summary.columns]
    ph_csv(frame, out / 'metricas_por_nodo_todas_semillas.csv')
    ph_csv(summary_seed, out / 'metricas_macro_por_semilla.csv')
    ph_csv(summary, out / 'resumen_por_modelo.csv')
    paired_rows = []
    for reference in ['Persistencia', 'GAT 100% libre']:
        base = summary_seed[summary_seed.Modelo == reference]
        for model in POSTHOC_MODELS:
            if model == reference:
                continue
            merged = summary_seed[summary_seed.Modelo == model].merge(base, on=['seed', 'cohorte'], suffixes=('_A', '_B'), validate='one_to_one')
            for _, row in merged.iterrows():
                for metric in class_names:
                    paired_rows.append(dict(Modelo=model, referencia=reference, seed=int(row.seed), cohorte=row.cohorte,
                        metrica=metric, delta_A_menos_B=float(row[metric + '_A'] - row[metric + '_B']),
                        direccion_favorable='menor' if metric == 'error_ordinal' else 'descriptiva' if metric.startswith('tasa_') else 'mayor'))
    ph_csv(pd.DataFrame(paired_rows), out / 'deltas_emparejados.csv')
    ctx['manifest']['specification'] = specification
    ctx['manifest']['interpretation'] = 'Discretizacion de predicciones continuas preexistentes; sin entrenar clasificador. Train fija umbrales y test solo se etiqueta. IDs de bins son locales a cada activo; no implican normalidad ni probabilidades calibradas. No reconstruye otro simplex ni calcula CE/KL de clases.'
    ph_json(ctx['manifest'], out / 'manifest_completo.json')
    print('Bins terminados. Accuracy/F1/balanced accuracy: mayor mejor; error ordinal: menor mejor.')
    display(summary)
    print('Resultados guardados en:', out)
    return out

RESULTADOS_TEMPORALES = ph_temporal(CARPETA_PROYECTO, VENTANA_TEMPORAL)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Temporal: leyendo predicciones de semilla 11; cero entrenamientos.
Semilla 11: redondeo compatible verificado; diferencia maxima 8.91e-09.
Temporal: leyendo predicciones de semilla 23; cero entrenamientos.
Temporal: leyendo predicciones de semilla 42; cero entrenamientos.
Temporal: leyendo predicciones de semilla 67; cero entrenamientos.
Temporal: leyendo predicciones de semilla 101; cero entrenamientos.
Temporal: leyendo predicciones de semilla 137; cero entrenamientos.
Temporal: leyendo predicciones de semilla 211; cero entrenamientos.
Temporal: leyendo predicciones de semilla 307; cero entrenamientos.
Temporal: leyendo predicciones de semilla 401; cero entrenamientos.
Temporal: leyendo predicciones de semilla 503; cero entrenamientos.
Evaluacion terminada. Resumen de bloques; menor es mejor. SD mide semillas, no tiempo.


,tipo,ventana,inicio,fin,N_test,Modelo,CE_mean,CE_std,KL_mean,KL_std,JS_mean,JS_std,MAE_mean,MAE_std,RMSE_mean,RMSE_std,Aitchison_mean,Aitchison_std,CLR_MSE_mean,CLR_MSE_std
56,bloque,1,2026-06-17 00:00:00+00:00,2026-06-17 23:50:00+00:00,144,GAT + NPP,2.999092,0.000408,0.160040,0.000408,0.034030,0.000110,0.000762,1.367685e-06,0.006181,0.000017,4.503114,0.030273,20.341691,0.270710
57,bloque,1,2026-06-17 00:00:00+00:00,2026-06-17 23:50:00+00:00,144,GAT 100% libre,2.999550,0.000635,0.160498,0.000635,0.034093,0.000159,0.000765,2.524265e-06,0.006197,0.000023,4.408764,0.032667,19.504500,0.287318
58,bloque,1,2026-06-17 00:00:00+00:00,2026-06-17 23:50:00+00:00,144,GAT NPP solo aristas,2.999504,0.000583,0.160452,0.000583,0.033891,0.000152,0.000762,2.047598e-06,0.006195,0.000023,4.425057,0.038992,19.648223,0.344469
59,bloque,1,2026-06-17 00:00:00+00:00,2026-06-17 23:50:00+00:00,144,Persistencia,3.068138,0.000000,0.229086,0.000000,0.043949,0.000000,0.000881,0.000000e+00,0.007280,0.000000,2.481773,0.000000,6.588901,0.000000
60,bloque,2,2026-06-18 00:00:00+00:00,2026-06-18 23:50:00+00:00,144,GAT + NPP,2.908969,0.000569,0.164765,0.000569,0.034563,0.000121,0.000780,1.864635e-06,0.006635,0.000028,4.621709,0.027201,21.456026,0.248764
61,bloque,2,2026-06-18 00:00:00+00:00,2026-06-18 23:50:00+00:00,144,GAT 100% libre,2.910103,0.000354,0.165899,0.000354,0.034710,0.000093,0.000782,2.224661e-06,0.006634,0.000017,4.525357,0.031745,20.581672,0.285457
62,bloque,2,2026-06-18 00:00:00+00:00,2026-06-18 23:50:00+00:00,144,GAT NPP solo aristas,2.910007,0.000367,0.165803,0.000367,0.034595,0.000126,0.000781,2.261012e-06,0.006658,0.000023,4.548759,0.035803,20.792396,0.324941
63,bloque,2,2026-06-18 00:00:00+00:00,2026-06-18 23:50:00+00:00,144,Persistencia,2.998791,0.000000,0.254587,0.000000,0.045788,0.000000,0.000909,0.000000e+00,0.007955,0.000000,2.763123,0.000000,8.139768,0.000000
64,bloque,3,2026-06-19 00:00:00+00:00,2026-06-19 23:50:00+00:00,144,GAT + NPP,3.104243,0.001006,0.186306,0.001006,0.038557,0.000108,0.000846,1.604064e-06,0.007009,0.000026,4.822104,0.026523,23.322563,0.253930
65,bloque,3,2026-06-19 00:00:00+00:00,2026-06-19 23:50:00+00:00,144,GAT 100% libre,3.105314,0.001032,0.187377,0.001032,0.038555,0.000211,0.000847,2.250540e-06,0.007003,0.000025,4.740006,0.029922,22.541355,0.282872


Resultados guardados en: /content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1/opcion_20_temporal_sin_entreno/efccd6f5d8/20260907T232129741100Z


In [ ]:
# Ventaja por dias
# | Comparación | CE/KL | JS | MAE | RMSE |
# |---|---:|---:|---:|---:|
# | Cada GAT supera a persistencia | 14/14 | 14/14 | 14/14 | 14/14 |
# | NPP solo aristas supera al libre | 14/14 | 14/14 | 14/14 | 7/14 |
# | GAT + NPP supera al libre | 14/14 | 12/14 | 12/14 | 8/14 |
# | GAT + NPP supera a NPP solo aristas | 12/14 | 5/14 | 6/14 | 11/14 |

# El aporte de las aristas NPP es pequeño, pero temporalmente consistente en
# CE, KL, JS y MAE. En cambio, no mejora sistemáticamente el RMSE:
# gana siete días y pierde siete; en el agregado queda un 0,033% peor que el libre.


# Dentro del test estudiado, los GAT mejoran de manera sostenida la predicción de
# masa frente a persistencia. Incorporar esta primera aproximacion de
# NPP aporta mejoras marginales, especialmente consistentes mediante las aristas,
# pero no domina todas las métricas.
# El deterioro en Aitchison también es sistemático.

## 21. Predicciones en bins definidos en train — sin reentrenamiento

Ejecutar **solo la siguiente celda**, incluso sin ejecutar la sección 20. CPU es suficiente. Se discretizan las predicciones continuas ya disponibles: no se cambia el target ni se entrena un clasificador nuevo.

Para cada nodo j se calculan media y desvío muestral (ddof=1) exclusivamente sobre sus **targets originales de train**, `P[train_t + horizon, j]`. Los cortes por defecto son `mu_j + k * sigma_j`, con k en −3, −2, −1, 0, 1, 2, 3. Se conservan únicamente cortes estrictamente interiores a (0, 1), se eliminan duplicados y se usa la misma partición congelada para etiquetar train, test y los cuatro pronósticos. Los intervalos son cerrados a izquierda, abiertos a derecha; el último incluye 1. Estos bins no suponen normalidad ni tienen la misma amplitud entre activos.

Los nodos con sigma <= 1e-8 se tratan como constantes/casi constantes en train, con cortes a mu ± 1e-8 dentro de (0,1). Esa tolerancia absorbe pequeñas diferencias de normalización float32 y se registra en el manifiesto. Se muestran por separado todos los nodos, los variables y los constantes en train.

**Entradas:** los mismos archivos de predicciones que la sección 20, más solo el `dfyp_path` del config guardado. No se leen precio, volumen ni las matrices de features. Se reconstruye el corte original con warmup 12128 y la configuración archivada, se exige identidad de targets/persistencia de test y se verifica la purga. Si el archivo cambió de longitud o contenido incompatible, se detiene. El hash actual de dfyp se conserva para reproducibilidad; los archivos antiguos no permiten certificar retrospectivamente que train nunca cambió.

**Comparación:** accuracy, balanced accuracy sobre clases reales presentes, macro-F1 sobre clases presentes en target o predicción, error ordinal, acierto dentro de un bin y tasas de sobre/subestimación. Se conservan MAE/RMSE continuos por nodo. Los resúmenes son promedios por nodo; en particular el promedio de RMSE por nodo no es el RMSE global del simplex. Los deltas de accuracy son diferencias de proporciones: multiplicar por 100 para puntos porcentuales. Un mejor acierto puede reflejar discretización y desbalance: consultar ocupación train/test y matrices de confusión. No hay probabilidades de clase calibradas ni CE/KL de clasificación; las etiquetas no se renormalizan para fabricar un nuevo simplex.

**Salidas:** umbrales y ocupación por nodo, etiquetas reales de train/test en NPZ, etiquetas predichas por modelo/semilla en NPZ, métricas y confusiones por nodo/semilla, resúmenes, deltas y manifiesto. Ruta: `GAT_crypto_extended_audits_v1/opcion_21_bins_sin_entreno/`, con subcarpetas por especificación y ejecución. Cada semilla se guarda al terminar su evaluación. El manifiesto completo se escribe al final; los resultados previos se conservan.

La primera lectura de dfyp puede tardar. No recalcula indicadores, no cambia los splits, no usa shuffle y no reentrena ninguna combinación. La evaluación reutiliza test conocido y es descriptiva; no constituye una nueva validación externa.


In [ ]:
# Ejecutar esta celda sola. Umbrales calculados exclusivamente en train.
CARPETA_PROYECTO = '/content/drive/MyDrive/Neural/NPP/Cripto'
CORTES_SIGMA = [-3., -2., -1., 0., 1., 2., 3.]

import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)


POSTHOC_VERSION = 'frozen_predictions_v1_target_precision_fix'
POSTHOC_SEEDS = [11, 23, 42, 67, 101, 137, 211, 307, 401, 503]
POSTHOC_MODELS = {
    'Persistencia': 'persistencia',
    'GAT 100% libre': 'gat_libre',
    'GAT NPP solo aristas': 'gat_npp_aristas',
    'GAT + NPP': 'gat_npp',
}


def ph_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def ph_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def ph_json(value, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)


def ph_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(tmp, path)


def ph_mount(root):
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        try:
            drive.mount('/content/drive')
            list(Path(root).iterdir())
        except OSError:
            drive.mount('/content/drive', force_remount=True)
    if not Path(root).is_dir():
        raise FileNotFoundError(f'No existe la carpeta del proyecto: {root}')


def ph_prob(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError('Predicciones/target deben ser matrices finitas y no negativas.')
    mass = values.sum(axis=1, keepdims=True)
    if np.any(mass <= 0) or not np.allclose(mass, 1., atol=1e-5, rtol=0):
        raise ValueError('El artefacto no contiene un simplex cerrado.')
    return values / mass


def ph_context(root):
    root = Path(root)
    ph_mount(root)
    base = root / 'GAT_crypto_results'
    config_path = base / 'config.json'
    reference_path = base / 'predicciones_test.npz'
    cfg = json.loads(config_path.read_text(encoding='utf-8'))
    with np.load(reference_path, allow_pickle=False) as saved:
        assets = saved['assets'].astype(str)
        t = saved['t'].copy()
        target_raw = saved['target'].copy()
        # Older exports stored timestamps as object arrays. Do not unpickle them.
        try:
            saved_dates = pd.to_datetime(saved['timestamps'].astype(str), utc=True)
        except ValueError as error:
            if 'Object arrays cannot be loaded' not in str(error):
                raise
            saved_dates = None
    if t.ndim != 1 or not np.issubdtype(t.dtype, np.integer) or len(t) < 2:
        raise ValueError('Indices temporales de referencia invalidos.')
    if np.any(np.diff(t) <= 0) or np.any(t < 0) or len(set(assets)) != len(assets):
        raise ValueError('Fechas duplicadas/desordenadas o activos duplicados.')
    target = ph_prob(target_raw)
    if target.shape != (len(t), len(assets)):
        raise ValueError('El orden/dimension de activos no coincide con el target.')
    step = pd.Timedelta(cfg['frequency'])
    expected_dates = pd.Timestamp(cfg['start_timestamp']) + (t + int(cfg['horizon'])) * step
    dates = pd.DatetimeIndex(expected_dates)
    if saved_dates is not None and not np.array_equal(saved_dates.asi8, dates.asi8):
        raise ValueError('Config y timestamps guardados pertenecen a experimentos distintos.')
    ablation = root / 'GAT_crypto_ablation_10_seeds'
    paths = {s: ablation / f'seed_{s:04d}' / 'predicciones_test.npz' for s in POSTHOC_SEEDS}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('Faltan predicciones. No se entrenara nada: ' + ', '.join(missing))
    manifest = {
        'version': POSTHOC_VERSION, 'training_performed': False,
        'prediction_source': 'predicciones_test.npz de ablacion de diez semillas',
        'test_reused': True, 'new_external_validation': False,
        'reference_path': str(reference_path), 'reference_sha256': ph_hash(reference_path),
        'config_path': str(config_path), 'config_sha256': ph_hash(config_path),
        'config': cfg, 'seeds': POSTHOC_SEEDS, 'assets': assets.tolist(),
        'n_test': len(t), 'target_start': str(dates[0]), 'target_end': str(dates[-1]),
        'timestamps_source': 'config + indices t; legacy object array not unpickled' if saved_dates is None else 'saved timestamps checked against config and t',
        'sources': [],
    }
    return dict(root=root, cfg=cfg, assets=assets, t=t, target=target,
                target_raw=target_raw, dates=dates, paths=paths, manifest=manifest)


def ph_load_seed(ctx, seed):
    path = ctx['paths'][seed]
    digest = ph_hash(path)
    with np.load(path, allow_pickle=False) as saved:
        if not np.array_equal(saved['t'], ctx['t']):
            raise ValueError(f'Semilla {seed}: distinto orden temporal.')
        seed_target_raw = saved['target'].copy()
        seed_target = ph_prob(seed_target_raw)
        reference = ctx.get('ablation_target_reference', ctx['target'])
        # El export base convierte a float32; el dataset puede producir float64
        # al dividir por una suma float64. Comparar el simplex, no sus bits.
        # Tolerancia relativa de cuatro eps float32, sin tolerancia absoluta:
        # un cero no se acepta como una cuota positiva, por diminuta que sea.
        tolerance = 4 * np.finfo(np.float32).eps
        if seed_target.shape != reference.shape:
            raise ValueError(f'Semilla {seed}: dimensiones del target incompatibles.')
        delta = np.abs(seed_target - reference)
        scale = np.maximum(np.abs(seed_target), np.abs(reference))
        same_support = np.array_equal(seed_target == 0, reference == 0)
        close = same_support and np.all(delta <= tolerance * scale)
        max_abs = float(delta.max())
        max_rel = float(np.max(np.divide(delta, scale, out=np.zeros_like(delta), where=scale > 0)))
        if not close:
            raise ValueError(
                f'Semilla {seed}: discrepancia real de target; no se mezclan archivos. '
                f'Max abs={max_abs:.6g}, max rel={max_rel:.6g}, '
                f'mismo soporte={same_support}, tolerancia rel={tolerance:.6g}. '
                'Revisar export base y ablacion; no hace falta reentrenar para diagnosticar.'
            )
        ctx['manifest'].setdefault('target_alignment', []).append({
            'seed': int(seed), 'comparison': 'normalized simplex, identical zero support',
            'relative_tolerance': float(tolerance), 'absolute_tolerance': 0.,
            'max_absolute_difference': max_abs, 'max_relative_difference': max_rel,
            'reference': 'first ablation seed' if 'ablation_target_reference' in ctx else 'base export',
        })
        if 'ablation_target_reference' not in ctx:
            # Evaluar contra los valores reales del mismo paquete de predicciones.
            ctx['ablation_target_reference'] = seed_target.copy()
            ctx['target'] = seed_target.copy()
            ctx['manifest']['metric_target_source'] = str(path)
        if max_abs > 0:
            print(f'Semilla {seed}: redondeo compatible verificado; diferencia maxima {max_abs:.3g}.', flush=True)
        pred = {name: ph_prob(saved[key]) for name, key in POSTHOC_MODELS.items()}
    for name, values in pred.items():
        if values.shape != ctx['target'].shape:
            raise ValueError(f'{seed}/{name}: dimensiones incompatibles.')
    if 'persistence_reference' not in ctx:
        ctx['persistence_reference'] = pred['Persistencia'].copy()
    elif not np.array_equal(pred['Persistencia'], ctx['persistence_reference']):
        raise ValueError('Persistencia cambio entre semillas; no se mezclan experimentos.')
    ctx['manifest']['sources'].append({'seed': seed, 'path': str(path), 'sha256': digest})
    return pred


def ph_metadata(model, seed):
    return dict(Modelo=model, seed=int(seed), modelo_entrenado=False,
                familia='Benchmark' if model == 'Persistencia' else 'CryptoGAT',
                npp_aristas=model in {'GAT NPP solo aristas', 'GAT + NPP'},
                npp_loss=model == 'GAT + NPP')


def ph_metrics(target, pred, eps):
    target = ph_prob(target)
    pred = np.maximum(ph_prob(pred), eps)
    pred /= pred.sum(axis=1, keepdims=True)
    safe = np.maximum(target, eps)
    lp, ly = np.log(pred), np.log(safe)
    ce = -(target * lp).sum(axis=1)
    entropy = -(target * ly).sum(axis=1)
    mid = .5 * (safe + pred)
    js = .5 * (target * (ly - np.log(mid))).sum(axis=1)
    js += .5 * (pred * (lp - np.log(mid))).sum(axis=1)
    clr_error = (ly - ly.mean(axis=1, keepdims=True)) - (lp - lp.mean(axis=1, keepdims=True))
    error = pred - target
    return dict(CE=float(ce.mean()), KL=float((ce - entropy).mean()), JS=float(js.mean()),
                MAE=float(np.abs(error).mean()), RMSE=float(np.sqrt(np.square(error).mean())),
                Aitchison=float(np.sqrt(np.square(clr_error).mean(axis=1)).mean()),
                CLR_MSE=float(np.square(clr_error).mean()))


def ph_temporal(root, window='1D'):
    ctx = ph_context(root)
    duration = pd.Timedelta(window)
    if duration <= pd.Timedelta(0):
        raise ValueError('La ventana debe ser positiva.')
    groups = np.asarray((ctx['dates'] - ctx['dates'][0]) // duration)
    windows = [np.flatnonzero(groups == value) for value in np.unique(groups)]
    specification = {'window': str(duration), 'origin': str(ctx['dates'][0]),
                     'evaluation': 'ventanas no solapadas y acumuladas; pesos congelados'}
    tag = hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()[:10]
    out = ctx['root'] / 'GAT_crypto_extended_audits_v1' / 'opcion_20_temporal_sin_entreno' / tag
    out.mkdir(parents=True, exist_ok=True)
    # Each evaluation writes a fresh run directory, preserving previous completed evaluations.
    out = out / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    out.mkdir()
    rows = []
    for seed in POSTHOC_SEEDS:
        print(f'Temporal: leyendo predicciones de semilla {seed}; cero entrenamientos.', flush=True)
        pred = ph_load_seed(ctx, seed)
        seed_rows = []
        for model, values in pred.items():
            for number, idx in enumerate(windows, 1):
                for kind, selected in [('bloque', idx), ('acumulada', np.arange(idx[-1] + 1))]:
                    seed_rows.append({**ph_metadata(model, seed), 'ventana': number, 'tipo': kind,
                        'inicio': str(ctx['dates'][selected[0]]), 'fin': str(ctx['dates'][selected[-1]]),
                        'N_test': len(selected),
                        **ph_metrics(ctx['target'][selected], values[selected], float(ctx['cfg']['eps']))})
        ph_csv(pd.DataFrame(seed_rows), out / f'seed_{seed:04d}.csv')
        rows.extend(seed_rows)
    frame = pd.DataFrame(rows)
    metrics = ['CE', 'KL', 'JS', 'MAE', 'RMSE', 'Aitchison', 'CLR_MSE']
    keys = ['tipo', 'ventana', 'inicio', 'fin', 'N_test']
    summary = frame.groupby(keys + ['Modelo'], as_index=False)[metrics].agg(['mean', 'std'])
    summary.columns = ['_'.join(filter(None, map(str, c))) if isinstance(c, tuple) else c for c in summary.columns]
    ph_csv(frame, out / 'metricas_por_ventana_modelo_semilla.csv')
    ph_csv(summary, out / 'resumen_por_ventana.csv')
    pairs = []
    for a, b in [('GAT + NPP', 'GAT 100% libre'), ('GAT NPP solo aristas', 'GAT 100% libre'),
                 ('GAT + NPP', 'GAT NPP solo aristas')] + [(m, 'Persistencia') for m in POSTHOC_MODELS if m != 'Persistencia']:
        merged = frame[frame.Modelo == a].merge(frame[frame.Modelo == b], on=keys + ['seed'], suffixes=('_A', '_B'), validate='one_to_one')
        for metric in metrics:
            for _, row in merged.iterrows():
                ref = row[metric + '_B']
                pairs.append({**{k: row[k] for k in keys + ['seed']}, 'modelo_A': a, 'modelo_B': b,
                              'metrica': metric, 'delta_A_menos_B': row[metric + '_A'] - ref,
                              'mejora_pct': 100 * (ref - row[metric + '_A']) / ref if abs(ref) > 1e-15 else np.nan})
    paired = pd.DataFrame(pairs)
    ph_csv(paired, out / 'deltas_emparejados.csv')
    ctx['manifest']['specification'] = specification
    ctx['manifest']['interpretation'] = 'Retrospectivo sobre test conocido; acumuladas dependientes. SD entre semillas, no IC temporal. Persistencia repetida solo para emparejar.'
    ph_json(ctx['manifest'], out / 'manifest_completo.json')
    print('Evaluacion terminada. Resumen de bloques; menor es mejor. SD mide semillas, no tiempo.')
    display(summary[summary['tipo'] == 'bloque'])
    print('Resultados guardados en:', out)
    return out


def ph_check_source(ctx, left, right, label):
    left, right = ph_prob(left), ph_prob(right)
    if left.shape != right.shape:
        raise ValueError(f'{label}: dimensiones incompatibles.')
    delta = np.abs(left - right)
    scale = np.maximum(np.abs(left), np.abs(right))
    tolerance = 4 * np.finfo(np.float32).eps
    same_support = np.array_equal(left == 0, right == 0)
    if not same_support or not np.all(delta <= tolerance * scale):
        raise ValueError(f'{label}: discrepancia real, max abs={delta.max():.6g}; no se mezclan datos.')
    ctx['manifest'].setdefault('source_alignment', []).append({
        'comparison': label, 'relative_tolerance': float(tolerance),
        'absolute_tolerance': 0., 'identical_zero_support': True,
        'max_absolute_difference': float(delta.max()),
    })


def ph_train_statistics(ctx, warmup=12128):
    cfg = ctx['cfg']
    path = Path(cfg['dfyp_path'])
    print('Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.', flush=True)
    digest = ph_hash(path)
    frame = pd.read_parquet(path) if path.suffix.lower() in {'.parquet', '.pq'} else pd.read_csv(path)
    if 'YS' in frame.columns and 'YS_p' not in frame.columns:
        frame = frame.rename(columns=lambda c: c[:-2] + '_p' if c.endswith('_y') else 'YS_p' if c == 'YS' else c)
    cols = [c for c in frame.columns if c.endswith('_p') and c != 'YS_p']
    if cols != [a + '_p' for a in ctx['assets']]:
        raise ValueError('dfyp no conserva el universo/orden de activos de las predicciones.')
    horizon, stride, purge = int(cfg['horizon']), int(cfg['sample_stride']), int(cfg['purge'])
    test_start = len(frame) - horizon - int(cfg['test_size'])
    validation_end = test_start - purge
    validation_start = validation_end - int(cfg['validation_size'])
    train_end = validation_start - purge
    all_t = np.arange(warmup, len(frame) - horizon, stride)
    train_t = all_t[all_t < train_end]
    val_t = all_t[(all_t >= validation_start) & (all_t < validation_end)]
    test_t = all_t[all_t >= test_start]
    if not np.array_equal(test_t, ctx['t']):
        raise ValueError('dfyp/config no reproduce el test archivado; no se recalculan cortes silenciosamente.')
    if len(train_t) < 2 or len(val_t) == 0 or train_t[-1] + horizon >= val_t[0] or val_t[-1] + horizon >= test_t[0]:
        raise ValueError('Split vacio o purga incompatible.')
    # Identical normalization to the original extraction; only train targets fit thresholds.
    def normalize_rows(indices):
        values = frame.iloc[indices][cols].to_numpy(dtype=np.float32, copy=True)
        if np.any(values < -1e-8):
            raise ValueError('dfyp contiene masas negativas.')
        values = np.nan_to_num(values, nan=0., posinf=0., neginf=0.)
        values = np.clip(values, 0., None)
        total = values.sum(axis=1, keepdims=True, dtype=np.float64)
        if np.any(total <= 0):
            raise ValueError('dfyp contiene filas sin masa.')
        return (values / total).astype(np.float32)
    ph_check_source(ctx, normalize_rows(test_t + horizon), ctx['target'], 'dfyp frente a target archivado')
    # Validate persistence against raw inputs before fitting any bins.
    with np.load(ctx['paths'][POSTHOC_SEEDS[0]], allow_pickle=False) as saved:
        ph_check_source(ctx, normalize_rows(test_t), saved['persistencia'], 'dfyp frente a persistencia archivada')
    train = ph_prob(normalize_rows(train_t + horizon))
    mu = train.mean(axis=0)
    sigma = train.std(axis=0, ddof=1)
    ctx['manifest']['train_bins'] = {
        'dfyp_path': str(path), 'dfyp_sha256': digest, 'warmup': warmup,
        'fit_rows': 'targets P[train_t + horizon]', 'ddof': 1, 'n_train': len(train_t),
        'train_origin_first': int(train_t[0]), 'train_origin_last': int(train_t[-1]),
        'last_train_target': int(train_t[-1] + horizon), 'validation_first_origin': int(val_t[0]),
        'test_first_origin': int(test_t[0]), 'fit_uses_validation_or_test': False,
        'legacy_provenance': 'Config y assets de resultados base, alineados con target y persistencia de ablacion, tolerancia relativa float32 documentada. Warmup original fijo 12128; archivos antiguos sin hash historico de train.'}
    return train, mu, sigma, train_t + horizon


def ph_bin_edges(mu, sigma, z_cuts):
    edges = []
    # float32 normalization can add tiny jitter to otherwise constant series.
    constant = sigma <= 1e-8
    for mean, std, is_constant in zip(mu, sigma, constant):
        if is_constant:
            # Three explicit classes around the constant train value, tolerance in share units.
            cuts = [mean - 1e-8, mean + 1e-8]
        else:
            cuts = mean + np.asarray(z_cuts) * std
        edges.append(np.unique([float(c) for c in cuts if 0 < c < 1]))
    return edges, constant


def ph_labels(values, edges):
    return np.column_stack([np.searchsorted(cuts, values[:, j], side='right')
                            for j, cuts in enumerate(edges)]).astype(np.uint8)


def ph_class_metrics(actual, predicted, n_classes):
    confusion = np.bincount(actual.astype(int) * n_classes + predicted.astype(int),
                            minlength=n_classes ** 2).reshape(n_classes, n_classes)
    support, guessed = confusion.sum(axis=1), confusion.sum(axis=0)
    tp = confusion.diagonal()
    recall = np.divide(tp, support, out=np.zeros(n_classes, float), where=support > 0)
    f1 = np.divide(2 * tp, support + guessed, out=np.zeros(n_classes, float), where=(support + guessed) > 0)
    delta = np.abs(actual.astype(int) - predicted.astype(int))
    result = dict(accuracy=float(np.mean(actual == predicted)),
                  balanced_accuracy=float(recall[support > 0].mean()),
                  macro_f1=float(f1[(support + guessed) > 0].mean()),
                  error_ordinal=float(delta.mean()), dentro_un_bin=float((delta <= 1).mean()),
                  tasa_subestimacion=float((predicted < actual).mean()),
                  tasa_sobreestimacion=float((predicted > actual).mean()),
                  n_clases=n_classes, clases_reales_presentes=int((support > 0).sum()))
    return result, confusion


def ph_bins(root, z_cuts=(-3., -2., -1., 0., 1., 2., 3.)):
    z_cuts = np.asarray(z_cuts, dtype=float)
    if z_cuts.ndim != 1 or len(z_cuts) == 0 or len(z_cuts) > 250 or not np.isfinite(z_cuts).all() or np.any(np.diff(z_cuts) <= 0):
        raise ValueError('Cortes z deben ser finitos, estrictamente crecientes y tener longitud 1..250.')
    ctx = ph_context(root)
    ph_load_seed(ctx, POSTHOC_SEEDS[0])
    ctx['manifest']['sources'].clear()
    ctx['manifest']['target_alignment_initial'] = ctx['manifest'].pop('target_alignment')
    train, mu, sigma, train_rows = ph_train_statistics(ctx)
    edges, constant = ph_bin_edges(mu, sigma, z_cuts)
    train_labels, actual = ph_labels(train, edges), ph_labels(ctx['target'], edges)
    specification = {'z_cuts': z_cuts.tolist(), 'constant_sigma_tolerance': 1e-8,
                     'constant_mass_tolerance': 1e-8, 'edge_rule': 'lower <= p < upper; last includes 1'}
    tag = hashlib.sha256(json.dumps(specification, sort_keys=True).encode()).hexdigest()[:10]
    out = ctx['root'] / 'GAT_crypto_extended_audits_v1' / 'opcion_21_bins_sin_entreno' / tag
    out.mkdir(parents=True, exist_ok=True)
    out = out / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    out.mkdir()
    threshold_rows, occupancy = [], []
    for j, cuts in enumerate(edges):
        limits = [0.] + cuts.tolist() + [1.]
        for label in range(len(cuts) + 1):
            threshold_rows.append(dict(Nodo=ctx['assets'][j], bin=label, inferior=limits[label],
                superior=limits[label + 1], superior_inclusivo=label == len(cuts),
                mu_train=mu[j], sigma_train=sigma[j], constante_train=bool(constant[j])))
            occupancy.append(dict(Nodo=ctx['assets'][j], bin=label,
                n_train=int((train_labels[:, j] == label).sum()), n_test=int((actual[:, j] == label).sum())))
    ph_csv(pd.DataFrame(threshold_rows), out / 'umbrales_train_por_nodo.csv')
    ph_csv(pd.DataFrame(occupancy), out / 'ocupacion_train_test.csv')
    ph_npz(out / 'etiquetas_reales.npz', assets=ctx['assets'], train_target_rows=train_rows,
           train_labels=train_labels, test_labels=actual, t=ctx['t'], timestamps=np.asarray(ctx['dates'].astype(str)))
    del train, train_labels
    rows = []
    class_names = ['accuracy', 'balanced_accuracy', 'macro_f1', 'error_ordinal', 'dentro_un_bin',
                   'tasa_subestimacion', 'tasa_sobreestimacion']
    for seed in POSTHOC_SEEDS:
        print(f'Bins: etiquetando predicciones guardadas de semilla {seed}; cero entrenamientos.', flush=True)
        predictions = ph_load_seed(ctx, seed)
        seed_rows, confusion_rows, encoded = [], [], {}
        for model, values in predictions.items():
            predicted = ph_labels(values, edges)
            encoded[POSTHOC_MODELS[model]] = predicted
            for j, cuts in enumerate(edges):
                metrics, confusion = ph_class_metrics(actual[:, j], predicted[:, j], len(cuts) + 1)
                meta = {**ph_metadata(model, seed), 'Nodo': ctx['assets'][j], 'constante_train': bool(constant[j])}
                seed_rows.append({**meta, **metrics,
                                  'MAE_continuo': float(np.abs(values[:, j] - ctx['target'][:, j]).mean()),
                                  'RMSE_continuo': float(np.sqrt(np.square(values[:, j] - ctx['target'][:, j]).mean()))})
                for a, b in zip(*np.nonzero(confusion)):
                    confusion_rows.append({**meta, 'bin_real': int(a), 'bin_predicho': int(b), 'n': int(confusion[a, b])})
        ph_csv(pd.DataFrame(seed_rows), out / f'metricas_por_nodo_seed_{seed:04d}.csv')
        ph_csv(pd.DataFrame(confusion_rows), out / f'confusion_por_nodo_seed_{seed:04d}.csv')
        ph_npz(out / f'predicciones_bins_seed_{seed:04d}.npz', seed=np.array(seed), assets=ctx['assets'],
               t=ctx['t'], target_bins=actual, **encoded)
        rows.extend(seed_rows)
    frame = pd.DataFrame(rows)
    metrics = class_names + ['MAE_continuo', 'RMSE_continuo']
    # Macro average over nodes: bin IDs refer to different thresholds for each asset.
    summary_frames = []
    for cohort, selected in [('todos', frame), ('variables_en_train', frame[~frame.constante_train]),
                             ('constantes_en_train', frame[frame.constante_train])]:
        if len(selected):
            summary_frames.append(selected.groupby(['Modelo', 'seed'], as_index=False)[metrics].mean().assign(cohorte=cohort))
    summary_seed = pd.concat(summary_frames, ignore_index=True)
    summary = summary_seed.groupby(['cohorte', 'Modelo'], as_index=False)[metrics].agg(['mean', 'std'])
    summary.columns = ['_'.join(filter(None, map(str, c))) if isinstance(c, tuple) else c for c in summary.columns]
    ph_csv(frame, out / 'metricas_por_nodo_todas_semillas.csv')
    ph_csv(summary_seed, out / 'metricas_macro_por_semilla.csv')
    ph_csv(summary, out / 'resumen_por_modelo.csv')
    paired_rows = []
    for reference in ['Persistencia', 'GAT 100% libre']:
        base = summary_seed[summary_seed.Modelo == reference]
        for model in POSTHOC_MODELS:
            if model == reference:
                continue
            merged = summary_seed[summary_seed.Modelo == model].merge(base, on=['seed', 'cohorte'], suffixes=('_A', '_B'), validate='one_to_one')
            for _, row in merged.iterrows():
                for metric in class_names:
                    paired_rows.append(dict(Modelo=model, referencia=reference, seed=int(row.seed), cohorte=row.cohorte,
                        metrica=metric, delta_A_menos_B=float(row[metric + '_A'] - row[metric + '_B']),
                        direccion_favorable='menor' if metric == 'error_ordinal' else 'descriptiva' if metric.startswith('tasa_') else 'mayor'))
    ph_csv(pd.DataFrame(paired_rows), out / 'deltas_emparejados.csv')
    ctx['manifest']['specification'] = specification
    ctx['manifest']['interpretation'] = 'Discretizacion de predicciones continuas preexistentes; sin entrenar clasificador. Train fija umbrales y test solo se etiqueta. IDs de bins son locales a cada activo; no implican normalidad ni probabilidades calibradas. No reconstruye otro simplex ni calcula CE/KL de clases.'
    ph_json(ctx['manifest'], out / 'manifest_completo.json')
    print('Bins terminados. Accuracy/F1/balanced accuracy: mayor mejor; error ordinal: menor mejor.')
    display(summary)
    print('Resultados guardados en:', out)
    return out

RESULTADOS_BINS = ph_bins(CARPETA_PROYECTO, CORTES_SIGMA)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Semilla 11: redondeo compatible verificado; diferencia maxima 8.91e-09.
Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.
Bins: etiquetando predicciones guardadas de semilla 11; cero entrenamientos.
Bins: etiquetando predicciones guardadas de semilla 23; cero entrenamientos.
Bins: etiquetando predicciones guardadas de semilla 42; cero entrenamientos.
Bins: etiquetando predicciones guardadas de semilla 67; cero entrenamientos.
Bins: etiquetando predicciones guardadas de semilla 101; cero entrenamientos.
Bins: etiquetando predicciones guardadas de semilla 137; cero entrenamientos.
Bins: etiquetando predicciones guardadas de semilla 211; cero entrenamientos.
Bins: etiquetando predicciones guardadas de semilla 307; cero entrenamientos.
Bins: etiquetando predicciones guardadas de semilla 401; cero entrenamientos.
Bins: etiquetand

,cohorte,Modelo,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,error_ordinal_mean,error_ordinal_std,dentro_un_bin_mean,dentro_un_bin_std,tasa_subestimacion_mean,tasa_subestimacion_std,tasa_sobreestimacion_mean,tasa_sobreestimacion_std,MAE_continuo_mean,MAE_continuo_std,RMSE_continuo_mean,RMSE_continuo_std
0,constantes_en_train,GAT + NPP,0.372665,0.000000,0.354167,0.000000,0.225718,0.000000,0.627335,0.000000,1.000000,0.000000,0.000000,0.000000,0.627335,0.000000,0.000412,3.789345e-06,0.000922,0.000005
1,constantes_en_train,GAT 100% libre,0.372665,0.000000,0.354167,0.000000,0.225718,0.000000,0.627335,0.000000,1.000000,0.000000,0.000000,0.000000,0.627335,0.000000,0.000420,5.772289e-06,0.000930,0.000009
2,constantes_en_train,GAT NPP solo aristas,0.372665,0.000000,0.354167,0.000000,0.225718,0.000000,0.627335,0.000000,1.000000,0.000000,0.000000,0.000000,0.627335,0.000000,0.000417,4.379228e-06,0.000927,0.000008
3,constantes_en_train,Persistencia,0.984458,0.000000,0.884515,0.000000,0.885269,0.000000,0.015542,0.000000,1.000000,0.000000,0.007875,0.000000,0.007668,0.000000,0.000460,0.000000e+00,0.001033,0.000000
4,todos,GAT + NPP,0.789124,0.002733,0.379128,0.001279,0.372781,0.001437,0.237191,0.002710,0.981073,0.000375,0.074193,0.001057,0.136683,0.003734,0.000843,6.315994e-07,0.001732,0.000002
5,todos,GAT 100% libre,0.788195,0.002691,0.380254,0.001516,0.370382,0.002421,0.238786,0.002897,0.980468,0.000356,0.074206,0.001358,0.137600,0.003950,0.000844,1.204516e-06,0.001733,0.000002
6,todos,GAT NPP solo aristas,0.794816,0.001869,0.376919,0.001490,0.369875,0.002941,0.231757,0.001989,0.980934,0.000196,0.076766,0.001244,0.128418,0.003017,0.000842,1.137329e-06,0.001731,0.000002
7,todos,Persistencia,0.829620,0.000000,0.413172,0.000000,0.412964,0.000000,0.204064,0.000000,0.976258,0.000000,0.084254,0.000000,0.086126,0.000000,0.000969,0.000000e+00,0.002014,0.000000
8,variables_en_train,GAT + NPP,0.811484,0.002880,0.380468,0.001348,0.380677,0.001514,0.216243,0.002856,0.980056,0.000395,0.078176,0.001113,0.110340,0.003934,0.000866,7.096215e-07,0.001776,0.000002
9,variables_en_train,GAT 100% libre,0.810505,0.002835,0.381654,0.001597,0.378149,0.002550,0.217924,0.003052,0.979419,0.000376,0.078190,0.001431,0.111305,0.004162,0.000867,1.197939e-06,0.001776,0.000002


Resultados guardados en: /content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1/opcion_21_bins_sin_entreno/a45d672f65/20260907T233401148887Z


In [ ]:
# La discretización revela que acertar niveles de participación y acertar estados
# categóricos son objetivos diferentes. Persistencia domina el acierto exacto,
# especialmente en nodos nulos o muy estables; los GAT conservan ventajas en errores
# continuos y muestran señales de mejor recuperación fuera del estado dominante,
# aunque no dominan el balance entre clases.

### 21.b. Permanencias y cambios de bin — auditoría sin reentrenamiento

Ejecutar **solo la celda siguiente**, incluso en runtime vacío y CPU. Lee exclusivamente
la ejecución completa de bins `20260907T233401148887Z` mediante `CARPETA_BINS`.
No necesita ejecutar 20, 21 ni cargar dfyp/modelos. Para otro experimento, cambiar
esa ruta por su carpeta completa, sin mezclar archivos de ejecuciones distintas.

La unidad es **activo × instante**. Definimos cambio real como
`bin_real(t+h) != bin_real(t)`. La predicción de persistencia guardada proporciona
`bin_real(t)` con los mismos umbrales de train. No se desplazan filas y no se pierde
el primer instante. La etiqueta futura se usa solo para auditar resultados ya
guardados, nunca como entrada, selección de modelo ni ajuste de umbrales.

Salidas: métricas por nodo y semilla separadas en permanencias/cambios; resúmenes
macro por nodo y agrupados por observaciones; detección de cambios (recall,
precisión, F1, falsas alarmas y acierto de dirección); contrastes emparejados.
Cada tabla identifica modelo y cohorte (todos, variables y constantes en train).
Los resúmenes macro de eventos excluyen nodos sin casos de ese régimen; su número
queda explícito en `macro_por_semilla.csv`. No se agrupan macro-F1 de clases de
activos distintos como si los identificadores de bin fueran una clase común.

**Interpretación:** persistencia acierta 100% en permanencias y 0% en cambios por
definición, cuando esos casos existen. Eso no es evidencia experimental por sí solo.
Un GAT debe recuperar cambios sin generar demasiadas falsas alarmas. Detectar que
algo cambia no implica acertar el bin de destino: por eso se reportan ambas cosas.
La precisión sin alertas y métricas sin casos quedan NaN, no cero. La dirección
se mide entre los cambios reales; predecir permanencia cuenta como dirección fallida.
`dentro_un_bin` admite distancia ordinal <= 1, no un error económico fijo.

Se reutiliza el mismo test; esto es diagnóstico retrospectivo, no otra validación
independiente. La desviación estándar es entre semillas, no un intervalo temporal.
No usar el desglose para ajustar sobre test y presentar luego el mismo test como nuevo.

Los resultados se guardan en una subcarpeta nueva `21b_permanencias_y_transiciones`
dentro de la ejecución de bins. Se conservan los archivos anteriores y se guarda
un manifiesto con las huellas SHA256 y una marca final solo al completar la exportación.


In [ ]:
# 21.b. Ejecutar SOLO esta celda en CPU, incluso con runtime vacio.
CARPETA_BINS = '/content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1/opcion_21_bins_sin_entreno/a45d672f65/20260907T233401148887Z'

import hashlib
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False))


def tr_ratio(n, d):
    return float(n / d) if d else np.nan


def tr_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for part in iter(lambda: f.read(1024 * 1024), b''):
            h.update(part)
    return h.hexdigest()


def tr_csv(df, path):
    tmp = path.with_suffix('.csv.tmp')
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def tr_classes(y, p, k):
    if len(y) == 0:
        return dict(accuracy=np.nan, balanced_accuracy=np.nan, macro_f1=np.nan,
                    error_ordinal=np.nan, dentro_un_bin=np.nan)
    cm = np.bincount(y * k + p, minlength=k*k).reshape(k, k)
    actual, predicted, tp = cm.sum(1), cm.sum(0), cm.diagonal()
    recall = np.divide(tp, actual, out=np.zeros(k, float), where=actual > 0)
    f1 = np.divide(2*tp, actual+predicted, out=np.zeros(k, float), where=actual+predicted > 0)
    distance = np.abs(y-p)
    return dict(accuracy=float((y == p).mean()),
                balanced_accuracy=float(recall[actual > 0].mean()),
                macro_f1=float(f1[actual+predicted > 0].mean()),
                error_ordinal=float(distance.mean()), dentro_un_bin=float((distance <= 1).mean()))


def tr_events(current, target, pred):
    changed, announced = target != current, pred != current
    tp = int((changed & announced).sum())
    fp = int((~changed & announced).sum())
    fn = int((changed & ~announced).sum())
    tn = int((~changed & ~announced).sum())
    return dict(N=int(target.size), N_cambios=int(changed.sum()), N_permanencias=int((~changed).sum()),
                TP=tp, FP=fp, FN=fn, TN=tn,
                tasa_cambio_real=float(changed.mean()),
                tasa_cambio_predicho=float(announced.mean()),
                recall_cambio=tr_ratio(tp, tp+fn), precision_cambio=tr_ratio(tp, tp+fp),
                F1_cambio=tr_ratio(2*tp, 2*tp+fp+fn),
                tasa_falsas_alarmas=tr_ratio(fp, fp+tn),
                acierto_direccion_en_cambios=tr_ratio(
                    int(((np.sign(target-current) == np.sign(pred-current)) & changed).sum()), int(changed.sum())))


def tr_run(folder):
    root = Path(folder)
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        drive.mount('/content/drive')
    models = {'Persistencia': 'persistencia', 'GAT 100% libre': 'gat_libre',
              'GAT NPP solo aristas': 'gat_npp_aristas', 'GAT + NPP': 'gat_npp'}
    seeds = [11, 23, 42, 67, 101, 137, 211, 307, 401, 503]
    fixed_files = ['manifest_completo.json', 'umbrales_train_por_nodo.csv', 'etiquetas_reales.npz']
    paths = {s: root / f'predicciones_bins_seed_{s:04d}.npz' for s in seeds}
    for path in [*(root / f for f in fixed_files), *paths.values()]:
        if not path.is_file():
            raise FileNotFoundError(f'Falta {path}. Esta celda no entrena ni reconstruye predicciones.')
    source_manifest = json.loads((root / fixed_files[0]).read_text(encoding='utf-8'))
    thresholds = pd.read_csv(root / fixed_files[1], keep_default_na=False)
    with np.load(root / fixed_files[2], allow_pickle=False) as z:
        assets, t, target = z['assets'].astype(str), z['t'].copy(), z['test_labels'].copy()
    if target.shape != (len(t), len(assets)) or len(set(assets)) != len(assets):
        raise ValueError('Dimensiones o activos duplicados en etiquetas reales.')
    if not np.issubdtype(t.dtype, np.integer) or np.any(np.diff(t) <= 0):
        raise ValueError('Indices de origen invalidos o desordenados.')
    if set(thresholds.Nodo) != set(assets):
        raise ValueError('Universo de umbrales distinto de etiquetas.')
    counts, constants = [], []
    for asset in assets:
        th = thresholds[thresholds.Nodo == asset].sort_values('bin')
        if not np.array_equal(th.bin.to_numpy(), np.arange(len(th))):
            raise ValueError(f'{asset}: bins duplicados o no consecutivos.')
        flags = th.constante_train.astype(str).str.lower().unique()
        if len(flags) != 1 or flags[0] not in ('true', 'false'):
            raise ValueError(f'{asset}: indicador de constancia invalido.')
        constants.append(flags[0] == 'true')
        counts.append(len(th))
    counts, constants = np.asarray(counts), np.asarray(constants)
    def check_labels(x, label):
        if x.shape != target.shape or not np.issubdtype(x.dtype, np.integer):
            raise ValueError(f'{label}: dimensiones o etiquetas no enteras.')
        if np.any(x < 0) or np.any(x >= counts[None, :]):
            raise ValueError(f'{label}: etiqueta fuera de rango.')
        return x.astype(np.int64)
    target = check_labels(target, 'target')
    output = root / '21b_permanencias_y_transiciones' / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    output.mkdir(parents=True)
    provenance = dict(version='transitions_v1', training_performed=False,
                      source_folder=str(root), source_manifest=source_manifest,
                      event_definition='bin_real(t+h) != bin_real(t); bin_real(t) = persistencia guardada',
                      retrospective_conditioning=True, tuning_performed=False,
                      absent_denominators='NaN: no se convierten a cero',
                      uncertainty='SD entre semillas; no IC temporal ni muestras independientes',
                      assets=assets.tolist(), seeds=seeds,
                      files=[dict(path=str(root/f), sha256=tr_hash(root/f)) for f in fixed_files])
    by_node, pooled, detection, detection_nodes = [], [], [], []
    current_reference = None
    for seed, path in paths.items():
        print(f'21.b: semilla {seed}; leyendo etiquetas, sin entrenamiento.', flush=True)
        with np.load(path, allow_pickle=False) as z:
            if int(z['seed']) != seed or not np.array_equal(z['t'], t) or not np.array_equal(z['assets'].astype(str), assets):
                raise ValueError(f'{seed}: semilla, fechas o activos incompatibles.')
            if not np.array_equal(z['target_bins'], target):
                raise ValueError(f'{seed}: target incompatible.')
            predictions = {m: check_labels(z[k], f'{seed}/{m}') for m, k in models.items()}
        current = predictions['Persistencia']
        if current_reference is None:
            current_reference = current.copy()
        elif not np.array_equal(current, current_reference):
            raise ValueError('Persistencia cambia entre semillas; archivos mezclados.')
        change = target != current
        local_nodes, local_pooled, local_detection, local_dn = [], [], [], []
        for model, pred in predictions.items():
            meta = dict(Modelo=model, seed=seed, npp_aristas=model in ('GAT NPP solo aristas', 'GAT + NPP'),
                        npp_loss=model == 'GAT + NPP', modelo_entrenado=False)
            for j, asset in enumerate(assets):
                node_meta = {**meta, 'Nodo': asset, 'constante_train': bool(constants[j])}
                local_dn.append({**node_meta, **tr_events(current[:, j], target[:, j], pred[:, j])})
                for regime, mask in [('permanencia', ~change[:, j]), ('cambio', change[:, j])]:
                    local_nodes.append({**node_meta, 'regimen': regime, 'N': int(mask.sum()),
                                        **tr_classes(target[mask, j], pred[mask, j], counts[j])})
            for cohort, cols in [('todos', np.ones(len(assets), bool)), ('variables_en_train', ~constants), ('constantes_en_train', constants)]:
                if not cols.any():
                    continue
                y, p, c = target[:, cols], pred[:, cols], current[:, cols]
                local_detection.append({**meta, 'cohorte': cohort, **tr_events(c, y, p)})
                for regime, mask in [('permanencia', y == c), ('cambio', y != c)]:
                    distance = np.abs(y[mask]-p[mask])
                    local_pooled.append({**meta, 'cohorte': cohort, 'regimen': regime, 'N': int(mask.sum()),
                                         'accuracy': float((distance == 0).mean()) if distance.size else np.nan,
                                         'error_ordinal': float(distance.mean()) if distance.size else np.nan,
                                         'dentro_un_bin': float((distance <= 1).mean()) if distance.size else np.nan})
        tr_csv(pd.DataFrame(local_nodes), output / f'nodos_seed_{seed:04d}.csv')
        tr_csv(pd.DataFrame(local_detection), output / f'deteccion_seed_{seed:04d}.csv')
        by_node.extend(local_nodes); pooled.extend(local_pooled)
        detection.extend(local_detection); detection_nodes.extend(local_dn)
        provenance['files'].append(dict(path=str(path), sha256=tr_hash(path)))
    nodes, pooled, events = pd.DataFrame(by_node), pd.DataFrame(pooled), pd.DataFrame(detection)
    macro = []
    metrics = ['accuracy', 'balanced_accuracy', 'macro_f1', 'error_ordinal', 'dentro_un_bin']
    for cohort, df in [('todos', nodes), ('variables_en_train', nodes[~nodes.constante_train]), ('constantes_en_train', nodes[nodes.constante_train])]:
        if df.empty:
            continue
        group = df.groupby(['Modelo', 'seed', 'regimen'], as_index=False)
        mean = group[metrics].mean()
        sizes = group.agg(N_nodos=('Nodo', 'nunique'), N_nodos_con_eventos=('N', lambda x: int((x > 0).sum())), N=('N', 'sum'))
        mean = mean.merge(sizes, on=['Modelo', 'seed', 'regimen'], validate='one_to_one')
        mean['cohorte'] = cohort
        macro.append(mean)
    macro = pd.concat(macro, ignore_index=True)
    def summary(df, keys, columns):
        grouped = df.groupby(keys, dropna=False)[columns].agg(['mean', 'std', 'count'])
        grouped.columns = ['_'.join(x) for x in grouped.columns]
        return grouped.reset_index()
    event_metrics = ['tasa_cambio_real', 'tasa_cambio_predicho', 'recall_cambio', 'precision_cambio', 'F1_cambio', 'tasa_falsas_alarmas', 'acierto_direccion_en_cambios']
    sm = summary(macro, ['cohorte', 'regimen', 'Modelo'], metrics)
    sp = summary(pooled, ['cohorte', 'regimen', 'Modelo'], ['accuracy', 'error_ordinal', 'dentro_un_bin'])
    se = summary(events, ['cohorte', 'Modelo'], event_metrics)
    for df, name in [(nodes, 'metricas_por_nodo'), (macro, 'macro_por_semilla'), (pooled, 'pooled_por_semilla'),
                     (events, 'deteccion_por_semilla'), (pd.DataFrame(detection_nodes), 'deteccion_por_nodo'),
                     (sm, 'resumen_macro'), (sp, 'resumen_pooled'), (se, 'resumen_deteccion')]:
        tr_csv(df, output / (name + '.csv'))
    deltas = []
    for reference in ('Persistencia', 'GAT 100% libre', 'GAT NPP solo aristas'):
        baseline = macro[macro.Modelo == reference]
        joined = macro[macro.Modelo != reference].merge(baseline, on=['seed', 'cohorte', 'regimen'], suffixes=('', '_ref'), validate='many_to_one')
        for _, row in joined.iterrows():
            for metric in metrics:
                delta = row[metric] - row[metric+'_ref']
                deltas.append(dict(Modelo=row.Modelo, Referencia=reference, seed=int(row.seed),
                                   cohorte=row.cohorte, regimen=row.regimen, metrica=metric, delta=delta,
                                   mejora_favorable=-delta if metric == 'error_ordinal' else delta))
    tr_csv(pd.DataFrame(deltas), output / 'deltas_macro_emparejados.csv')
    # Completion marker written only after all exports succeed.
    tmp = output / 'manifest_completo.json.tmp'
    tmp.write_text(json.dumps(provenance, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(tmp, output / 'manifest_completo.json')
    print('MACRO por nodo: cada nodo con eventos pesa igual; NaN indica ausencia de casos.')
    display(sm[['cohorte', 'regimen', 'Modelo', 'accuracy_mean', 'balanced_accuracy_mean', 'macro_f1_mean', 'error_ordinal_mean']])
    print('DETECCION agrupada por observaciones: falsas alarmas y cambios detectados.')
    display(se[['cohorte', 'Modelo', *[m+'_mean' for m in event_metrics]]])
    print('Persistencia: 100% en permanencia y 0% en cambio por definicion, si existen esos casos.')
    print('Resultados guardados en:', output)
    return output


RESULTADOS_TRANSICIONES_21 = tr_run(CARPETA_BINS)


Mounted at /content/drive
21.b: semilla 11; leyendo etiquetas, sin entrenamiento.
21.b: semilla 23; leyendo etiquetas, sin entrenamiento.
21.b: semilla 42; leyendo etiquetas, sin entrenamiento.
21.b: semilla 67; leyendo etiquetas, sin entrenamiento.
21.b: semilla 101; leyendo etiquetas, sin entrenamiento.
21.b: semilla 137; leyendo etiquetas, sin entrenamiento.
21.b: semilla 211; leyendo etiquetas, sin entrenamiento.
21.b: semilla 307; leyendo etiquetas, sin entrenamiento.
21.b: semilla 401; leyendo etiquetas, sin entrenamiento.
21.b: semilla 503; leyendo etiquetas, sin entrenamiento.
MACRO por nodo: cada nodo con eventos pesa igual; NaN indica ausencia de casos.


,cohorte,regimen,Modelo,accuracy_mean,balanced_accuracy_mean,macro_f1_mean,error_ordinal_mean
0,constantes_en_train,cambio,GAT + NPP,0.582095,0.566667,0.428854,0.417905
1,constantes_en_train,cambio,GAT 100% libre,0.582095,0.566667,0.428854,0.417905
2,constantes_en_train,cambio,GAT NPP solo aristas,0.582095,0.566667,0.428854,0.417905
3,constantes_en_train,cambio,Persistencia,0.000000,0.000000,0.000000,1.000000
4,constantes_en_train,permanencia,GAT + NPP,0.374373,0.375000,0.246621,0.625627
5,constantes_en_train,permanencia,GAT 100% libre,0.374373,0.375000,0.246621,0.625627
6,constantes_en_train,permanencia,GAT NPP solo aristas,0.374373,0.375000,0.246621,0.625627
7,constantes_en_train,permanencia,Persistencia,1.000000,1.000000,1.000000,0.000000
8,todos,cambio,GAT + NPP,0.266665,0.170515,0.151178,0.843618
9,todos,cambio,GAT 100% libre,0.257418,0.167503,0.149213,0.854315


DETECCION agrupada por observaciones: falsas alarmas y cambios detectados.


,cohorte,Modelo,tasa_cambio_real_mean,tasa_cambio_predicho_mean,recall_cambio_mean,precision_cambio_mean,F1_cambio_mean,tasa_falsas_alarmas_mean,acierto_direccion_en_cambios_mean
0,constantes_en_train,GAT + NPP,0.015542,0.627542,0.506649,0.012548,0.024490,0.629451,0.506649
1,constantes_en_train,GAT 100% libre,0.015542,0.627542,0.506649,0.012548,0.024490,0.629451,0.506649
2,constantes_en_train,GAT NPP solo aristas,0.015542,0.627542,0.506649,0.012548,0.024490,0.629451,0.506649
3,constantes_en_train,Persistencia,0.015542,0.000000,0.000000,NaN,0.000000,0.000000,0.000000
4,todos,GAT + NPP,0.170380,0.137604,0.328503,0.406858,0.363481,0.098399,0.309643
5,todos,GAT 100% libre,0.170380,0.136091,0.321646,0.402775,0.357654,0.097984,0.301421
6,todos,GAT NPP solo aristas,0.170380,0.133860,0.333994,0.425140,0.374084,0.092758,0.314724
7,todos,Persistencia,0.170380,0.000000,0.000000,NaN,0.000000,0.000000,0.000000
8,variables_en_train,GAT + NPP,0.178693,0.111299,0.327671,0.526326,0.403848,0.064222,0.308723
9,variables_en_train,GAT 100% libre,0.178693,0.109704,0.320782,0.522692,0.397546,0.063780,0.300463


Persistencia: 100% en permanencia y 0% en cambio por definicion, si existen esos casos.
Resultados guardados en: /content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1/opcion_21_bins_sin_entreno/a45d672f65/20260907T233401148887Z/21b_permanencias_y_transiciones/20260908T001252642740Z


In [ ]:
# los GAT sí anticipan parte de los cambios, pero todavía dejan pasar muchos y
# generan falsas alarmas. El NPP solo en aristas consigue el mejor equilibrio de los tres para esta tarea.

# | Modelo | Cambios detectados — recall | Precisión de las alertas | F1 de cambio | Falsas alarmas¹ |
# |---|---:|---:|---:|---:|
# | GAT libre | 32,08% | 52,27% | 39,75% | 6,38% |
# | NPP solo aristas | **33,32%** | **55,47%** | **41,63%** | **5,82%** |
# | GAT + NPP | 32,77% | 52,63% | 40,38% | 6,42% |

# De cada 100 cambios reales, detecta aproximadamente 33.
# De cada 100 alertas, aproximadamente 55 corresponden a un cambio real.
# Entre los casos sin cambio, genera unas 6 falsas alarmas cada 100.

## 22. Pérdida, estabilidad y ganancia de participación — experimento independiente sin entrenamiento

Ejecutar **solo la siguiente celda en CPU**, incluso con runtime vacío. No requiere
ejecutar 20, 21, 21.b ni los otros nuevos experimentos. Lee config/activos del resultado
base, las predicciones continuas de las diez semillas y únicamente dfyp para recuperar
train. No carga pesos ni construye features ni entrena modelos. Puede pedir autorización
para montar Drive. El acceso y lectura de archivos sí llevan tiempo.

Mide **redistribución**, no cruces de fronteras de nivel. Por activo:
`delta = p(t+h)-p(t)` y `tau = max(mediana_train(abs(delta)), 1e-8)`.
Clases: pérdida si delta < -tau, estabilidad si -tau <= delta <= tau, ganancia si delta > tau.
La predicción se etiqueta con `p_pred(t+h)-p(t)`. No se ajusta un clasificador.
El piso 1e-8 es una convención numérica, NO una magnitud económica validada. No garantiza
clases equilibradas y se documenta por nodo. Se separan nodos constantes/variables en train.

**En qué fijarse:** macro-F1 de las tres clases, recall/precisión de pérdida y ganancia,
recall de cambio relevante y falsas alarmas. `direccion_correcta_en_cambios` se exporta
en eventos_por_semilla. Persistencia siempre predice estabilidad; su buen accuracy
no demuestra anticipación. Comparar también contra el bin modal de train.


**Control metodológico:** train y validación conservan sus cortes y purgas originales;
solo train define estadísticas/umbrales y controles. Test se reutiliza de forma
exploratoria: estos diseños nacen después de mirarlo y no son confirmación independiente.
No shuffle, no modificación de predicciones, no selección por rendimiento en test.
La dispersión entre semillas no es incertidumbre temporal.

CSV por modelo/nodo/semilla, matrices de confusión, ocupaciones, etiquetas NPZ, umbrales,
comparaciones emparejadas y manifiesto se guardan en
`GAT_crypto_extended_audits_v1/21_delta_sin_entreno/`, en una carpeta nueva por ejecución.
El benchmark modal y persistencia se repiten solo para emparejar: no son diez modelos
independientes. La marca manifest_completo aparece solo al terminar todos los guardados.


In [1]:
# Ejecutar solo esta celda. No necesita GPU ni reentrena.
CARPETA_PROYECTO = '/content/drive/MyDrive/Neural/NPP/Cripto'
CONFIG_EXPERIMENTO = {'quantile_abs_delta': 0.5, 'min_change': 1e-08}

import hashlib

import json

import os

from pathlib import Path

import numpy as np

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)

POSTHOC_VERSION = 'frozen_predictions_v1_target_precision_fix'

POSTHOC_SEEDS = [11, 23, 42, 67, 101, 137, 211, 307, 401, 503]

POSTHOC_MODELS = {
    'Persistencia': 'persistencia',
    'GAT 100% libre': 'gat_libre',
    'GAT NPP solo aristas': 'gat_npp_aristas',
    'GAT + NPP': 'gat_npp',
}

def ph_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def ph_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def ph_json(value, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)

def ph_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(tmp, path)

def ph_mount(root):
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        try:
            drive.mount('/content/drive')
            list(Path(root).iterdir())
        except OSError:
            drive.mount('/content/drive', force_remount=True)
    if not Path(root).is_dir():
        raise FileNotFoundError(f'No existe la carpeta del proyecto: {root}')

def ph_prob(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError('Predicciones/target deben ser matrices finitas y no negativas.')
    mass = values.sum(axis=1, keepdims=True)
    if np.any(mass <= 0) or not np.allclose(mass, 1., atol=1e-5, rtol=0):
        raise ValueError('El artefacto no contiene un simplex cerrado.')
    return values / mass

def ph_context(root):
    root = Path(root)
    ph_mount(root)
    base = root / 'GAT_crypto_results'
    config_path = base / 'config.json'
    reference_path = base / 'predicciones_test.npz'
    cfg = json.loads(config_path.read_text(encoding='utf-8'))
    with np.load(reference_path, allow_pickle=False) as saved:
        assets = saved['assets'].astype(str)
        t = saved['t'].copy()
        target_raw = saved['target'].copy()
        # Older exports stored timestamps as object arrays. Do not unpickle them.
        try:
            saved_dates = pd.to_datetime(saved['timestamps'].astype(str), utc=True)
        except ValueError as error:
            if 'Object arrays cannot be loaded' not in str(error):
                raise
            saved_dates = None
    if t.ndim != 1 or not np.issubdtype(t.dtype, np.integer) or len(t) < 2:
        raise ValueError('Indices temporales de referencia invalidos.')
    if np.any(np.diff(t) <= 0) or np.any(t < 0) or len(set(assets)) != len(assets):
        raise ValueError('Fechas duplicadas/desordenadas o activos duplicados.')
    target = ph_prob(target_raw)
    if target.shape != (len(t), len(assets)):
        raise ValueError('El orden/dimension de activos no coincide con el target.')
    step = pd.Timedelta(cfg['frequency'])
    expected_dates = pd.Timestamp(cfg['start_timestamp']) + (t + int(cfg['horizon'])) * step
    dates = pd.DatetimeIndex(expected_dates)
    if saved_dates is not None and not np.array_equal(saved_dates.asi8, dates.asi8):
        raise ValueError('Config y timestamps guardados pertenecen a experimentos distintos.')
    ablation = root / 'GAT_crypto_ablation_10_seeds'
    paths = {s: ablation / f'seed_{s:04d}' / 'predicciones_test.npz' for s in POSTHOC_SEEDS}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('Faltan predicciones. No se entrenara nada: ' + ', '.join(missing))
    manifest = {
        'version': POSTHOC_VERSION, 'training_performed': False,
        'prediction_source': 'predicciones_test.npz de ablacion de diez semillas',
        'test_reused': True, 'new_external_validation': False,
        'reference_path': str(reference_path), 'reference_sha256': ph_hash(reference_path),
        'config_path': str(config_path), 'config_sha256': ph_hash(config_path),
        'config': cfg, 'seeds': POSTHOC_SEEDS, 'assets': assets.tolist(),
        'n_test': len(t), 'target_start': str(dates[0]), 'target_end': str(dates[-1]),
        'timestamps_source': 'config + indices t; legacy object array not unpickled' if saved_dates is None else 'saved timestamps checked against config and t',
        'sources': [],
    }
    return dict(root=root, cfg=cfg, assets=assets, t=t, target=target,
                target_raw=target_raw, dates=dates, paths=paths, manifest=manifest)

def ph_load_seed(ctx, seed):
    path = ctx['paths'][seed]
    digest = ph_hash(path)
    with np.load(path, allow_pickle=False) as saved:
        if not np.array_equal(saved['t'], ctx['t']):
            raise ValueError(f'Semilla {seed}: distinto orden temporal.')
        seed_target_raw = saved['target'].copy()
        seed_target = ph_prob(seed_target_raw)
        reference = ctx.get('ablation_target_reference', ctx['target'])
        # El export base convierte a float32; el dataset puede producir float64
        # al dividir por una suma float64. Comparar el simplex, no sus bits.
        # Tolerancia relativa de cuatro eps float32, sin tolerancia absoluta:
        # un cero no se acepta como una cuota positiva, por diminuta que sea.
        tolerance = 4 * np.finfo(np.float32).eps
        if seed_target.shape != reference.shape:
            raise ValueError(f'Semilla {seed}: dimensiones del target incompatibles.')
        delta = np.abs(seed_target - reference)
        scale = np.maximum(np.abs(seed_target), np.abs(reference))
        same_support = np.array_equal(seed_target == 0, reference == 0)
        close = same_support and np.all(delta <= tolerance * scale)
        max_abs = float(delta.max())
        max_rel = float(np.max(np.divide(delta, scale, out=np.zeros_like(delta), where=scale > 0)))
        if not close:
            raise ValueError(
                f'Semilla {seed}: discrepancia real de target; no se mezclan archivos. '
                f'Max abs={max_abs:.6g}, max rel={max_rel:.6g}, '
                f'mismo soporte={same_support}, tolerancia rel={tolerance:.6g}. '
                'Revisar export base y ablacion; no hace falta reentrenar para diagnosticar.'
            )
        ctx['manifest'].setdefault('target_alignment', []).append({
            'seed': int(seed), 'comparison': 'normalized simplex, identical zero support',
            'relative_tolerance': float(tolerance), 'absolute_tolerance': 0.,
            'max_absolute_difference': max_abs, 'max_relative_difference': max_rel,
            'reference': 'first ablation seed' if 'ablation_target_reference' in ctx else 'base export',
        })
        if 'ablation_target_reference' not in ctx:
            # Evaluar contra los valores reales del mismo paquete de predicciones.
            ctx['ablation_target_reference'] = seed_target.copy()
            ctx['target'] = seed_target.copy()
            ctx['manifest']['metric_target_source'] = str(path)
        if max_abs > 0:
            print(f'Semilla {seed}: redondeo compatible verificado; diferencia maxima {max_abs:.3g}.', flush=True)
        pred = {name: ph_prob(saved[key]) for name, key in POSTHOC_MODELS.items()}
    for name, values in pred.items():
        if values.shape != ctx['target'].shape:
            raise ValueError(f'{seed}/{name}: dimensiones incompatibles.')
    if 'persistence_reference' not in ctx:
        ctx['persistence_reference'] = pred['Persistencia'].copy()
    elif not np.array_equal(pred['Persistencia'], ctx['persistence_reference']):
        raise ValueError('Persistencia cambio entre semillas; no se mezclan experimentos.')
    ctx['manifest']['sources'].append({'seed': seed, 'path': str(path), 'sha256': digest})
    return pred

def ph_check_source(ctx, left, right, label):
    left, right = ph_prob(left), ph_prob(right)
    if left.shape != right.shape:
        raise ValueError(f'{label}: dimensiones incompatibles.')
    delta = np.abs(left - right)
    scale = np.maximum(np.abs(left), np.abs(right))
    tolerance = 4 * np.finfo(np.float32).eps
    same_support = np.array_equal(left == 0, right == 0)
    if not same_support or not np.all(delta <= tolerance * scale):
        raise ValueError(f'{label}: discrepancia real, max abs={delta.max():.6g}; no se mezclan datos.')
    ctx['manifest'].setdefault('source_alignment', []).append({
        'comparison': label, 'relative_tolerance': float(tolerance),
        'absolute_tolerance': 0., 'identical_zero_support': True,
        'max_absolute_difference': float(delta.max()),
    })

def ph_train_statistics(ctx, warmup=12128):
    cfg = ctx['cfg']
    path = Path(cfg['dfyp_path'])
    print('Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.', flush=True)
    digest = ph_hash(path)
    frame = pd.read_parquet(path) if path.suffix.lower() in {'.parquet', '.pq'} else pd.read_csv(path)
    if 'YS' in frame.columns and 'YS_p' not in frame.columns:
        frame = frame.rename(columns=lambda c: c[:-2] + '_p' if c.endswith('_y') else 'YS_p' if c == 'YS' else c)
    cols = [c for c in frame.columns if c.endswith('_p') and c != 'YS_p']
    if cols != [a + '_p' for a in ctx['assets']]:
        raise ValueError('dfyp no conserva el universo/orden de activos de las predicciones.')
    horizon, stride, purge = int(cfg['horizon']), int(cfg['sample_stride']), int(cfg['purge'])
    test_start = len(frame) - horizon - int(cfg['test_size'])
    validation_end = test_start - purge
    validation_start = validation_end - int(cfg['validation_size'])
    train_end = validation_start - purge
    all_t = np.arange(warmup, len(frame) - horizon, stride)
    train_t = all_t[all_t < train_end]
    val_t = all_t[(all_t >= validation_start) & (all_t < validation_end)]
    test_t = all_t[all_t >= test_start]
    if not np.array_equal(test_t, ctx['t']):
        raise ValueError('dfyp/config no reproduce el test archivado; no se recalculan cortes silenciosamente.')
    if len(train_t) < 2 or len(val_t) == 0 or train_t[-1] + horizon >= val_t[0] or val_t[-1] + horizon >= test_t[0]:
        raise ValueError('Split vacio o purga incompatible.')
    # Identical normalization to the original extraction; only train targets fit thresholds.
    def normalize_rows(indices):
        values = frame.iloc[indices][cols].to_numpy(dtype=np.float32, copy=True)
        if np.any(values < -1e-8):
            raise ValueError('dfyp contiene masas negativas.')
        values = np.nan_to_num(values, nan=0., posinf=0., neginf=0.)
        values = np.clip(values, 0., None)
        total = values.sum(axis=1, keepdims=True, dtype=np.float64)
        if np.any(total <= 0):
            raise ValueError('dfyp contiene filas sin masa.')
        return (values / total).astype(np.float32)
    ph_check_source(ctx, normalize_rows(test_t + horizon), ctx['target'], 'dfyp frente a target archivado')
    # Validate persistence against raw inputs before fitting any bins.
    with np.load(ctx['paths'][POSTHOC_SEEDS[0]], allow_pickle=False) as saved:
        ph_check_source(ctx, normalize_rows(test_t), saved['persistencia'], 'dfyp frente a persistencia archivada')
    train = ph_prob(normalize_rows(train_t + horizon))
    mu = train.mean(axis=0)
    sigma = train.std(axis=0, ddof=1)
    ctx['manifest']['train_bins'] = {
        'dfyp_path': str(path), 'dfyp_sha256': digest, 'warmup': warmup,
        'fit_rows': 'targets P[train_t + horizon]', 'ddof': 1, 'n_train': len(train_t),
        'train_origin_first': int(train_t[0]), 'train_origin_last': int(train_t[-1]),
        'last_train_target': int(train_t[-1] + horizon), 'validation_first_origin': int(val_t[0]),
        'test_first_origin': int(test_t[0]), 'fit_uses_validation_or_test': False,
        'legacy_provenance': 'Config y assets de resultados base, alineados con target y persistencia de ablacion, tolerancia relativa float32 documentada. Warmup original fijo 12128; archivos antiguos sin hash historico de train.'}
    return train, mu, sigma, train_t + horizon

def ph_class_metrics(actual, predicted, n_classes):
    confusion = np.bincount(actual.astype(int) * n_classes + predicted.astype(int),
                            minlength=n_classes ** 2).reshape(n_classes, n_classes)
    support, guessed = confusion.sum(axis=1), confusion.sum(axis=0)
    tp = confusion.diagonal()
    recall = np.divide(tp, support, out=np.zeros(n_classes, float), where=support > 0)
    f1 = np.divide(2 * tp, support + guessed, out=np.zeros(n_classes, float), where=(support + guessed) > 0)
    delta = np.abs(actual.astype(int) - predicted.astype(int))
    result = dict(accuracy=float(np.mean(actual == predicted)),
                  balanced_accuracy=float(recall[support > 0].mean()),
                  macro_f1=float(f1[(support + guessed) > 0].mean()),
                  error_ordinal=float(delta.mean()), dentro_un_bin=float((delta <= 1).mean()),
                  tasa_subestimacion=float((predicted < actual).mean()),
                  tasa_sobreestimacion=float((predicted > actual).mean()),
                  n_clases=n_classes, clases_reales_presentes=int((support > 0).sum()))
    return result, confusion

# Helpers compartidos, embebidos completos en cada celda independiente.
def ex_div(n, d):
    return float(n / d) if d else np.nan


def ex_event(y, p):
    y, p = np.asarray(y, bool), np.asarray(p, bool)
    tp, fp = int((y & p).sum()), int((~y & p).sum())
    fn, tn = int((y & ~p).sum()), int((~y & ~p).sum())
    return dict(N=int(y.size), TP=tp, FP=fp, FN=fn, TN=tn,
                prevalencia=ex_div(tp+fn, y.size), tasa_alertas=ex_div(tp+fp, y.size),
                recall=ex_div(tp, tp+fn), precision=ex_div(tp, tp+fp),
                F1=ex_div(2*tp, 2*tp+fp+fn), falsas_alarmas=ex_div(fp, fp+tn))


def ex_current_train(ctx, rows):
    path = Path(ctx['cfg']['dfyp_path'])
    frame = pd.read_parquet(path) if path.suffix.lower() in ('.parquet', '.pq') else pd.read_csv(path)
    if 'YS' in frame.columns and 'YS_p' not in frame.columns:
        frame = frame.rename(columns=lambda c: c[:-2]+'_p' if c.endswith('_y') else 'YS_p' if c == 'YS' else c)
    values = frame.iloc[rows-int(ctx['cfg']['horizon'])][[a+'_p' for a in ctx['assets']]].to_numpy(dtype=np.float32)
    if np.any(values < -1e-8):
        raise ValueError('Masas negativas en train.')
    values = np.clip(np.nan_to_num(values, nan=0., posinf=0., neginf=0.), 0., None)
    total = values.sum(axis=1, keepdims=True, dtype=np.float64)
    if np.any(total <= 0):
        raise ValueError('Train sin masa.')
    values = (values/total).astype(np.float32)
    if ph_hash(path) != ctx['manifest']['train_bins']['dfyp_sha256']:
        raise ValueError('dfyp cambio durante la lectura; detener sin mezclar archivos.')
    return ph_prob(values)


def ex_delta_labels(values, tau):
    # Clase 0=baja, 1=estable (incluye ambos limites), 2=sube.
    return np.where(values < -tau, 0, np.where(values > tau, 2, 1)).astype(np.int64)


def ex_level_labels(values, cuts, activity):
    labels = np.zeros(values.shape, dtype=np.int64)
    for j, boundaries in enumerate(cuts):
        active = values[:, j] > activity
        labels[active, j] = 1 + np.searchsorted(boundaries, values[active, j], side='right')
    return labels


def ex_run(root, experiment, config):
    if experiment not in ('delta', 'cuantiles', 'actividad'):
        raise ValueError('Experimento desconocido.')
    ctx = ph_context(root)
    first = ph_load_seed(ctx, POSTHOC_SEEDS[0])
    current = first['Persistencia'].copy()
    del first
    ctx['manifest']['initial_alignment'] = ctx['manifest'].pop('target_alignment')
    ctx['manifest']['sources'].clear()
    train, mu, sigma, rows = ph_train_statistics(ctx)
    constants = sigma <= 1e-8
    target = ctx['target']
    n_nodes = len(ctx['assets'])
    settings = []
    specifications = []
    coverage = []
    if experiment == 'delta':
        q, floor = float(config['quantile_abs_delta']), float(config['min_change'])
        if not 0 < q < 1 or floor <= 0:
            raise ValueError('Cuantil o piso de cambio invalido.')
        previous = ex_current_train(ctx, rows)
        delta = train-previous
        tau = np.maximum(np.quantile(np.abs(delta), q, axis=0), floor)
        train_y = ex_delta_labels(delta, tau)
        actual = ex_delta_labels(target-current, tau)
        settings.append(('mediana_cambio_absoluto', train_y, actual, np.full(n_nodes, 3), np.ones(n_nodes, bool), tau))
        for j, a in enumerate(ctx['assets']):
            specifications.append(dict(Nodo=a, tau=float(tau[j]), cuantil=q, piso=floor,
                                       constante_train=bool(constants[j]), n_train=len(train)))
    elif experiment == 'cuantiles':
        activity = float(config['activity_floor'])
        qs = np.asarray(config['quantiles'], float)
        minimum = int(config['min_positive_train'])
        if activity <= 0 or minimum < 2 or qs.ndim != 1 or not len(qs) or np.any(np.diff(qs) <= 0) or np.any((qs <= 0) | (qs >= 1)):
            raise ValueError('Configuracion de cuantiles invalida.')
        cuts, eligible, counts = [], [], []
        for j, a in enumerate(ctx['assets']):
            positive = train[train[:, j] > activity, j]
            # No confundir jitter de cierre float32 con variacion positiva real.
            ok = (len(positive) >= minimum and
                  np.ptp(positive) > 4*np.finfo(np.float32).eps*max(float(positive.max()), activity))
            boundaries = np.unique(np.quantile(positive, qs)) if ok else np.array([])
            # Cortes extremos no crean categorias positivas vacias en train.
            if ok:
                boundaries = boundaries[(boundaries > positive.min()) & (boundaries < positive.max())]
            cuts.append(boundaries)
            eligible.append(ok)
            counts.append(len(boundaries)+2)
            coverage.append(dict(Nodo=a, elegible=ok, n_positivos_train=len(positive),
                                 n_bins_positivos=len(boundaries)+1 if ok else 0,
                                 motivo='ok' if ok else 'historia positiva insuficiente o sin variacion'))
            specifications.append(dict(Nodo=a, umbral_actividad=activity, cortes_positivos=json.dumps(boundaries.tolist()),
                                       elegible=ok, min_positivos_train=minimum))
        eligible = np.asarray(eligible, bool)
        if not eligible.any():
            raise ValueError('Ningun activo tiene historia positiva suficiente; no se inventan cuantiles.')
        settings.append(('actividad_y_cuartiles_positivos', ex_level_labels(train, cuts, activity),
                         ex_level_labels(target, cuts, activity), np.asarray(counts), eligible, (cuts, activity)))
    else:
        levels = np.asarray(config['activity_floors'], float)
        if levels.ndim != 1 or not len(levels) or np.any((levels <= 0) | (levels >= 1)) or np.any(np.diff(levels) <= 0):
            raise ValueError('Umbrales de actividad invalidos.')
        for floor in levels:
            settings.append((f'umbral_{floor:g}', (train > floor).astype(np.int64),
                             (target > floor).astype(np.int64), np.full(n_nodes, 2), np.ones(n_nodes, bool), float(floor)))
            for j, a in enumerate(ctx['assets']):
                specifications.append(dict(Nodo=a, umbral_actividad=float(floor),
                                           tasa_actividad_train=float((train[:, j] > floor).mean()), constante_train=bool(constants[j])))
    tag = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:10]
    out = ctx['root'] / 'GAT_crypto_extended_audits_v1' / ('21_'+experiment+'_sin_entreno') / tag / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    out.mkdir(parents=True)
    ph_csv(pd.DataFrame(specifications), out/'umbrales_train.csv')
    if coverage:
        ph_csv(pd.DataFrame(coverage), out/'cobertura_train.csv')
    occupation = []
    modes = {}
    for name, train_y, actual, counts, eligible, state in settings:
        modes[name] = np.array([np.bincount(train_y[:, j], minlength=int(counts[j])).argmax() for j in range(n_nodes)])
        for j, asset in enumerate(ctx['assets']):
            if not eligible[j]:
                continue
            for label in range(int(counts[j])):
                occupation.append(dict(escenario=name, Nodo=asset, clase=label,
                                       n_train=int((train_y[:, j] == label).sum()), n_test=int((actual[:, j] == label).sum())))
        ph_npz(out/f'etiquetas_reales_{name}.npz', assets=ctx['assets'], t=ctx['t'], elegible=eligible,
               train_target_rows=rows, train_labels=train_y, test_labels=actual)
    ph_csv(pd.DataFrame(occupation), out/'ocupacion_train_test.csv')
    del train
    all_nodes, all_events = [], []
    for seed in POSTHOC_SEEDS:
        print(f'{experiment}: semilla {seed}, leyendo predicciones; cero entrenamientos.', flush=True)
        predictions = ph_load_seed(ctx, seed)
        node_rows, event_rows, confusion_rows = [], [], []
        for name, train_y, actual, counts, eligible, state in settings:
            encoded = {}
            for model, values in [*predictions.items(), ('Bin modal de train', None)]:
                if values is None:
                    pred = np.broadcast_to(modes[name], actual.shape).copy()
                elif experiment == 'delta':
                    pred = ex_delta_labels(values-current, state)
                elif experiment == 'cuantiles':
                    pred = ex_level_labels(values, *state)
                else:
                    pred = (values > state).astype(np.int64)
                encoded[POSTHOC_MODELS.get(model, 'modal_train')] = pred
                for j, asset in enumerate(ctx['assets']):
                    if not eligible[j]:
                        continue
                    y, p = actual[:, j], pred[:, j]
                    metrics, cm = ph_class_metrics(y, p, int(counts[j]))
                    meta = dict(Modelo=model, seed=seed, escenario=name, Nodo=asset,
                                constante_train=bool(constants[j]), modelo_entrenado=False,
                                npp_aristas=model in ('GAT NPP solo aristas', 'GAT + NPP'), npp_loss=model == 'GAT + NPP')
                    node_rows.append({**meta, **metrics, 'N_test': len(y)})
                    for a, b in zip(*np.nonzero(cm)):
                        confusion_rows.append({**meta, 'clase_real': int(a), 'clase_predicha': int(b), 'n': int(cm[a, b])})
                for cohort, cols in [('todos_elegibles', eligible), ('variables_en_train', eligible & ~constants), ('constantes_en_train', eligible & constants)]:
                    if not cols.any():
                        continue
                    y, p = actual[:, cols], pred[:, cols]
                    meta = dict(Modelo=model, seed=seed, escenario=name, cohorte=cohort, modelo_entrenado=False)
                    if experiment == 'delta':
                        event_rows.append({**meta, 'evento': 'cambio_relevante', **ex_event(y != 1, p != 1),
                                           'direccion_correcta_en_cambios': ex_div(int(((y == p) & (y != 1)).sum()), int((y != 1).sum()))})
                        for label, event in [(0, 'perdida'), (2, 'ganancia')]:
                            event_rows.append({**meta, 'evento': event, **ex_event(y == label, p == label)})
                    elif experiment == 'actividad':
                        now = current[:, cols] > state
                        event_rows.append({**meta, 'evento': 'estado_activo', **ex_event(y == 1, p == 1)})
                        event_rows.append({**meta, 'evento': 'activacion_desde_inactivo', **ex_event((y == 1)[~now], (p == 1)[~now])})
                        event_rows.append({**meta, 'evento': 'desactivacion_desde_activo', **ex_event((y == 0)[now], (p == 0)[now])})
            ph_npz(out/f'predicciones_{name}_seed_{seed:04d}.npz', assets=ctx['assets'], seed=np.array(seed), t=ctx['t'], elegible=eligible, target_labels=actual, **encoded)
        ph_csv(pd.DataFrame(node_rows), out/f'metricas_nodos_seed_{seed:04d}.csv')
        ph_csv(pd.DataFrame(confusion_rows), out/f'confusion_seed_{seed:04d}.csv')
        if event_rows:
            ph_csv(pd.DataFrame(event_rows), out/f'eventos_seed_{seed:04d}.csv')
        all_nodes.extend(node_rows); all_events.extend(event_rows)
    nodes = pd.DataFrame(all_nodes)
    metrics = ['accuracy', 'balanced_accuracy', 'macro_f1', 'error_ordinal', 'dentro_un_bin']
    macros = []
    for cohort, subset in [('todos_elegibles', nodes), ('variables_en_train', nodes[~nodes.constante_train]), ('constantes_en_train', nodes[nodes.constante_train])]:
        if subset.empty:
            continue
        grouped = subset.groupby(['escenario', 'Modelo', 'seed'], as_index=False)
        means = grouped[metrics].mean()
        means = means.merge(grouped.agg(N_nodos=('Nodo', 'nunique')), on=['escenario', 'Modelo', 'seed'], validate='one_to_one')
        means['cohorte'] = cohort
        macros.append(means)
    macro = pd.concat(macros, ignore_index=True)
    def summarize(df, keys, cols):
        result = df.groupby(keys)[cols].agg(['mean', 'std', 'count'])
        result.columns = ['_'.join(c) for c in result.columns]
        return result.reset_index()
    summary = summarize(macro, ['escenario', 'cohorte', 'Modelo'], metrics)
    ph_csv(nodes, out/'metricas_por_nodo.csv'); ph_csv(macro, out/'macro_por_semilla.csv')
    ph_csv(summary, out/'resumen_modelos.csv')
    paired = []
    for ref in ['Persistencia', 'GAT 100% libre', 'GAT NPP solo aristas', 'Bin modal de train']:
        merged = macro[macro.Modelo != ref].merge(macro[macro.Modelo == ref], on=['escenario', 'cohorte', 'seed'], suffixes=('', '_ref'), validate='many_to_one')
        for _, row in merged.iterrows():
            for m in metrics:
                d = row[m]-row[m+'_ref']
                paired.append(dict(Modelo=row.Modelo, Referencia=ref, seed=int(row.seed), escenario=row.escenario, cohorte=row.cohorte,
                                   metrica=m, delta=float(d), mejora_favorable=float(-d if m == 'error_ordinal' else d)))
    ph_csv(pd.DataFrame(paired), out/'deltas_emparejados.csv')
    display(summary[['escenario', 'cohorte', 'Modelo', 'accuracy_mean', 'balanced_accuracy_mean', 'macro_f1_mean']])
    if all_events:
        events = pd.DataFrame(all_events)
        event_summary = summarize(events, ['escenario', 'cohorte', 'evento', 'Modelo'], ['prevalencia', 'tasa_alertas', 'recall', 'precision', 'F1', 'falsas_alarmas'])
        ph_csv(events, out/'eventos_por_semilla.csv'); ph_csv(event_summary, out/'resumen_eventos.csv')
        display(event_summary[['escenario', 'cohorte', 'evento', 'Modelo', 'recall_mean', 'precision_mean', 'F1_mean', 'falsas_alarmas_mean']])
    ctx['manifest'].update(experiment=experiment, configuration=config, training_performed=False,
                           exploratory_reused_test=True, thresholds_fit='train only',
                           uncertainty='SD entre semillas, no IC temporal; benchmark determinista repetido solo para emparejar',
                           quantile_rule='linear; cortes repetidos/extremos fusionados',
                           activity_rule='p > umbral; igualdad pertenece a inactivo',
                           delta_rule='estable si -tau <= delta <= tau',
                           metric_caution='error ordinal no es masa economica; para actividad dentro_un_bin siempre 1')
    ph_json(ctx['manifest'], out/'manifest_completo.json')
    print('Guardado completo:', out)
    return out


RESULTADOS_22 = ex_run(CARPETA_PROYECTO, 'delta', CONFIG_EXPERIMENTO)


Mounted at /content/drive
Semilla 11: redondeo compatible verificado; diferencia maxima 8.91e-09.
Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.
delta: semilla 11, leyendo predicciones; cero entrenamientos.
delta: semilla 23, leyendo predicciones; cero entrenamientos.
delta: semilla 42, leyendo predicciones; cero entrenamientos.
delta: semilla 67, leyendo predicciones; cero entrenamientos.
delta: semilla 101, leyendo predicciones; cero entrenamientos.
delta: semilla 137, leyendo predicciones; cero entrenamientos.
delta: semilla 211, leyendo predicciones; cero entrenamientos.
delta: semilla 307, leyendo predicciones; cero entrenamientos.
delta: semilla 401, leyendo predicciones; cero entrenamientos.
delta: semilla 503, leyendo predicciones; cero entrenamientos.


,escenario,cohorte,Modelo,accuracy_mean,balanced_accuracy_mean,macro_f1_mean
0,mediana_cambio_absoluto,constantes_en_train,Bin modal de train,0.619812,0.541667,0.440599
1,mediana_cambio_absoluto,constantes_en_train,GAT + NPP,0.230944,0.273094,0.179264
2,mediana_cambio_absoluto,constantes_en_train,GAT 100% libre,0.230328,0.267693,0.169696
3,mediana_cambio_absoluto,constantes_en_train,GAT NPP solo aristas,0.231465,0.268766,0.172269
4,mediana_cambio_absoluto,constantes_en_train,Persistencia,0.619812,0.541667,0.440599
5,mediana_cambio_absoluto,todos_elegibles,Bin modal de train,0.574502,0.373673,0.278898
6,mediana_cambio_absoluto,todos_elegibles,GAT + NPP,0.549987,0.443794,0.428754
7,mediana_cambio_absoluto,todos_elegibles,GAT 100% libre,0.554499,0.441775,0.422475
8,mediana_cambio_absoluto,todos_elegibles,GAT NPP solo aristas,0.562158,0.450150,0.435327
9,mediana_cambio_absoluto,todos_elegibles,Persistencia,0.574502,0.373673,0.278898


,escenario,cohorte,evento,Modelo,recall_mean,precision_mean,F1_mean,falsas_alarmas_mean
0,mediana_cambio_absoluto,constantes_en_train,cambio_relevante,Bin modal de train,0.000000,NaN,0.000000,0.000000
1,mediana_cambio_absoluto,constantes_en_train,cambio_relevante,GAT + NPP,0.999957,0.380177,0.550904,1.000000
2,mediana_cambio_absoluto,constantes_en_train,cambio_relevante,GAT 100% libre,0.999940,0.380174,0.550898,1.000000
3,mediana_cambio_absoluto,constantes_en_train,cambio_relevante,GAT NPP solo aristas,0.999946,0.380175,0.550900,1.000000
4,mediana_cambio_absoluto,constantes_en_train,cambio_relevante,Persistencia,0.000000,NaN,0.000000,0.000000
5,mediana_cambio_absoluto,constantes_en_train,ganancia,Bin modal de train,0.000000,NaN,0.000000,0.000000
6,mediana_cambio_absoluto,constantes_en_train,ganancia,GAT + NPP,0.903259,0.183713,0.305325,0.923785
7,mediana_cambio_absoluto,constantes_en_train,ganancia,GAT 100% libre,0.901690,0.183384,0.304781,0.924179
8,mediana_cambio_absoluto,constantes_en_train,ganancia,GAT NPP solo aristas,0.898023,0.183137,0.304230,0.921942
9,mediana_cambio_absoluto,constantes_en_train,ganancia,Persistencia,0.000000,NaN,0.000000,0.000000


Guardado completo: /content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1/21_delta_sin_entreno/e4556bf43c/20260908T105356512986Z


## 23. Estados de participación por cuantiles de train — experimento independiente sin entrenamiento

Ejecutar **solo la siguiente celda en CPU**, incluso con runtime vacío. No requiere
ejecutar 20, 21, 21.b ni los otros nuevos experimentos. Lee config/activos del resultado
base, las predicciones continuas de las diez semillas y únicamente dfyp para recuperar
train. No carga pesos ni construye features ni entrena modelos. Puede pedir autorización
para montar Drive. El acceso y lectura de archivos sí llevan tiempo.

Mide **nivel relativo**. Clase 0: p <= 1e-8. Entre las observaciones positivas de
train se calculan Q25, Q50 y Q75 por activo para hasta cuatro niveles positivos.
Los límites se congelan y se aplican a todos los modelos. Se eliminan cortes repetidos
y extremos: no se separan observaciones iguales al azar. Los nodos con menos de 100
observaciones positivas o sin variación positiva quedan fuera de esta evaluación,
identificados en cobertura_train.csv. Cien barras no equivalen a cien muestras independientes.

**En qué fijarse:** cobertura de activos, ocupación train/test, macro-F1, balanced accuracy
y confusión por clase. El equilibrio solo se busca entre valores positivos en train;
ni la clase inactiva ni test quedan necesariamente equilibrados. Revisar el control
`Bin modal de train`. Un bin tiene significado relativo al activo, no la misma amplitud
económica que el bin de otro activo. Las comparaciones con experimentos anteriores
deben respetar la cobertura: no atribuir a mejora del modelo una exclusión de nodos.


**Control metodológico:** train y validación conservan sus cortes y purgas originales;
solo train define estadísticas/umbrales y controles. Test se reutiliza de forma
exploratoria: estos diseños nacen después de mirarlo y no son confirmación independiente.
No shuffle, no modificación de predicciones, no selección por rendimiento en test.
La dispersión entre semillas no es incertidumbre temporal.

CSV por modelo/nodo/semilla, matrices de confusión, ocupaciones, etiquetas NPZ, umbrales,
comparaciones emparejadas y manifiesto se guardan en
`GAT_crypto_extended_audits_v1/21_cuantiles_sin_entreno/`, en una carpeta nueva por ejecución.
El benchmark modal y persistencia se repiten solo para emparejar: no son diez modelos
independientes. La marca manifest_completo aparece solo al terminar todos los guardados.


In [2]:
# Ejecutar solo esta celda. No necesita GPU ni reentrena.
CARPETA_PROYECTO = '/content/drive/MyDrive/Neural/NPP/Cripto'
CONFIG_EXPERIMENTO = {'activity_floor': 1e-08, 'quantiles': [0.25, 0.5, 0.75], 'min_positive_train': 100}

import hashlib

import json

import os

from pathlib import Path

import numpy as np

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)

POSTHOC_VERSION = 'frozen_predictions_v1_target_precision_fix'

POSTHOC_SEEDS = [11, 23, 42, 67, 101, 137, 211, 307, 401, 503]

POSTHOC_MODELS = {
    'Persistencia': 'persistencia',
    'GAT 100% libre': 'gat_libre',
    'GAT NPP solo aristas': 'gat_npp_aristas',
    'GAT + NPP': 'gat_npp',
}

def ph_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def ph_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def ph_json(value, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)

def ph_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(tmp, path)

def ph_mount(root):
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        try:
            drive.mount('/content/drive')
            list(Path(root).iterdir())
        except OSError:
            drive.mount('/content/drive', force_remount=True)
    if not Path(root).is_dir():
        raise FileNotFoundError(f'No existe la carpeta del proyecto: {root}')

def ph_prob(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError('Predicciones/target deben ser matrices finitas y no negativas.')
    mass = values.sum(axis=1, keepdims=True)
    if np.any(mass <= 0) or not np.allclose(mass, 1., atol=1e-5, rtol=0):
        raise ValueError('El artefacto no contiene un simplex cerrado.')
    return values / mass

def ph_context(root):
    root = Path(root)
    ph_mount(root)
    base = root / 'GAT_crypto_results'
    config_path = base / 'config.json'
    reference_path = base / 'predicciones_test.npz'
    cfg = json.loads(config_path.read_text(encoding='utf-8'))
    with np.load(reference_path, allow_pickle=False) as saved:
        assets = saved['assets'].astype(str)
        t = saved['t'].copy()
        target_raw = saved['target'].copy()
        # Older exports stored timestamps as object arrays. Do not unpickle them.
        try:
            saved_dates = pd.to_datetime(saved['timestamps'].astype(str), utc=True)
        except ValueError as error:
            if 'Object arrays cannot be loaded' not in str(error):
                raise
            saved_dates = None
    if t.ndim != 1 or not np.issubdtype(t.dtype, np.integer) or len(t) < 2:
        raise ValueError('Indices temporales de referencia invalidos.')
    if np.any(np.diff(t) <= 0) or np.any(t < 0) or len(set(assets)) != len(assets):
        raise ValueError('Fechas duplicadas/desordenadas o activos duplicados.')
    target = ph_prob(target_raw)
    if target.shape != (len(t), len(assets)):
        raise ValueError('El orden/dimension de activos no coincide con el target.')
    step = pd.Timedelta(cfg['frequency'])
    expected_dates = pd.Timestamp(cfg['start_timestamp']) + (t + int(cfg['horizon'])) * step
    dates = pd.DatetimeIndex(expected_dates)
    if saved_dates is not None and not np.array_equal(saved_dates.asi8, dates.asi8):
        raise ValueError('Config y timestamps guardados pertenecen a experimentos distintos.')
    ablation = root / 'GAT_crypto_ablation_10_seeds'
    paths = {s: ablation / f'seed_{s:04d}' / 'predicciones_test.npz' for s in POSTHOC_SEEDS}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('Faltan predicciones. No se entrenara nada: ' + ', '.join(missing))
    manifest = {
        'version': POSTHOC_VERSION, 'training_performed': False,
        'prediction_source': 'predicciones_test.npz de ablacion de diez semillas',
        'test_reused': True, 'new_external_validation': False,
        'reference_path': str(reference_path), 'reference_sha256': ph_hash(reference_path),
        'config_path': str(config_path), 'config_sha256': ph_hash(config_path),
        'config': cfg, 'seeds': POSTHOC_SEEDS, 'assets': assets.tolist(),
        'n_test': len(t), 'target_start': str(dates[0]), 'target_end': str(dates[-1]),
        'timestamps_source': 'config + indices t; legacy object array not unpickled' if saved_dates is None else 'saved timestamps checked against config and t',
        'sources': [],
    }
    return dict(root=root, cfg=cfg, assets=assets, t=t, target=target,
                target_raw=target_raw, dates=dates, paths=paths, manifest=manifest)

def ph_load_seed(ctx, seed):
    path = ctx['paths'][seed]
    digest = ph_hash(path)
    with np.load(path, allow_pickle=False) as saved:
        if not np.array_equal(saved['t'], ctx['t']):
            raise ValueError(f'Semilla {seed}: distinto orden temporal.')
        seed_target_raw = saved['target'].copy()
        seed_target = ph_prob(seed_target_raw)
        reference = ctx.get('ablation_target_reference', ctx['target'])
        # El export base convierte a float32; el dataset puede producir float64
        # al dividir por una suma float64. Comparar el simplex, no sus bits.
        # Tolerancia relativa de cuatro eps float32, sin tolerancia absoluta:
        # un cero no se acepta como una cuota positiva, por diminuta que sea.
        tolerance = 4 * np.finfo(np.float32).eps
        if seed_target.shape != reference.shape:
            raise ValueError(f'Semilla {seed}: dimensiones del target incompatibles.')
        delta = np.abs(seed_target - reference)
        scale = np.maximum(np.abs(seed_target), np.abs(reference))
        same_support = np.array_equal(seed_target == 0, reference == 0)
        close = same_support and np.all(delta <= tolerance * scale)
        max_abs = float(delta.max())
        max_rel = float(np.max(np.divide(delta, scale, out=np.zeros_like(delta), where=scale > 0)))
        if not close:
            raise ValueError(
                f'Semilla {seed}: discrepancia real de target; no se mezclan archivos. '
                f'Max abs={max_abs:.6g}, max rel={max_rel:.6g}, '
                f'mismo soporte={same_support}, tolerancia rel={tolerance:.6g}. '
                'Revisar export base y ablacion; no hace falta reentrenar para diagnosticar.'
            )
        ctx['manifest'].setdefault('target_alignment', []).append({
            'seed': int(seed), 'comparison': 'normalized simplex, identical zero support',
            'relative_tolerance': float(tolerance), 'absolute_tolerance': 0.,
            'max_absolute_difference': max_abs, 'max_relative_difference': max_rel,
            'reference': 'first ablation seed' if 'ablation_target_reference' in ctx else 'base export',
        })
        if 'ablation_target_reference' not in ctx:
            # Evaluar contra los valores reales del mismo paquete de predicciones.
            ctx['ablation_target_reference'] = seed_target.copy()
            ctx['target'] = seed_target.copy()
            ctx['manifest']['metric_target_source'] = str(path)
        if max_abs > 0:
            print(f'Semilla {seed}: redondeo compatible verificado; diferencia maxima {max_abs:.3g}.', flush=True)
        pred = {name: ph_prob(saved[key]) for name, key in POSTHOC_MODELS.items()}
    for name, values in pred.items():
        if values.shape != ctx['target'].shape:
            raise ValueError(f'{seed}/{name}: dimensiones incompatibles.')
    if 'persistence_reference' not in ctx:
        ctx['persistence_reference'] = pred['Persistencia'].copy()
    elif not np.array_equal(pred['Persistencia'], ctx['persistence_reference']):
        raise ValueError('Persistencia cambio entre semillas; no se mezclan experimentos.')
    ctx['manifest']['sources'].append({'seed': seed, 'path': str(path), 'sha256': digest})
    return pred

def ph_check_source(ctx, left, right, label):
    left, right = ph_prob(left), ph_prob(right)
    if left.shape != right.shape:
        raise ValueError(f'{label}: dimensiones incompatibles.')
    delta = np.abs(left - right)
    scale = np.maximum(np.abs(left), np.abs(right))
    tolerance = 4 * np.finfo(np.float32).eps
    same_support = np.array_equal(left == 0, right == 0)
    if not same_support or not np.all(delta <= tolerance * scale):
        raise ValueError(f'{label}: discrepancia real, max abs={delta.max():.6g}; no se mezclan datos.')
    ctx['manifest'].setdefault('source_alignment', []).append({
        'comparison': label, 'relative_tolerance': float(tolerance),
        'absolute_tolerance': 0., 'identical_zero_support': True,
        'max_absolute_difference': float(delta.max()),
    })

def ph_train_statistics(ctx, warmup=12128):
    cfg = ctx['cfg']
    path = Path(cfg['dfyp_path'])
    print('Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.', flush=True)
    digest = ph_hash(path)
    frame = pd.read_parquet(path) if path.suffix.lower() in {'.parquet', '.pq'} else pd.read_csv(path)
    if 'YS' in frame.columns and 'YS_p' not in frame.columns:
        frame = frame.rename(columns=lambda c: c[:-2] + '_p' if c.endswith('_y') else 'YS_p' if c == 'YS' else c)
    cols = [c for c in frame.columns if c.endswith('_p') and c != 'YS_p']
    if cols != [a + '_p' for a in ctx['assets']]:
        raise ValueError('dfyp no conserva el universo/orden de activos de las predicciones.')
    horizon, stride, purge = int(cfg['horizon']), int(cfg['sample_stride']), int(cfg['purge'])
    test_start = len(frame) - horizon - int(cfg['test_size'])
    validation_end = test_start - purge
    validation_start = validation_end - int(cfg['validation_size'])
    train_end = validation_start - purge
    all_t = np.arange(warmup, len(frame) - horizon, stride)
    train_t = all_t[all_t < train_end]
    val_t = all_t[(all_t >= validation_start) & (all_t < validation_end)]
    test_t = all_t[all_t >= test_start]
    if not np.array_equal(test_t, ctx['t']):
        raise ValueError('dfyp/config no reproduce el test archivado; no se recalculan cortes silenciosamente.')
    if len(train_t) < 2 or len(val_t) == 0 or train_t[-1] + horizon >= val_t[0] or val_t[-1] + horizon >= test_t[0]:
        raise ValueError('Split vacio o purga incompatible.')
    # Identical normalization to the original extraction; only train targets fit thresholds.
    def normalize_rows(indices):
        values = frame.iloc[indices][cols].to_numpy(dtype=np.float32, copy=True)
        if np.any(values < -1e-8):
            raise ValueError('dfyp contiene masas negativas.')
        values = np.nan_to_num(values, nan=0., posinf=0., neginf=0.)
        values = np.clip(values, 0., None)
        total = values.sum(axis=1, keepdims=True, dtype=np.float64)
        if np.any(total <= 0):
            raise ValueError('dfyp contiene filas sin masa.')
        return (values / total).astype(np.float32)
    ph_check_source(ctx, normalize_rows(test_t + horizon), ctx['target'], 'dfyp frente a target archivado')
    # Validate persistence against raw inputs before fitting any bins.
    with np.load(ctx['paths'][POSTHOC_SEEDS[0]], allow_pickle=False) as saved:
        ph_check_source(ctx, normalize_rows(test_t), saved['persistencia'], 'dfyp frente a persistencia archivada')
    train = ph_prob(normalize_rows(train_t + horizon))
    mu = train.mean(axis=0)
    sigma = train.std(axis=0, ddof=1)
    ctx['manifest']['train_bins'] = {
        'dfyp_path': str(path), 'dfyp_sha256': digest, 'warmup': warmup,
        'fit_rows': 'targets P[train_t + horizon]', 'ddof': 1, 'n_train': len(train_t),
        'train_origin_first': int(train_t[0]), 'train_origin_last': int(train_t[-1]),
        'last_train_target': int(train_t[-1] + horizon), 'validation_first_origin': int(val_t[0]),
        'test_first_origin': int(test_t[0]), 'fit_uses_validation_or_test': False,
        'legacy_provenance': 'Config y assets de resultados base, alineados con target y persistencia de ablacion, tolerancia relativa float32 documentada. Warmup original fijo 12128; archivos antiguos sin hash historico de train.'}
    return train, mu, sigma, train_t + horizon

def ph_class_metrics(actual, predicted, n_classes):
    confusion = np.bincount(actual.astype(int) * n_classes + predicted.astype(int),
                            minlength=n_classes ** 2).reshape(n_classes, n_classes)
    support, guessed = confusion.sum(axis=1), confusion.sum(axis=0)
    tp = confusion.diagonal()
    recall = np.divide(tp, support, out=np.zeros(n_classes, float), where=support > 0)
    f1 = np.divide(2 * tp, support + guessed, out=np.zeros(n_classes, float), where=(support + guessed) > 0)
    delta = np.abs(actual.astype(int) - predicted.astype(int))
    result = dict(accuracy=float(np.mean(actual == predicted)),
                  balanced_accuracy=float(recall[support > 0].mean()),
                  macro_f1=float(f1[(support + guessed) > 0].mean()),
                  error_ordinal=float(delta.mean()), dentro_un_bin=float((delta <= 1).mean()),
                  tasa_subestimacion=float((predicted < actual).mean()),
                  tasa_sobreestimacion=float((predicted > actual).mean()),
                  n_clases=n_classes, clases_reales_presentes=int((support > 0).sum()))
    return result, confusion

# Helpers compartidos, embebidos completos en cada celda independiente.
def ex_div(n, d):
    return float(n / d) if d else np.nan


def ex_event(y, p):
    y, p = np.asarray(y, bool), np.asarray(p, bool)
    tp, fp = int((y & p).sum()), int((~y & p).sum())
    fn, tn = int((y & ~p).sum()), int((~y & ~p).sum())
    return dict(N=int(y.size), TP=tp, FP=fp, FN=fn, TN=tn,
                prevalencia=ex_div(tp+fn, y.size), tasa_alertas=ex_div(tp+fp, y.size),
                recall=ex_div(tp, tp+fn), precision=ex_div(tp, tp+fp),
                F1=ex_div(2*tp, 2*tp+fp+fn), falsas_alarmas=ex_div(fp, fp+tn))


def ex_current_train(ctx, rows):
    path = Path(ctx['cfg']['dfyp_path'])
    frame = pd.read_parquet(path) if path.suffix.lower() in ('.parquet', '.pq') else pd.read_csv(path)
    if 'YS' in frame.columns and 'YS_p' not in frame.columns:
        frame = frame.rename(columns=lambda c: c[:-2]+'_p' if c.endswith('_y') else 'YS_p' if c == 'YS' else c)
    values = frame.iloc[rows-int(ctx['cfg']['horizon'])][[a+'_p' for a in ctx['assets']]].to_numpy(dtype=np.float32)
    if np.any(values < -1e-8):
        raise ValueError('Masas negativas en train.')
    values = np.clip(np.nan_to_num(values, nan=0., posinf=0., neginf=0.), 0., None)
    total = values.sum(axis=1, keepdims=True, dtype=np.float64)
    if np.any(total <= 0):
        raise ValueError('Train sin masa.')
    values = (values/total).astype(np.float32)
    if ph_hash(path) != ctx['manifest']['train_bins']['dfyp_sha256']:
        raise ValueError('dfyp cambio durante la lectura; detener sin mezclar archivos.')
    return ph_prob(values)


def ex_delta_labels(values, tau):
    # Clase 0=baja, 1=estable (incluye ambos limites), 2=sube.
    return np.where(values < -tau, 0, np.where(values > tau, 2, 1)).astype(np.int64)


def ex_level_labels(values, cuts, activity):
    labels = np.zeros(values.shape, dtype=np.int64)
    for j, boundaries in enumerate(cuts):
        active = values[:, j] > activity
        labels[active, j] = 1 + np.searchsorted(boundaries, values[active, j], side='right')
    return labels


def ex_run(root, experiment, config):
    if experiment not in ('delta', 'cuantiles', 'actividad'):
        raise ValueError('Experimento desconocido.')
    ctx = ph_context(root)
    first = ph_load_seed(ctx, POSTHOC_SEEDS[0])
    current = first['Persistencia'].copy()
    del first
    ctx['manifest']['initial_alignment'] = ctx['manifest'].pop('target_alignment')
    ctx['manifest']['sources'].clear()
    train, mu, sigma, rows = ph_train_statistics(ctx)
    constants = sigma <= 1e-8
    target = ctx['target']
    n_nodes = len(ctx['assets'])
    settings = []
    specifications = []
    coverage = []
    if experiment == 'delta':
        q, floor = float(config['quantile_abs_delta']), float(config['min_change'])
        if not 0 < q < 1 or floor <= 0:
            raise ValueError('Cuantil o piso de cambio invalido.')
        previous = ex_current_train(ctx, rows)
        delta = train-previous
        tau = np.maximum(np.quantile(np.abs(delta), q, axis=0), floor)
        train_y = ex_delta_labels(delta, tau)
        actual = ex_delta_labels(target-current, tau)
        settings.append(('mediana_cambio_absoluto', train_y, actual, np.full(n_nodes, 3), np.ones(n_nodes, bool), tau))
        for j, a in enumerate(ctx['assets']):
            specifications.append(dict(Nodo=a, tau=float(tau[j]), cuantil=q, piso=floor,
                                       constante_train=bool(constants[j]), n_train=len(train)))
    elif experiment == 'cuantiles':
        activity = float(config['activity_floor'])
        qs = np.asarray(config['quantiles'], float)
        minimum = int(config['min_positive_train'])
        if activity <= 0 or minimum < 2 or qs.ndim != 1 or not len(qs) or np.any(np.diff(qs) <= 0) or np.any((qs <= 0) | (qs >= 1)):
            raise ValueError('Configuracion de cuantiles invalida.')
        cuts, eligible, counts = [], [], []
        for j, a in enumerate(ctx['assets']):
            positive = train[train[:, j] > activity, j]
            # No confundir jitter de cierre float32 con variacion positiva real.
            ok = (len(positive) >= minimum and
                  np.ptp(positive) > 4*np.finfo(np.float32).eps*max(float(positive.max()), activity))
            boundaries = np.unique(np.quantile(positive, qs)) if ok else np.array([])
            # Cortes extremos no crean categorias positivas vacias en train.
            if ok:
                boundaries = boundaries[(boundaries > positive.min()) & (boundaries < positive.max())]
            cuts.append(boundaries)
            eligible.append(ok)
            counts.append(len(boundaries)+2)
            coverage.append(dict(Nodo=a, elegible=ok, n_positivos_train=len(positive),
                                 n_bins_positivos=len(boundaries)+1 if ok else 0,
                                 motivo='ok' if ok else 'historia positiva insuficiente o sin variacion'))
            specifications.append(dict(Nodo=a, umbral_actividad=activity, cortes_positivos=json.dumps(boundaries.tolist()),
                                       elegible=ok, min_positivos_train=minimum))
        eligible = np.asarray(eligible, bool)
        if not eligible.any():
            raise ValueError('Ningun activo tiene historia positiva suficiente; no se inventan cuantiles.')
        settings.append(('actividad_y_cuartiles_positivos', ex_level_labels(train, cuts, activity),
                         ex_level_labels(target, cuts, activity), np.asarray(counts), eligible, (cuts, activity)))
    else:
        levels = np.asarray(config['activity_floors'], float)
        if levels.ndim != 1 or not len(levels) or np.any((levels <= 0) | (levels >= 1)) or np.any(np.diff(levels) <= 0):
            raise ValueError('Umbrales de actividad invalidos.')
        for floor in levels:
            settings.append((f'umbral_{floor:g}', (train > floor).astype(np.int64),
                             (target > floor).astype(np.int64), np.full(n_nodes, 2), np.ones(n_nodes, bool), float(floor)))
            for j, a in enumerate(ctx['assets']):
                specifications.append(dict(Nodo=a, umbral_actividad=float(floor),
                                           tasa_actividad_train=float((train[:, j] > floor).mean()), constante_train=bool(constants[j])))
    tag = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:10]
    out = ctx['root'] / 'GAT_crypto_extended_audits_v1' / ('21_'+experiment+'_sin_entreno') / tag / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    out.mkdir(parents=True)
    ph_csv(pd.DataFrame(specifications), out/'umbrales_train.csv')
    if coverage:
        ph_csv(pd.DataFrame(coverage), out/'cobertura_train.csv')
    occupation = []
    modes = {}
    for name, train_y, actual, counts, eligible, state in settings:
        modes[name] = np.array([np.bincount(train_y[:, j], minlength=int(counts[j])).argmax() for j in range(n_nodes)])
        for j, asset in enumerate(ctx['assets']):
            if not eligible[j]:
                continue
            for label in range(int(counts[j])):
                occupation.append(dict(escenario=name, Nodo=asset, clase=label,
                                       n_train=int((train_y[:, j] == label).sum()), n_test=int((actual[:, j] == label).sum())))
        ph_npz(out/f'etiquetas_reales_{name}.npz', assets=ctx['assets'], t=ctx['t'], elegible=eligible,
               train_target_rows=rows, train_labels=train_y, test_labels=actual)
    ph_csv(pd.DataFrame(occupation), out/'ocupacion_train_test.csv')
    del train
    all_nodes, all_events = [], []
    for seed in POSTHOC_SEEDS:
        print(f'{experiment}: semilla {seed}, leyendo predicciones; cero entrenamientos.', flush=True)
        predictions = ph_load_seed(ctx, seed)
        node_rows, event_rows, confusion_rows = [], [], []
        for name, train_y, actual, counts, eligible, state in settings:
            encoded = {}
            for model, values in [*predictions.items(), ('Bin modal de train', None)]:
                if values is None:
                    pred = np.broadcast_to(modes[name], actual.shape).copy()
                elif experiment == 'delta':
                    pred = ex_delta_labels(values-current, state)
                elif experiment == 'cuantiles':
                    pred = ex_level_labels(values, *state)
                else:
                    pred = (values > state).astype(np.int64)
                encoded[POSTHOC_MODELS.get(model, 'modal_train')] = pred
                for j, asset in enumerate(ctx['assets']):
                    if not eligible[j]:
                        continue
                    y, p = actual[:, j], pred[:, j]
                    metrics, cm = ph_class_metrics(y, p, int(counts[j]))
                    meta = dict(Modelo=model, seed=seed, escenario=name, Nodo=asset,
                                constante_train=bool(constants[j]), modelo_entrenado=False,
                                npp_aristas=model in ('GAT NPP solo aristas', 'GAT + NPP'), npp_loss=model == 'GAT + NPP')
                    node_rows.append({**meta, **metrics, 'N_test': len(y)})
                    for a, b in zip(*np.nonzero(cm)):
                        confusion_rows.append({**meta, 'clase_real': int(a), 'clase_predicha': int(b), 'n': int(cm[a, b])})
                for cohort, cols in [('todos_elegibles', eligible), ('variables_en_train', eligible & ~constants), ('constantes_en_train', eligible & constants)]:
                    if not cols.any():
                        continue
                    y, p = actual[:, cols], pred[:, cols]
                    meta = dict(Modelo=model, seed=seed, escenario=name, cohorte=cohort, modelo_entrenado=False)
                    if experiment == 'delta':
                        event_rows.append({**meta, 'evento': 'cambio_relevante', **ex_event(y != 1, p != 1),
                                           'direccion_correcta_en_cambios': ex_div(int(((y == p) & (y != 1)).sum()), int((y != 1).sum()))})
                        for label, event in [(0, 'perdida'), (2, 'ganancia')]:
                            event_rows.append({**meta, 'evento': event, **ex_event(y == label, p == label)})
                    elif experiment == 'actividad':
                        now = current[:, cols] > state
                        event_rows.append({**meta, 'evento': 'estado_activo', **ex_event(y == 1, p == 1)})
                        event_rows.append({**meta, 'evento': 'activacion_desde_inactivo', **ex_event((y == 1)[~now], (p == 1)[~now])})
                        event_rows.append({**meta, 'evento': 'desactivacion_desde_activo', **ex_event((y == 0)[now], (p == 0)[now])})
            ph_npz(out/f'predicciones_{name}_seed_{seed:04d}.npz', assets=ctx['assets'], seed=np.array(seed), t=ctx['t'], elegible=eligible, target_labels=actual, **encoded)
        ph_csv(pd.DataFrame(node_rows), out/f'metricas_nodos_seed_{seed:04d}.csv')
        ph_csv(pd.DataFrame(confusion_rows), out/f'confusion_seed_{seed:04d}.csv')
        if event_rows:
            ph_csv(pd.DataFrame(event_rows), out/f'eventos_seed_{seed:04d}.csv')
        all_nodes.extend(node_rows); all_events.extend(event_rows)
    nodes = pd.DataFrame(all_nodes)
    metrics = ['accuracy', 'balanced_accuracy', 'macro_f1', 'error_ordinal', 'dentro_un_bin']
    macros = []
    for cohort, subset in [('todos_elegibles', nodes), ('variables_en_train', nodes[~nodes.constante_train]), ('constantes_en_train', nodes[nodes.constante_train])]:
        if subset.empty:
            continue
        grouped = subset.groupby(['escenario', 'Modelo', 'seed'], as_index=False)
        means = grouped[metrics].mean()
        means = means.merge(grouped.agg(N_nodos=('Nodo', 'nunique')), on=['escenario', 'Modelo', 'seed'], validate='one_to_one')
        means['cohorte'] = cohort
        macros.append(means)
    macro = pd.concat(macros, ignore_index=True)
    def summarize(df, keys, cols):
        result = df.groupby(keys)[cols].agg(['mean', 'std', 'count'])
        result.columns = ['_'.join(c) for c in result.columns]
        return result.reset_index()
    summary = summarize(macro, ['escenario', 'cohorte', 'Modelo'], metrics)
    ph_csv(nodes, out/'metricas_por_nodo.csv'); ph_csv(macro, out/'macro_por_semilla.csv')
    ph_csv(summary, out/'resumen_modelos.csv')
    paired = []
    for ref in ['Persistencia', 'GAT 100% libre', 'GAT NPP solo aristas', 'Bin modal de train']:
        merged = macro[macro.Modelo != ref].merge(macro[macro.Modelo == ref], on=['escenario', 'cohorte', 'seed'], suffixes=('', '_ref'), validate='many_to_one')
        for _, row in merged.iterrows():
            for m in metrics:
                d = row[m]-row[m+'_ref']
                paired.append(dict(Modelo=row.Modelo, Referencia=ref, seed=int(row.seed), escenario=row.escenario, cohorte=row.cohorte,
                                   metrica=m, delta=float(d), mejora_favorable=float(-d if m == 'error_ordinal' else d)))
    ph_csv(pd.DataFrame(paired), out/'deltas_emparejados.csv')
    display(summary[['escenario', 'cohorte', 'Modelo', 'accuracy_mean', 'balanced_accuracy_mean', 'macro_f1_mean']])
    if all_events:
        events = pd.DataFrame(all_events)
        event_summary = summarize(events, ['escenario', 'cohorte', 'evento', 'Modelo'], ['prevalencia', 'tasa_alertas', 'recall', 'precision', 'F1', 'falsas_alarmas'])
        ph_csv(events, out/'eventos_por_semilla.csv'); ph_csv(event_summary, out/'resumen_eventos.csv')
        display(event_summary[['escenario', 'cohorte', 'evento', 'Modelo', 'recall_mean', 'precision_mean', 'F1_mean', 'falsas_alarmas_mean']])
    ctx['manifest'].update(experiment=experiment, configuration=config, training_performed=False,
                           exploratory_reused_test=True, thresholds_fit='train only',
                           uncertainty='SD entre semillas, no IC temporal; benchmark determinista repetido solo para emparejar',
                           quantile_rule='linear; cortes repetidos/extremos fusionados',
                           activity_rule='p > umbral; igualdad pertenece a inactivo',
                           delta_rule='estable si -tau <= delta <= tau',
                           metric_caution='error ordinal no es masa economica; para actividad dentro_un_bin siempre 1')
    ph_json(ctx['manifest'], out/'manifest_completo.json')
    print('Guardado completo:', out)
    return out


RESULTADOS_23 = ex_run(CARPETA_PROYECTO, 'cuantiles', CONFIG_EXPERIMENTO)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Semilla 11: redondeo compatible verificado; diferencia maxima 8.91e-09.
Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.
cuantiles: semilla 11, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 23, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 42, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 67, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 101, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 137, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 211, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 307, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 401, leyendo predicciones; cero entrenamientos.
cuantiles: semilla 503, leyendo predicciones; cero entrenamientos.


,escenario,cohorte,Modelo,accuracy_mean,balanced_accuracy_mean,macro_f1_mean
0,actividad_y_cuartiles_positivos,todos_elegibles,Bin modal de train,0.282178,0.250410,0.122246
1,actividad_y_cuartiles_positivos,todos_elegibles,GAT + NPP,0.370693,0.297173,0.264025
2,actividad_y_cuartiles_positivos,todos_elegibles,GAT 100% libre,0.375591,0.301026,0.270642
3,actividad_y_cuartiles_positivos,todos_elegibles,GAT NPP solo aristas,0.376650,0.299061,0.268927
4,actividad_y_cuartiles_positivos,todos_elegibles,Persistencia,0.477053,0.385847,0.385839
5,actividad_y_cuartiles_positivos,variables_en_train,Bin modal de train,0.282178,0.250410,0.122246
6,actividad_y_cuartiles_positivos,variables_en_train,GAT + NPP,0.370693,0.297173,0.264025
7,actividad_y_cuartiles_positivos,variables_en_train,GAT 100% libre,0.375591,0.301026,0.270642
8,actividad_y_cuartiles_positivos,variables_en_train,GAT NPP solo aristas,0.376650,0.299061,0.268927
9,actividad_y_cuartiles_positivos,variables_en_train,Persistencia,0.477053,0.385847,0.385839


Guardado completo: /content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1/21_cuantiles_sin_entreno/d37bec88d4/20260908T105437491990Z


In [ ]:
# la mejora continua observada anteriormente no implica una mejor identificación del cuartil futuro de participación.
# En esta discretización, la persistencia constituye el referente más fuerte.
# El resultado delimita el alcance predictivo de la implementación actual de NPP,
# sin demostrar por sí mismo sobreajuste ni invalidar su utilidad para otras tareas.

# Una explicación compatible con los resultados es que un modelo puede acercarse
# más al valor continuo y, aun así, cruzar una frontera de clasificación equivocada.
# Por ejemplo, predecir cerca de un cuartil puede reducir el error numérico
# sin acertar el intervalo. Los modelos originales tampoco fueron entrenados
# para optimizar estas etiquetas.

## 24. Actividad, activación y desactivación — experimento independiente sin entrenamiento

Ejecutar **solo la siguiente celda en CPU**, incluso con runtime vacío. No requiere
ejecutar 20, 21, 21.b ni los otros nuevos experimentos. Lee config/activos del resultado
base, las predicciones continuas de las diez semillas y únicamente dfyp para recuperar
train. No carga pesos ni construye features ni entrena modelos. Puede pedir autorización
para montar Drive. El acceso y lectura de archivos sí llevan tiempo.

Mide **presencia de masa**, separada de su nivel. Se reportan TODOS los umbrales
fijados de antemano: 1e-8, 1e-6 y 1e-4 de cuota (0.000001%, 0.0001% y 0.01%).
Son umbrales operativos de sensibilidad, no límites económicos inferidos ni optimizados.
Activo significa p > umbral; la igualdad pertenece a inactivo.

**En qué fijarse:** prevalencia, recall/precisión/F1 y falsas alarmas. Se distinguen
estado activo, activación condicionada a estar inactivo en t, y desactivación condicionada
a estar activo en t. Los grupos usan el estado disponible en t, no el futuro.
Se reportan todos, variables y constantes en train. No interpretar un recall alto
como éxito si la precisión es baja. NaN significa que no hay denominador, no error cero.
En dos clases, dentro_un_bin siempre vale 1 y no es informativo.
No escoger retrospectivamente el umbral que favorezca NPP en este test.


**Control metodológico:** train y validación conservan sus cortes y purgas originales;
solo train define estadísticas/umbrales y controles. Test se reutiliza de forma
exploratoria: estos diseños nacen después de mirarlo y no son confirmación independiente.
No shuffle, no modificación de predicciones, no selección por rendimiento en test.
La dispersión entre semillas no es incertidumbre temporal.

CSV por modelo/nodo/semilla, matrices de confusión, ocupaciones, etiquetas NPZ, umbrales,
comparaciones emparejadas y manifiesto se guardan en
`GAT_crypto_extended_audits_v1/21_actividad_sin_entreno/`, en una carpeta nueva por ejecución.
El benchmark modal y persistencia se repiten solo para emparejar: no son diez modelos
independientes. La marca manifest_completo aparece solo al terminar todos los guardados.


In [3]:
# Ejecutar solo esta celda. No necesita GPU ni reentrena.
CARPETA_PROYECTO = '/content/drive/MyDrive/Neural/NPP/Cripto'
CONFIG_EXPERIMENTO = {'activity_floors': [1e-08, 1e-06, 0.0001]}

import hashlib

import json

import os

from pathlib import Path

import numpy as np

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)

POSTHOC_VERSION = 'frozen_predictions_v1_target_precision_fix'

POSTHOC_SEEDS = [11, 23, 42, 67, 101, 137, 211, 307, 401, 503]

POSTHOC_MODELS = {
    'Persistencia': 'persistencia',
    'GAT 100% libre': 'gat_libre',
    'GAT NPP solo aristas': 'gat_npp_aristas',
    'GAT + NPP': 'gat_npp',
}

def ph_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def ph_csv(frame, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def ph_json(value, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(tmp, path)

def ph_npz(path, **arrays):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as stream:
        np.savez_compressed(stream, **arrays)
    os.replace(tmp, path)

def ph_mount(root):
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        try:
            drive.mount('/content/drive')
            list(Path(root).iterdir())
        except OSError:
            drive.mount('/content/drive', force_remount=True)
    if not Path(root).is_dir():
        raise FileNotFoundError(f'No existe la carpeta del proyecto: {root}')

def ph_prob(values):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 2 or not np.isfinite(values).all() or np.any(values < 0):
        raise ValueError('Predicciones/target deben ser matrices finitas y no negativas.')
    mass = values.sum(axis=1, keepdims=True)
    if np.any(mass <= 0) or not np.allclose(mass, 1., atol=1e-5, rtol=0):
        raise ValueError('El artefacto no contiene un simplex cerrado.')
    return values / mass

def ph_context(root):
    root = Path(root)
    ph_mount(root)
    base = root / 'GAT_crypto_results'
    config_path = base / 'config.json'
    reference_path = base / 'predicciones_test.npz'
    cfg = json.loads(config_path.read_text(encoding='utf-8'))
    with np.load(reference_path, allow_pickle=False) as saved:
        assets = saved['assets'].astype(str)
        t = saved['t'].copy()
        target_raw = saved['target'].copy()
        # Older exports stored timestamps as object arrays. Do not unpickle them.
        try:
            saved_dates = pd.to_datetime(saved['timestamps'].astype(str), utc=True)
        except ValueError as error:
            if 'Object arrays cannot be loaded' not in str(error):
                raise
            saved_dates = None
    if t.ndim != 1 or not np.issubdtype(t.dtype, np.integer) or len(t) < 2:
        raise ValueError('Indices temporales de referencia invalidos.')
    if np.any(np.diff(t) <= 0) or np.any(t < 0) or len(set(assets)) != len(assets):
        raise ValueError('Fechas duplicadas/desordenadas o activos duplicados.')
    target = ph_prob(target_raw)
    if target.shape != (len(t), len(assets)):
        raise ValueError('El orden/dimension de activos no coincide con el target.')
    step = pd.Timedelta(cfg['frequency'])
    expected_dates = pd.Timestamp(cfg['start_timestamp']) + (t + int(cfg['horizon'])) * step
    dates = pd.DatetimeIndex(expected_dates)
    if saved_dates is not None and not np.array_equal(saved_dates.asi8, dates.asi8):
        raise ValueError('Config y timestamps guardados pertenecen a experimentos distintos.')
    ablation = root / 'GAT_crypto_ablation_10_seeds'
    paths = {s: ablation / f'seed_{s:04d}' / 'predicciones_test.npz' for s in POSTHOC_SEEDS}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError('Faltan predicciones. No se entrenara nada: ' + ', '.join(missing))
    manifest = {
        'version': POSTHOC_VERSION, 'training_performed': False,
        'prediction_source': 'predicciones_test.npz de ablacion de diez semillas',
        'test_reused': True, 'new_external_validation': False,
        'reference_path': str(reference_path), 'reference_sha256': ph_hash(reference_path),
        'config_path': str(config_path), 'config_sha256': ph_hash(config_path),
        'config': cfg, 'seeds': POSTHOC_SEEDS, 'assets': assets.tolist(),
        'n_test': len(t), 'target_start': str(dates[0]), 'target_end': str(dates[-1]),
        'timestamps_source': 'config + indices t; legacy object array not unpickled' if saved_dates is None else 'saved timestamps checked against config and t',
        'sources': [],
    }
    return dict(root=root, cfg=cfg, assets=assets, t=t, target=target,
                target_raw=target_raw, dates=dates, paths=paths, manifest=manifest)

def ph_load_seed(ctx, seed):
    path = ctx['paths'][seed]
    digest = ph_hash(path)
    with np.load(path, allow_pickle=False) as saved:
        if not np.array_equal(saved['t'], ctx['t']):
            raise ValueError(f'Semilla {seed}: distinto orden temporal.')
        seed_target_raw = saved['target'].copy()
        seed_target = ph_prob(seed_target_raw)
        reference = ctx.get('ablation_target_reference', ctx['target'])
        # El export base convierte a float32; el dataset puede producir float64
        # al dividir por una suma float64. Comparar el simplex, no sus bits.
        # Tolerancia relativa de cuatro eps float32, sin tolerancia absoluta:
        # un cero no se acepta como una cuota positiva, por diminuta que sea.
        tolerance = 4 * np.finfo(np.float32).eps
        if seed_target.shape != reference.shape:
            raise ValueError(f'Semilla {seed}: dimensiones del target incompatibles.')
        delta = np.abs(seed_target - reference)
        scale = np.maximum(np.abs(seed_target), np.abs(reference))
        same_support = np.array_equal(seed_target == 0, reference == 0)
        close = same_support and np.all(delta <= tolerance * scale)
        max_abs = float(delta.max())
        max_rel = float(np.max(np.divide(delta, scale, out=np.zeros_like(delta), where=scale > 0)))
        if not close:
            raise ValueError(
                f'Semilla {seed}: discrepancia real de target; no se mezclan archivos. '
                f'Max abs={max_abs:.6g}, max rel={max_rel:.6g}, '
                f'mismo soporte={same_support}, tolerancia rel={tolerance:.6g}. '
                'Revisar export base y ablacion; no hace falta reentrenar para diagnosticar.'
            )
        ctx['manifest'].setdefault('target_alignment', []).append({
            'seed': int(seed), 'comparison': 'normalized simplex, identical zero support',
            'relative_tolerance': float(tolerance), 'absolute_tolerance': 0.,
            'max_absolute_difference': max_abs, 'max_relative_difference': max_rel,
            'reference': 'first ablation seed' if 'ablation_target_reference' in ctx else 'base export',
        })
        if 'ablation_target_reference' not in ctx:
            # Evaluar contra los valores reales del mismo paquete de predicciones.
            ctx['ablation_target_reference'] = seed_target.copy()
            ctx['target'] = seed_target.copy()
            ctx['manifest']['metric_target_source'] = str(path)
        if max_abs > 0:
            print(f'Semilla {seed}: redondeo compatible verificado; diferencia maxima {max_abs:.3g}.', flush=True)
        pred = {name: ph_prob(saved[key]) for name, key in POSTHOC_MODELS.items()}
    for name, values in pred.items():
        if values.shape != ctx['target'].shape:
            raise ValueError(f'{seed}/{name}: dimensiones incompatibles.')
    if 'persistence_reference' not in ctx:
        ctx['persistence_reference'] = pred['Persistencia'].copy()
    elif not np.array_equal(pred['Persistencia'], ctx['persistence_reference']):
        raise ValueError('Persistencia cambio entre semillas; no se mezclan experimentos.')
    ctx['manifest']['sources'].append({'seed': seed, 'path': str(path), 'sha256': digest})
    return pred

def ph_check_source(ctx, left, right, label):
    left, right = ph_prob(left), ph_prob(right)
    if left.shape != right.shape:
        raise ValueError(f'{label}: dimensiones incompatibles.')
    delta = np.abs(left - right)
    scale = np.maximum(np.abs(left), np.abs(right))
    tolerance = 4 * np.finfo(np.float32).eps
    same_support = np.array_equal(left == 0, right == 0)
    if not same_support or not np.all(delta <= tolerance * scale):
        raise ValueError(f'{label}: discrepancia real, max abs={delta.max():.6g}; no se mezclan datos.')
    ctx['manifest'].setdefault('source_alignment', []).append({
        'comparison': label, 'relative_tolerance': float(tolerance),
        'absolute_tolerance': 0., 'identical_zero_support': True,
        'max_absolute_difference': float(delta.max()),
    })

def ph_train_statistics(ctx, warmup=12128):
    cfg = ctx['cfg']
    path = Path(cfg['dfyp_path'])
    print('Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.', flush=True)
    digest = ph_hash(path)
    frame = pd.read_parquet(path) if path.suffix.lower() in {'.parquet', '.pq'} else pd.read_csv(path)
    if 'YS' in frame.columns and 'YS_p' not in frame.columns:
        frame = frame.rename(columns=lambda c: c[:-2] + '_p' if c.endswith('_y') else 'YS_p' if c == 'YS' else c)
    cols = [c for c in frame.columns if c.endswith('_p') and c != 'YS_p']
    if cols != [a + '_p' for a in ctx['assets']]:
        raise ValueError('dfyp no conserva el universo/orden de activos de las predicciones.')
    horizon, stride, purge = int(cfg['horizon']), int(cfg['sample_stride']), int(cfg['purge'])
    test_start = len(frame) - horizon - int(cfg['test_size'])
    validation_end = test_start - purge
    validation_start = validation_end - int(cfg['validation_size'])
    train_end = validation_start - purge
    all_t = np.arange(warmup, len(frame) - horizon, stride)
    train_t = all_t[all_t < train_end]
    val_t = all_t[(all_t >= validation_start) & (all_t < validation_end)]
    test_t = all_t[all_t >= test_start]
    if not np.array_equal(test_t, ctx['t']):
        raise ValueError('dfyp/config no reproduce el test archivado; no se recalculan cortes silenciosamente.')
    if len(train_t) < 2 or len(val_t) == 0 or train_t[-1] + horizon >= val_t[0] or val_t[-1] + horizon >= test_t[0]:
        raise ValueError('Split vacio o purga incompatible.')
    # Identical normalization to the original extraction; only train targets fit thresholds.
    def normalize_rows(indices):
        values = frame.iloc[indices][cols].to_numpy(dtype=np.float32, copy=True)
        if np.any(values < -1e-8):
            raise ValueError('dfyp contiene masas negativas.')
        values = np.nan_to_num(values, nan=0., posinf=0., neginf=0.)
        values = np.clip(values, 0., None)
        total = values.sum(axis=1, keepdims=True, dtype=np.float64)
        if np.any(total <= 0):
            raise ValueError('dfyp contiene filas sin masa.')
        return (values / total).astype(np.float32)
    ph_check_source(ctx, normalize_rows(test_t + horizon), ctx['target'], 'dfyp frente a target archivado')
    # Validate persistence against raw inputs before fitting any bins.
    with np.load(ctx['paths'][POSTHOC_SEEDS[0]], allow_pickle=False) as saved:
        ph_check_source(ctx, normalize_rows(test_t), saved['persistencia'], 'dfyp frente a persistencia archivada')
    train = ph_prob(normalize_rows(train_t + horizon))
    mu = train.mean(axis=0)
    sigma = train.std(axis=0, ddof=1)
    ctx['manifest']['train_bins'] = {
        'dfyp_path': str(path), 'dfyp_sha256': digest, 'warmup': warmup,
        'fit_rows': 'targets P[train_t + horizon]', 'ddof': 1, 'n_train': len(train_t),
        'train_origin_first': int(train_t[0]), 'train_origin_last': int(train_t[-1]),
        'last_train_target': int(train_t[-1] + horizon), 'validation_first_origin': int(val_t[0]),
        'test_first_origin': int(test_t[0]), 'fit_uses_validation_or_test': False,
        'legacy_provenance': 'Config y assets de resultados base, alineados con target y persistencia de ablacion, tolerancia relativa float32 documentada. Warmup original fijo 12128; archivos antiguos sin hash historico de train.'}
    return train, mu, sigma, train_t + horizon

def ph_class_metrics(actual, predicted, n_classes):
    confusion = np.bincount(actual.astype(int) * n_classes + predicted.astype(int),
                            minlength=n_classes ** 2).reshape(n_classes, n_classes)
    support, guessed = confusion.sum(axis=1), confusion.sum(axis=0)
    tp = confusion.diagonal()
    recall = np.divide(tp, support, out=np.zeros(n_classes, float), where=support > 0)
    f1 = np.divide(2 * tp, support + guessed, out=np.zeros(n_classes, float), where=(support + guessed) > 0)
    delta = np.abs(actual.astype(int) - predicted.astype(int))
    result = dict(accuracy=float(np.mean(actual == predicted)),
                  balanced_accuracy=float(recall[support > 0].mean()),
                  macro_f1=float(f1[(support + guessed) > 0].mean()),
                  error_ordinal=float(delta.mean()), dentro_un_bin=float((delta <= 1).mean()),
                  tasa_subestimacion=float((predicted < actual).mean()),
                  tasa_sobreestimacion=float((predicted > actual).mean()),
                  n_clases=n_classes, clases_reales_presentes=int((support > 0).sum()))
    return result, confusion

# Helpers compartidos, embebidos completos en cada celda independiente.
def ex_div(n, d):
    return float(n / d) if d else np.nan


def ex_event(y, p):
    y, p = np.asarray(y, bool), np.asarray(p, bool)
    tp, fp = int((y & p).sum()), int((~y & p).sum())
    fn, tn = int((y & ~p).sum()), int((~y & ~p).sum())
    return dict(N=int(y.size), TP=tp, FP=fp, FN=fn, TN=tn,
                prevalencia=ex_div(tp+fn, y.size), tasa_alertas=ex_div(tp+fp, y.size),
                recall=ex_div(tp, tp+fn), precision=ex_div(tp, tp+fp),
                F1=ex_div(2*tp, 2*tp+fp+fn), falsas_alarmas=ex_div(fp, fp+tn))


def ex_current_train(ctx, rows):
    path = Path(ctx['cfg']['dfyp_path'])
    frame = pd.read_parquet(path) if path.suffix.lower() in ('.parquet', '.pq') else pd.read_csv(path)
    if 'YS' in frame.columns and 'YS_p' not in frame.columns:
        frame = frame.rename(columns=lambda c: c[:-2]+'_p' if c.endswith('_y') else 'YS_p' if c == 'YS' else c)
    values = frame.iloc[rows-int(ctx['cfg']['horizon'])][[a+'_p' for a in ctx['assets']]].to_numpy(dtype=np.float32)
    if np.any(values < -1e-8):
        raise ValueError('Masas negativas en train.')
    values = np.clip(np.nan_to_num(values, nan=0., posinf=0., neginf=0.), 0., None)
    total = values.sum(axis=1, keepdims=True, dtype=np.float64)
    if np.any(total <= 0):
        raise ValueError('Train sin masa.')
    values = (values/total).astype(np.float32)
    if ph_hash(path) != ctx['manifest']['train_bins']['dfyp_sha256']:
        raise ValueError('dfyp cambio durante la lectura; detener sin mezclar archivos.')
    return ph_prob(values)


def ex_delta_labels(values, tau):
    # Clase 0=baja, 1=estable (incluye ambos limites), 2=sube.
    return np.where(values < -tau, 0, np.where(values > tau, 2, 1)).astype(np.int64)


def ex_level_labels(values, cuts, activity):
    labels = np.zeros(values.shape, dtype=np.int64)
    for j, boundaries in enumerate(cuts):
        active = values[:, j] > activity
        labels[active, j] = 1 + np.searchsorted(boundaries, values[active, j], side='right')
    return labels


def ex_run(root, experiment, config):
    if experiment not in ('delta', 'cuantiles', 'actividad'):
        raise ValueError('Experimento desconocido.')
    ctx = ph_context(root)
    first = ph_load_seed(ctx, POSTHOC_SEEDS[0])
    current = first['Persistencia'].copy()
    del first
    ctx['manifest']['initial_alignment'] = ctx['manifest'].pop('target_alignment')
    ctx['manifest']['sources'].clear()
    train, mu, sigma, rows = ph_train_statistics(ctx)
    constants = sigma <= 1e-8
    target = ctx['target']
    n_nodes = len(ctx['assets'])
    settings = []
    specifications = []
    coverage = []
    if experiment == 'delta':
        q, floor = float(config['quantile_abs_delta']), float(config['min_change'])
        if not 0 < q < 1 or floor <= 0:
            raise ValueError('Cuantil o piso de cambio invalido.')
        previous = ex_current_train(ctx, rows)
        delta = train-previous
        tau = np.maximum(np.quantile(np.abs(delta), q, axis=0), floor)
        train_y = ex_delta_labels(delta, tau)
        actual = ex_delta_labels(target-current, tau)
        settings.append(('mediana_cambio_absoluto', train_y, actual, np.full(n_nodes, 3), np.ones(n_nodes, bool), tau))
        for j, a in enumerate(ctx['assets']):
            specifications.append(dict(Nodo=a, tau=float(tau[j]), cuantil=q, piso=floor,
                                       constante_train=bool(constants[j]), n_train=len(train)))
    elif experiment == 'cuantiles':
        activity = float(config['activity_floor'])
        qs = np.asarray(config['quantiles'], float)
        minimum = int(config['min_positive_train'])
        if activity <= 0 or minimum < 2 or qs.ndim != 1 or not len(qs) or np.any(np.diff(qs) <= 0) or np.any((qs <= 0) | (qs >= 1)):
            raise ValueError('Configuracion de cuantiles invalida.')
        cuts, eligible, counts = [], [], []
        for j, a in enumerate(ctx['assets']):
            positive = train[train[:, j] > activity, j]
            # No confundir jitter de cierre float32 con variacion positiva real.
            ok = (len(positive) >= minimum and
                  np.ptp(positive) > 4*np.finfo(np.float32).eps*max(float(positive.max()), activity))
            boundaries = np.unique(np.quantile(positive, qs)) if ok else np.array([])
            # Cortes extremos no crean categorias positivas vacias en train.
            if ok:
                boundaries = boundaries[(boundaries > positive.min()) & (boundaries < positive.max())]
            cuts.append(boundaries)
            eligible.append(ok)
            counts.append(len(boundaries)+2)
            coverage.append(dict(Nodo=a, elegible=ok, n_positivos_train=len(positive),
                                 n_bins_positivos=len(boundaries)+1 if ok else 0,
                                 motivo='ok' if ok else 'historia positiva insuficiente o sin variacion'))
            specifications.append(dict(Nodo=a, umbral_actividad=activity, cortes_positivos=json.dumps(boundaries.tolist()),
                                       elegible=ok, min_positivos_train=minimum))
        eligible = np.asarray(eligible, bool)
        if not eligible.any():
            raise ValueError('Ningun activo tiene historia positiva suficiente; no se inventan cuantiles.')
        settings.append(('actividad_y_cuartiles_positivos', ex_level_labels(train, cuts, activity),
                         ex_level_labels(target, cuts, activity), np.asarray(counts), eligible, (cuts, activity)))
    else:
        levels = np.asarray(config['activity_floors'], float)
        if levels.ndim != 1 or not len(levels) or np.any((levels <= 0) | (levels >= 1)) or np.any(np.diff(levels) <= 0):
            raise ValueError('Umbrales de actividad invalidos.')
        for floor in levels:
            settings.append((f'umbral_{floor:g}', (train > floor).astype(np.int64),
                             (target > floor).astype(np.int64), np.full(n_nodes, 2), np.ones(n_nodes, bool), float(floor)))
            for j, a in enumerate(ctx['assets']):
                specifications.append(dict(Nodo=a, umbral_actividad=float(floor),
                                           tasa_actividad_train=float((train[:, j] > floor).mean()), constante_train=bool(constants[j])))
    tag = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:10]
    out = ctx['root'] / 'GAT_crypto_extended_audits_v1' / ('21_'+experiment+'_sin_entreno') / tag / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
    out.mkdir(parents=True)
    ph_csv(pd.DataFrame(specifications), out/'umbrales_train.csv')
    if coverage:
        ph_csv(pd.DataFrame(coverage), out/'cobertura_train.csv')
    occupation = []
    modes = {}
    for name, train_y, actual, counts, eligible, state in settings:
        modes[name] = np.array([np.bincount(train_y[:, j], minlength=int(counts[j])).argmax() for j in range(n_nodes)])
        for j, asset in enumerate(ctx['assets']):
            if not eligible[j]:
                continue
            for label in range(int(counts[j])):
                occupation.append(dict(escenario=name, Nodo=asset, clase=label,
                                       n_train=int((train_y[:, j] == label).sum()), n_test=int((actual[:, j] == label).sum())))
        ph_npz(out/f'etiquetas_reales_{name}.npz', assets=ctx['assets'], t=ctx['t'], elegible=eligible,
               train_target_rows=rows, train_labels=train_y, test_labels=actual)
    ph_csv(pd.DataFrame(occupation), out/'ocupacion_train_test.csv')
    del train
    all_nodes, all_events = [], []
    for seed in POSTHOC_SEEDS:
        print(f'{experiment}: semilla {seed}, leyendo predicciones; cero entrenamientos.', flush=True)
        predictions = ph_load_seed(ctx, seed)
        node_rows, event_rows, confusion_rows = [], [], []
        for name, train_y, actual, counts, eligible, state in settings:
            encoded = {}
            for model, values in [*predictions.items(), ('Bin modal de train', None)]:
                if values is None:
                    pred = np.broadcast_to(modes[name], actual.shape).copy()
                elif experiment == 'delta':
                    pred = ex_delta_labels(values-current, state)
                elif experiment == 'cuantiles':
                    pred = ex_level_labels(values, *state)
                else:
                    pred = (values > state).astype(np.int64)
                encoded[POSTHOC_MODELS.get(model, 'modal_train')] = pred
                for j, asset in enumerate(ctx['assets']):
                    if not eligible[j]:
                        continue
                    y, p = actual[:, j], pred[:, j]
                    metrics, cm = ph_class_metrics(y, p, int(counts[j]))
                    meta = dict(Modelo=model, seed=seed, escenario=name, Nodo=asset,
                                constante_train=bool(constants[j]), modelo_entrenado=False,
                                npp_aristas=model in ('GAT NPP solo aristas', 'GAT + NPP'), npp_loss=model == 'GAT + NPP')
                    node_rows.append({**meta, **metrics, 'N_test': len(y)})
                    for a, b in zip(*np.nonzero(cm)):
                        confusion_rows.append({**meta, 'clase_real': int(a), 'clase_predicha': int(b), 'n': int(cm[a, b])})
                for cohort, cols in [('todos_elegibles', eligible), ('variables_en_train', eligible & ~constants), ('constantes_en_train', eligible & constants)]:
                    if not cols.any():
                        continue
                    y, p = actual[:, cols], pred[:, cols]
                    meta = dict(Modelo=model, seed=seed, escenario=name, cohorte=cohort, modelo_entrenado=False)
                    if experiment == 'delta':
                        event_rows.append({**meta, 'evento': 'cambio_relevante', **ex_event(y != 1, p != 1),
                                           'direccion_correcta_en_cambios': ex_div(int(((y == p) & (y != 1)).sum()), int((y != 1).sum()))})
                        for label, event in [(0, 'perdida'), (2, 'ganancia')]:
                            event_rows.append({**meta, 'evento': event, **ex_event(y == label, p == label)})
                    elif experiment == 'actividad':
                        now = current[:, cols] > state
                        event_rows.append({**meta, 'evento': 'estado_activo', **ex_event(y == 1, p == 1)})
                        event_rows.append({**meta, 'evento': 'activacion_desde_inactivo', **ex_event((y == 1)[~now], (p == 1)[~now])})
                        event_rows.append({**meta, 'evento': 'desactivacion_desde_activo', **ex_event((y == 0)[now], (p == 0)[now])})
            ph_npz(out/f'predicciones_{name}_seed_{seed:04d}.npz', assets=ctx['assets'], seed=np.array(seed), t=ctx['t'], elegible=eligible, target_labels=actual, **encoded)
        ph_csv(pd.DataFrame(node_rows), out/f'metricas_nodos_seed_{seed:04d}.csv')
        ph_csv(pd.DataFrame(confusion_rows), out/f'confusion_seed_{seed:04d}.csv')
        if event_rows:
            ph_csv(pd.DataFrame(event_rows), out/f'eventos_seed_{seed:04d}.csv')
        all_nodes.extend(node_rows); all_events.extend(event_rows)
    nodes = pd.DataFrame(all_nodes)
    metrics = ['accuracy', 'balanced_accuracy', 'macro_f1', 'error_ordinal', 'dentro_un_bin']
    macros = []
    for cohort, subset in [('todos_elegibles', nodes), ('variables_en_train', nodes[~nodes.constante_train]), ('constantes_en_train', nodes[nodes.constante_train])]:
        if subset.empty:
            continue
        grouped = subset.groupby(['escenario', 'Modelo', 'seed'], as_index=False)
        means = grouped[metrics].mean()
        means = means.merge(grouped.agg(N_nodos=('Nodo', 'nunique')), on=['escenario', 'Modelo', 'seed'], validate='one_to_one')
        means['cohorte'] = cohort
        macros.append(means)
    macro = pd.concat(macros, ignore_index=True)
    def summarize(df, keys, cols):
        result = df.groupby(keys)[cols].agg(['mean', 'std', 'count'])
        result.columns = ['_'.join(c) for c in result.columns]
        return result.reset_index()
    summary = summarize(macro, ['escenario', 'cohorte', 'Modelo'], metrics)
    ph_csv(nodes, out/'metricas_por_nodo.csv'); ph_csv(macro, out/'macro_por_semilla.csv')
    ph_csv(summary, out/'resumen_modelos.csv')
    paired = []
    for ref in ['Persistencia', 'GAT 100% libre', 'GAT NPP solo aristas', 'Bin modal de train']:
        merged = macro[macro.Modelo != ref].merge(macro[macro.Modelo == ref], on=['escenario', 'cohorte', 'seed'], suffixes=('', '_ref'), validate='many_to_one')
        for _, row in merged.iterrows():
            for m in metrics:
                d = row[m]-row[m+'_ref']
                paired.append(dict(Modelo=row.Modelo, Referencia=ref, seed=int(row.seed), escenario=row.escenario, cohorte=row.cohorte,
                                   metrica=m, delta=float(d), mejora_favorable=float(-d if m == 'error_ordinal' else d)))
    ph_csv(pd.DataFrame(paired), out/'deltas_emparejados.csv')
    display(summary[['escenario', 'cohorte', 'Modelo', 'accuracy_mean', 'balanced_accuracy_mean', 'macro_f1_mean']])
    if all_events:
        events = pd.DataFrame(all_events)
        event_summary = summarize(events, ['escenario', 'cohorte', 'evento', 'Modelo'], ['prevalencia', 'tasa_alertas', 'recall', 'precision', 'F1', 'falsas_alarmas'])
        ph_csv(events, out/'eventos_por_semilla.csv'); ph_csv(event_summary, out/'resumen_eventos.csv')
        display(event_summary[['escenario', 'cohorte', 'evento', 'Modelo', 'recall_mean', 'precision_mean', 'F1_mean', 'falsas_alarmas_mean']])
    ctx['manifest'].update(experiment=experiment, configuration=config, training_performed=False,
                           exploratory_reused_test=True, thresholds_fit='train only',
                           uncertainty='SD entre semillas, no IC temporal; benchmark determinista repetido solo para emparejar',
                           quantile_rule='linear; cortes repetidos/extremos fusionados',
                           activity_rule='p > umbral; igualdad pertenece a inactivo',
                           delta_rule='estable si -tau <= delta <= tau',
                           metric_caution='error ordinal no es masa economica; para actividad dentro_un_bin siempre 1')
    ph_json(ctx['manifest'], out/'manifest_completo.json')
    print('Guardado completo:', out)
    return out


RESULTADOS_24 = ex_run(CARPETA_PROYECTO, 'actividad', CONFIG_EXPERIMENTO)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Semilla 11: redondeo compatible verificado; diferencia maxima 8.91e-09.
Bins: leyendo solo dfyp para recuperar el train original; no se construyen features.
actividad: semilla 11, leyendo predicciones; cero entrenamientos.
actividad: semilla 23, leyendo predicciones; cero entrenamientos.
actividad: semilla 42, leyendo predicciones; cero entrenamientos.
actividad: semilla 67, leyendo predicciones; cero entrenamientos.
actividad: semilla 101, leyendo predicciones; cero entrenamientos.
actividad: semilla 137, leyendo predicciones; cero entrenamientos.
actividad: semilla 211, leyendo predicciones; cero entrenamientos.
actividad: semilla 307, leyendo predicciones; cero entrenamientos.
actividad: semilla 401, leyendo predicciones; cero entrenamientos.
actividad: semilla 503, leyendo predicciones; cero entrenamientos.


,escenario,cohorte,Modelo,accuracy_mean,balanced_accuracy_mean,macro_f1_mean
0,umbral_0.0001,constantes_en_train,Bin modal de train,0.822173,0.666667,0.596938
1,umbral_0.0001,constantes_en_train,GAT + NPP,0.931610,0.803642,0.786572
2,umbral_0.0001,constantes_en_train,GAT 100% libre,0.928466,0.821422,0.779619
3,umbral_0.0001,constantes_en_train,GAT NPP solo aristas,0.928102,0.823561,0.778778
4,umbral_0.0001,constantes_en_train,Persistencia,0.937624,0.788478,0.788643
5,umbral_0.0001,todos_elegibles,Bin modal de train,0.752891,0.556263,0.472331
6,umbral_0.0001,todos_elegibles,GAT + NPP,0.827159,0.672387,0.662186
7,umbral_0.0001,todos_elegibles,GAT 100% libre,0.825949,0.679396,0.667744
8,umbral_0.0001,todos_elegibles,GAT NPP solo aristas,0.831653,0.673654,0.665395
9,umbral_0.0001,todos_elegibles,Persistencia,0.832954,0.686978,0.686985


,escenario,cohorte,evento,Modelo,recall_mean,precision_mean,F1_mean,falsas_alarmas_mean
0,umbral_0.0001,constantes_en_train,activacion_desde_inactivo,Bin modal de train,0.000000,NaN,0.000000,0.000000
1,umbral_0.0001,constantes_en_train,activacion_desde_inactivo,GAT + NPP,0.361376,0.390074,0.374404,0.022438
2,umbral_0.0001,constantes_en_train,activacion_desde_inactivo,GAT 100% libre,0.347090,0.349124,0.345728,0.025997
3,umbral_0.0001,constantes_en_train,activacion_desde_inactivo,GAT NPP solo aristas,0.356349,0.346758,0.348983,0.027031
4,umbral_0.0001,constantes_en_train,activacion_desde_inactivo,Persistencia,0.000000,NaN,0.000000,0.000000
...,...,...,...,...,...,...,...,...
130,umbral_1e-08,variables_en_train,estado_activo,Bin modal de train,0.989437,0.962028,0.975540,0.463375
131,umbral_1e-08,variables_en_train,estado_activo,GAT + NPP,1.000000,0.922269,0.959563,1.000000
132,umbral_1e-08,variables_en_train,estado_activo,GAT 100% libre,1.000000,0.922269,0.959563,1.000000
133,umbral_1e-08,variables_en_train,estado_activo,GAT NPP solo aristas,1.000000,0.922269,0.959563,1.000000


Guardado completo: /content/drive/MyDrive/Neural/NPP/Cripto/GAT_crypto_extended_audits_v1/21_actividad_sin_entreno/fc6720cabe/20260908T105507937306Z


# 25. Fin de ciclo de estos datos: conclusiones y continuidad

Cierre consolidado: 9 de septiembre de 2026. Continuación del contraste y de la ablación de diez semillas, ya integrados en `main`. Las cifras describen ejecuciones anteriores; esta actualización es documental y no genera nuevos resultados.

Este documento cierra el ciclo exploratorio sobre el conjunto de datos y las predicciones disponibles; no declara agotado el potencial de los datos, de la arquitectura ni del principio NPP. Las notas anteriores se conservan como historia del experimento y deben leerse junto con estas salvedades posteriores.

## Alcance y procedencia

Se preservan las 67 celdas anteriores a este cierre, incluidos código, salidas, metadatos y notas. Se amplía únicamente la sección 25 existente. La evidencia procede de las salidas y notas del notebook, de los resúmenes de la ablación de diez semillas y de las tablas guardadas de desarrollo. Las cifras de una ejecución de referencia, las medias entre semillas y los resúmenes por nodo no se intercambian como si fueran la misma agregación.

El notebook estudia 471 nodos, con barras de 10 minutos y horizonte de una barra; el test de referencia contiene 2.016 instantes. No debe confundirse con el nuevo pipeline de extracción de cinco minutos. Se pronostican participaciones transaccionales, no rendimientos.

En GitHub se incluyen el notebook, este cierre en Markdown, tres tablas resumen de desarrollo y un registro de verificación. Las tablas históricas de tres y diez semillas ya están en el repositorio. Los restantes artefactos externos, pesos y datos de entrada continúan en las rutas de Drive documentadas por el notebook; esta publicación no afirma copiarlos todos.

Fuente: [GAT crypto - contraste experimental](https://drive.google.com/file/d/1oQS_XZ57mKxqUtMb4JZSMWrQIvu24PNX/view).

## Balance de la predicción continua

En el test cronológico original de diez semillas, GAT + NPP reduce aproximadamente CE 2,68%, KL 31,05%, JS 23,80%, MAE 12,98% y RMSE 16,02% frente a persistencia. Frente al GAT libre, las reducciones son mucho menores: KL 0,71%, MAE 0,18% y RMSE 0,04%. NPP solo en aristas obtiene mejores medias de JS y MAE que la regularización completa; no existe dominio de una variante en todas las métricas.

CE y KL no son evidencia independiente: para un mismo objetivo difieren por su entropía. La ventaja continua no implica rentabilidad, anticipación de precios ni identificación causal de gp y eta. Las variables constituyen proxies iniciales derivados de OHLCV y del simplex, no mediciones estructurales directas del mecanismo teórico.

## Arquitectura, índice y alcance de la identificación

El GAT procesa el vector completo de participaciones: cada activo es un nodo con historia y factores propios; la atención combina mensajes de los vecinos y el softmax final normaliza entre activos. El GAT libre puede aprender relaciones sin imponer el canal energético NPP. En la ablación de diez semillas, el control **100% libre** usa similitud CLR pura; no debe confundirse con cualquier modelo anterior llamado simplemente «libre».

El peso `gamma_npp` modifica la combinación de similitud histórica CLR y proximidad energética usada para seleccionar y ponderar aristas. No ordena directamente aumentar o disminuir la cuota de un nodo. Los pesos de las aristas entran como sesgo logarítmico en la atención; el aprendizaje determina cómo usar esos mensajes. `lambda_scale` modifica el peso de los términos auxiliares NPP durante el entrenamiento, mientras `lambda_geometry` penaliza discrepancias CLR de la distribución predicha. Son intervenciones diferentes.

El notebook incorpora como feature el índice operativo `n = EMA_precio_1m / sorpresa`, con sorpresa de Shannon, además de rezagos y otros factores. Este contraste no aísla por sí solo el aporte individual de esa feature: la ablación principal modifica aristas y pérdida. Por ello, una mejora atribuible al tratamiento NPP no demuestra automáticamente la utilidad marginal del índice de neguentropía tomado por separado.

Las cabezas `gp` y `eta` son latentes. La parametrización energética usa `q_NPP = softmax(-gp/eta)`. Para cuotas positivas, `E = -CLR(p)` representa energías relativas; con ceros requiere el tratamiento numérico documentado. Representar una distribución mediante energías no demuestra el mecanismo económico: la prueba empírica es la utilidad de las restricciones o de la representación para pronosticar fuera del entrenamiento. Las cuotas por sí solas no identifican causalmente `gp` y `eta` por separado.

## Ejecución de referencia y ablación de diez semillas — secciones 10 a 13

En la ejecución de referencia, GAT + NPP superó a persistencia en RMSE en 427 de 440 nodos activos. Frente al GAT libre ganó en RMSE en 246 y en MAE en 163. Son conteos de esa ejecución, no victorias universales ni diez réplicas temporales: la ventaja no se distribuye por igual entre nodos ni métricas.

En la ablación emparejada de diez semillas, los tres GAT superan a persistencia en CE, KL, JS, MAE y RMSE. El canal NPP de aristas mejora CE, KL, JS y MAE frente al control 100% libre en las diez semillas. GAT + NPP obtiene las mejores medias de CE/KL y RMSE, mientras NPP solo aristas obtiene las mejores medias de JS y MAE. El incremento frente al GAT libre es mucho menor que la ganancia total frente a persistencia.

La distancia de Aitchison sigue favoreciendo a persistencia en la ablación original, incluso en la evaluación de activos. Las masas pequeñas y el tratamiento de ceros afectan intensamente los log-ratios; esto exige informar tanto ajuste de masa como geometría. RMSE no es una banda de confianza ni un intervalo simétrico de error. Los intervalos bootstrap sobre deltas entre semillas describen sensibilidad a la aleatoriedad del ajuste condicionada al mismo conjunto temporal; no representan incertidumbre sobre futuros regímenes.

## Intensidad NPP — sección 15

La grilla de 16 configuraciones y tres semillas favorece, dentro de la región evaluada, `gamma_npp=0.50` y `lambda_scale=1.00`: CE de validación 2,961361 frente a 2,963339 del control sin ambos tratamientos, una reducción aproximada del 0,0668%. La CE media mejora al aumentar el componente de aristas en el intervalo explorado; la respuesta a la regularización es no lineal, con una región favorable entre 0,5 y 1 según las notas de ejecución.

Esto es una señal exploratoria de respuesta a la intensidad, no una ley monótona ni un óptimo universal. `gamma=0.50` está en el borde de la grilla. La mejor CE empeora Aitchison aproximadamente 3,26% y CLR-MSE 6,57% frente al control. Las notas registran que 34 de 48 corridas eligieron la época máxima 20; ese límite impide asumir convergencia suficiente o saturación del modelo.

## Identificación interna — sección 16

Con el grafo híbrido fijo, `energy_ce` aislada obtiene CE de validación 2,961651 frente a 2,962534 de la pérdida predictiva sola. El residuo también aporta una pequeña mejora, pero sumar componentes no produce una mejora aditiva garantizada. Estas comparaciones aíslan tratamientos del modelo, no causas económicas.

La cabeza energética autónoma alcanza CE 2,969290: alrededor de 0,23% peor que la cabeza predictiva de control con el mismo grafo, a cambio de reducir Aitchison aproximadamente 14% y CLR-MSE 26%. Es una alternativa con un compromiso diferente entre masa y geometría. No prueba que las variables latentes sean mediciones de capitales reales.

## Coherencia composicional — sección 17

| `lambda_geometry` | CE de validación | Aitchison | CLR-MSE |
|---|---:|---:|---:|
| 0 | 2,961797 | 4,633726 | 21,531628 |
| 0,001 | 2,964038 | 2,133120 | 4,981605 |

En este sweep, `lambda_geometry=0.001` reduce Aitchison cerca del 54% y CLR-MSE cerca del 77%, con un aumento de CE de aproximadamente 0,076%. Es un compromiso de validación prometedor, condicionado al criterio elegido; no una superioridad única sobre toda la frontera de Pareto. Aumentar a 0,01 mejora algo más la geometría a costa de mayor CE.

La hipótesis de gradientes compatibles entre pérdida predictiva y geométrica sigue pendiente: un sweep de métricas finales no identifica el signo del producto interno de sus gradientes. Haría falta medirlo en parámetros compartidos durante el entrenamiento. Tampoco corresponde atribuir esta mejora geométrica a las predicciones congeladas de modelos entrenados sin esa penalización.

## Sensibilidad de nodos y cola — secciones 18 y 19

`U` y BTC concentran aproximadamente 74% del error cuadrático del GAT + NPP; los diez mayores concentran alrededor de 92,5%. La identidad económica de `U` debe verificarse contra las columnas fuente. La auditoría conserva una ventaja CE/KL frente al libre al excluir nodos dominantes y volver a cerrar el simplex, de modo que esa ventaja no depende exclusivamente de ellos.

Excluir componentes y renormalizar cambia el objeto evaluado: es sensibilidad de las predicciones existentes, no reentrenamiento ni intervención causal. La cola puede tener MAE diminuto y errores CLR elevados. Tampoco debe equipararse automáticamente un cero observado a un cero estructural económico; hay que distinguir inactividad, ausencia y reglas de construcción del dato.

## Estabilidad dentro del test — sección 20

Según el resumen diario registrado, los tres GAT superan a persistencia en CE/KL, JS, MAE y RMSE en los 14 bloques diarios. NPP solo aristas supera al libre los 14 días en CE/KL, JS y MAE, pero solo 7 de 14 en RMSE. NPP completo gana al libre 14 de 14 en CE/KL, 12 de 14 en JS y MAE, y 8 de 14 en RMSE. El deterioro geométrico de la ablación original también persiste en este análisis.

Hay estabilidad descriptiva dentro del período observado, particularmente para el canal de aristas. No es un walk-forward con reentrenamiento: los pesos están congelados y los bloques pertenecen al mismo test. Los acumulados se solapan y los bloques diarios pueden conservar dependencia temporal.

## Bins de nivel y transiciones — secciones 21 y 21.b

Discretizar participaciones continuas cambia el objetivo: acertar un valor cercano no garantiza acertar su intervalo. Persistencia domina el acierto exacto de niveles en varias comparaciones. Un accuracy elevado puede reflejar permanencias frecuentes; por eso se separan permanencia, cambio y destino del cambio.

En la cohorte de 447 nodos variables en train, el resumen agrupado de detección de cambios de bin es:

| Modelo | Recall de cambio | Precisión | F1 | Falsas alarmas entre permanencias |
|---|---:|---:|---:|---:|
| GAT libre | 32,08% | 52,27% | 39,75% | 6,38% |
| NPP solo aristas | 33,32% | 55,47% | 41,63% | 5,82% |
| GAT + NPP | 32,77% | 52,63% | 40,38% | 6,42% |

NPP de aristas ofrece el mejor equilibrio en esa comparación, pero detecta solo un tercio de los cambios. Detectar un cambio no equivale a acertar su destino. Persistencia acierta permanencias y falla cambios por construcción; recuperar cambios puede no compensar las falsas alarmas en accuracy total.

En los nodos constantes en train aparecen muchas falsas alertas al cruzar un umbral diminuto. Se deben conservar como cohorte separada. Las medias macro por nodo y las métricas agrupadas por observaciones tienen denominadores distintos y no se sustituyen entre sí.

## Pérdida, estabilidad y ganancia — sección 22

Se etiqueta `delta = p(t+1)-p(t)` con `tau_j=max(mediana_train(abs(delta_j)), 1e-8)`. La relevancia del cambio es estadística bajo esa regla; el piso numérico no establece relevancia económica.

| Modelo, nodos variables en train | Accuracy | Balanced accuracy | Macro-F1 |
|---|---:|---:|---:|
| Persistencia / bin modal | 57,21% | 36,47% | 27,02% |
| GAT libre | 57,19% | 45,11% | 43,60% |
| NPP solo aristas | 57,99% | 45,99% | 44,95% |
| GAT + NPP | 56,71% | 45,30% | 44,21% |

El canal de aristas mejora aproximadamente 1,34 puntos de macro-F1 frente al libre. Su ventaja resulta especialmente visible en pérdidas: recall 39,22%, precisión 79,10% y F1 52,42%, frente a F1 49,24% del libre. En ganancias, la precisión de los tres GAT ronda 28–29%; NPP completo logra el mayor recall, 39,25%, y F1 33,08%, pero las falsas alertas siguen siendo numerosas.

NPP completo detecta más cambios relevantes (recall 51,67%) y también produce más falsas alarmas (29,19%) que NPP de aristas (50,34% y 26,64%). La diferencia de F1 de detección entre ambos es pequeña, alrededor de 0,05 puntos. En nodos constantes en train, un recall casi total convive con falsas alarmas del 100% para cambio relevante: no es buena discriminación. Estas ventajas medias son descriptivas y no acreditan por sí solas significación temporal ni persistencia en cada semilla.

## Sección 23: actividad y cuartiles positivos

Los cortes se obtienen del entrenamiento por activo; se separa la inactividad y se usan cuartiles de valores positivos elegibles. Se discretizan predicciones continuas existentes, sin entrenar un clasificador.

| Modelo | Accuracy | Balanced accuracy | Macro-F1 |
|---|---:|---:|---:|
| Bin modal de train | 28,22% | 25,04% | 12,22% |
| GAT libre | 37,56% | 30,10% | 27,06% |
| NPP solo aristas | 37,66% | 29,91% | 26,89% |
| GAT + NPP | 37,07% | 29,72% | 26,40% |
| Persistencia | 47,71% | 38,58% | 38,58% |

Persistencia supera al mejor GAT de cada métrica en aproximadamente 10,04 puntos de accuracy y 11,52 puntos de macro-F1 (calculados antes del redondeo de la tabla). La ventaja no se limita al acierto de la clase dominante. Los GAT superan al bin modal, pero no al último estado observado. NPP completo no mejora esta tarea; el libre tiene las mejores métricas balanceadas entre GAT.

Una predicción continua más cercana puede cruzar incorrectamente una frontera de bin. Esa explicación es compatible con los resultados, no una causa demostrada. Los cuartiles no garantizan balance en test ni en la clase de inactividad. La sección delimita el alcance del método sin demostrar sobreajuste por sí sola.

## Sección 24: actividad y transiciones

Actividad significa participación mayor que el umbral, no presencia económica absoluta. Cambiar el umbral cambia el objetivo. La tabla siguiente corresponde a todos los nodos elegibles; los mejores GAT de accuracy y macro-F1 pueden ser variantes distintas.

| Umbral | Accuracy persistencia | Mejor accuracy GAT | Macro-F1 persistencia | Mejor macro-F1 GAT |
|---|---:|---:|---:|---:|
| 1e-8 | 97,36% | 89,43% | 74,79% | 64,04% |
| 1e-6 | 94,91% | 87,99% | 71,45% | 59,59% |
| 1e-4 | 83,30% | 83,17% | 68,70% | 66,77% |

Persistencia domina globalmente. Con 1e-8, los GAT predicen actividad en todos los casos de la cohorte variable: recall y tasa de falsos positivos de 100%. El accuracy alto refleja prevalencia, no discriminación de inactividad.

No obstante, persistencia no anticipa cambios de estado por construcción. En la cohorte constante en train y umbral 1e-4, GAT + NPP detecta 36,14% de las activaciones, con precisión 39,01% y F1 37,44%; persistencia tiene recall cero. Es una ventaja localizada, no global. No corresponde escoger 1e-4 como umbral óptimo después de mirar test.

## Conclusión del ciclo

Los GAT aportan valor predictivo para la distribución continua de participaciones. NPP aporta una mejora incremental modesta, condicionada por la representación, la pérdida y la tarea, con evidencia favorable al canal topológico en varias comparaciones. Persistencia sigue siendo especialmente fuerte para niveles discretos y estados. No existe un ganador universal.

Diez semillas miden incertidumbre de optimización condicional al mismo dataset, no diez muestras independientes del proceso generador ni bootstrap temporal. Los análisis sucesivos del mismo test son exploratorios. La evidencia no permite afirmar causalidad, validez universal ni proximidad a un límite ontológico de descomposición. La repetición del benchmark determinista por semilla sirve para emparejar comparaciones, no crea diez benchmarks independientes. Evitar leakage de variables no elimina el sesgo de diseñar nuevos análisis después de observar test. Las afirmaciones anteriores de «pre-registro» o «test intacto» se refieren a la etapa original, no al conjunto de lecturas exploratorias posteriores.

## ¿Tenemos evidencia de que la tipología NPP es útil?

Sí, en un sentido predictivo y acotado a este experimento. La comparación entre GAT de base comparable, con y sin representación NPP en las aristas, muestra mejoras pequeñas pero consistentes en varias métricas distributivas del test utilizado. Es defendible concluir:

> La representación relacional NPP contiene estructura útil para predecir ciertos aspectos del sistema, bajo los datos, la arquitectura y el protocolo evaluados.

Aquí «tipología» se refiere a la representación del fenómeno; «topología informacional» es más preciso cuando hablamos de conexiones y pesos del grafo. La utilidad observada puede surgir de una representación o un sesgo inductivo mejor: si NPP se construye a partir de los mismos datos, no implica agregar información externa nueva. Tampoco demuestra causalidad, identificación estructural de gp/eta, ni superioridad universal: los resultados de las secciones 23 y 24 impiden esa generalización.

La idea es conceptualmente afín a [SpotV2Net, Brini y Toscano (2025)](https://doi.org/10.1016/j.ijforecast.2024.11.004), que incorpora estimaciones de volatilidad y covolatilidad en nodos y características de volatilidad de volatilidad en aristas para pronosticar volatilidad intradiaria. La analogía es el uso de información relacional estructurada en un GAT; no se trata de una réplica, ni de evidencia externa que valide específicamente NPP. Sus objetivos, variables y mercados difieren de los de este notebook.

## Siguiente ciclo: tres continuaciones, todavía no ejecutadas

| Continuación | Diseño requerido | Pregunta científica |
|---|---|---|
| Walk-forward con reentrenamiento | Ventanas rolling o expanding; reentrenar cada origen con pasado disponible, validación interna cronológica, purga según horizonte; grafo, escaladores y selección ajustados solo al pasado; evaluar el bloque posterior intacto. | ¿Se sostiene la ventaja cuando cambian el período y el conjunto de entrenamiento? |
| Esquema de regímenes | Definir volatilidad, liquidez, concentración o estrés con información disponible al instante; calibrar cortes en train; comparar modelos dentro de cada régimen, reportando soporte e incertidumbre temporal. Un modelo condicionado por régimen requerirá entrenamiento específico. | ¿Bajo qué restricciones o condiciones de mercado aporta NPP? |
| GAT nativamente entrenado con bins | Definir el objetivo antes del test (nivel, cambio o actividad), estimar cortes solo en train y entrenar una cabeza clasificadora/ordinal con pérdida adecuada. Comparar persistencia y bin modal; reportar macro-F1, balanced accuracy, matrices de confusión, transiciones y calibración. | ¿La desventaja categórica proviene en parte de entrenar para un objetivo continuo distinto? |

Un softmax sobre clases por nodo no garantiza por sí mismo que las cuotas reconstruidas sumen uno entre activos: el nuevo diseño con bins debe explicitar cómo conserva o reconstruye la coherencia composicional. Las clases no deben elegirse por el mejor resultado del test reutilizado.

Estas rutas abren una nueva fase de experimentación; no son resultados de este cierre. Se requiere un protocolo predefinido y bloques de evaluación no usados para seleccionar configuraciones. Se conserva el principio operativo: sin shuffle temporal, sin información futura y con guardado recuperable por corrida.
